<div style="text-align:right;font-size:12px;color:#444;">Created by <b>Ram Lakhan Meena, INSPECTOR CBIC</b><br>2026-10-09</div># GST-Prime Preliminary Investigation

**Pick the taxpayer, upload the GST-Prime Analytica exports, and this notebook produces the preliminary investigation workbook.**

Everything is kept in Google Drive, **filed under the taxpayer's GSTIN**. Each GSTIN gets its own folder, so uploads for one taxpayer can never mix with another's. Files already uploaded in an earlier session are picked up automatically, so you only ever upload what is new.

---

### The order to run in

| Step | Cell | What happens |
|---|---|---|
| 1 | Mount Drive, list the taxpayers already on file | see what is there before you start |
| 2 | **Enter the GSTIN and the name** — a form, not code | the case folder is created or re-opened |
| 3–5 | Load the engine | parser, checks, workbook builder |
| 6 | Upload the Prime exports | filed under the GSTIN shown in the banner |
| 7 | Extract | every row kept with its provenance |
| 8 | Run the checks and build the workbook | report written to Drive and downloaded |

```
GST_Prime_Investigation/
  33AAXCA1234A1ZX/            <- one folder per GSTIN
      case.json               taxpayer name and profile, remembered
      00_ENGINE/              the engine, so it survives between sessions
      01_RAW_UPLOADS/         the Prime exports exactly as received
          _MANIFEST.csv       every file, its checksum, when it came, under which GSTIN
          _DUPLICATES/        byte-identical copies, set aside — review and clear with the utility cell
      02_EXTRACTED/           parsed rows with provenance
      03_REPORTS/             the workbook, versioned and dated
  33ZZZZZ9999Z1ZZ/
      ...
```

### The rules this notebook works to

1. **Nothing is invented.** No value is estimated, interpolated or filled in. Where a report has not been uploaded, the test that needed it says so on its own sheet and is not run.
2. **A blank is not a nil.** A blank cell in a Prime export is kept blank and shaded light red. `Dec-2022 GSTR-1 blank` means *not filed*; it does not mean *nil turnover*. Reading one as the other is how a case gets lost.
3. **Source first, then formulas.** Every Prime report is written to its own `SRC_` sheet exactly as received. Every computed sheet then reads those with live Excel formulas, so any figure can be traced to the source cell it came from.
4. **The same data is never counted twice.** A file byte-identical to one already held is not stored again, whatever it is named. A later export carrying the same records plus new ones is accepted, and the records already held are replaced rather than added again.
5. **This is a preliminary investigation on returns data.** It is not a finding of fact. Every figure is to be re-verified against the portal, the ledgers, the returns and the books before any notice is issued.

### Version log

| Version | Date | Changes |
|---|---|---|
| v1.8 | 2026-10-09 | **A later export of a multi-line period now replaces that period as a whole.** Found by testing a re-upload: for reports where one period carries several lines — the HSN list, and the month-wise GSTR-7 detail — a later export was matched line by line, so a restated line (a deduction corrected from Rs.6,15,970 to Rs.6,25,970) was kept twice, once at the old value and once at the new, and a line the later export dropped was kept. Tested: Mar-2026 grew from 11 to 13 lines and Rs.41,44,869 to Rs.50,70,839 instead of 12 lines and Rs.44,54,869. Now every period such a file covers is replaced as a block; periods it does not cover are untouched; each added or dropped line is listed in `09A_RESTATEMENTS` (dropped = REVIEW, added = NOTE, both REVIEW when the two exports came in the same upload). Single-line reports (GSTR-3B, GSTR-1 and R7 summaries, challans) already replaced record by record. Keep ONE export per month for the month-wise reports — a second, different export of the same month is read as the later one. Regression unchanged (Rs.4,98,32,147). |
| v1.7 | 2026-10-09 | **GSTR-7 party-wise detail — the 16th report type — and a correction to how GSTR-7 is read.** Prime's *R7 Filing Details For The Month* export (GSTIN, trade name, taxable value and tax for each line) is now recognised. It carries no tax period inside it, so the period is taken from the file name and the register says so against every such file — keep Prime's file names unchanged. New sheet **`19D_TDS_PARTIES`**: every listed party with its registration type (position 14 of the GSTIN — D is a tax-deductor registration, Z an ordinary one), its PAN holder type (position 4 of the PAN — P individual, F firm, C company, G Government, L local authority), and whether section 51(1) read with Notification 50/2018-CT allows it to deduct at all; live COUNTIF/SUMIF totals by party and by month; a month-by-month reconciliation with the summary report; lines deducted at other than 1% + 1%; lines just above the Rs.2.5 lakh threshold; and the months whose detail is still missing. A party that is not a permitted deductor produces a VERY HIGH finding and its own action points (portal check, referral of the irregular deductor registrations, refund hold, statements under s.70). **Correction:** v1.6 called every GSTR-7 deduction 'Government' — it is now worded neutrally, because who the deductor is is a question the detail answers, not an assumption. **New for every report:** the GSTIN an export is titled for is read from the export and Step 7 warns, loudly, when it is not the case GSTIN. Findings are now ordered by amount and then by risk, and action points by priority. Regression: the earlier 29-file case is unchanged (Rs.4,98,32,147). |
| v1.6 | 2026-10-09 | **GSTR-7 (TDS by Government) added — the 15th report type.** Prime's *R7 FILINGS IN THE YEAR* export is now recognised at upload by its header (Return Period, No Of Deductors, SGST, CGST, IGST, Total GST) and filed, de-duplicated and restatement-checked like every other report. It shows tax deducted at source FROM the taxpayer by Government deductors under section 51, so (Total GST − cess) ÷ 2% is the value of supplies Government has PAID for — a floor on turnover. New sheet **`19C_TDS_GSTR7`**: month by month and FY-wise, with live formulas, against GSTR-3B (status, taxable turnover, outward CGST) and GSTR-1, and ten legal cross-checks each with its provision and a RUN / NOT RUN status — value paid for against turnover declared (cumulative within the FY, because by s.13(2) and s.31 a supply paid for must have been declared by that month); months with TDS but no or NIL GSTR-3B (s.46 GSTR-3A, s.62); CGST deducted at 1% above the CGST declared; GSTR-1 for each TDS month; TDS credited to the cash ledger but not used (refund watch, s.54); ITC against 2A; GSTR-9 / 9C, e-invoicing and QRMP thresholds on Government receipts alone; Rule 86B; intra- or inter-State. Months whose return is not yet due are shown NOT YET DUE and never counted as a shortfall. The only assumed figure — tax at a rate typed in H3 — stays on that sheet and out of the findings total. Findings, action points (GSTR-7A and deductor-wise detail, letters to the DDOs, GSTR-3A and s.62, refund watch, Form 26AS) and the report narrative are written automatically. The workbook now also builds when **no GSTR-3B report is uploaded at all** — the master and FY sheets say so instead of failing. Regression: the earlier 29-file case is unchanged (Rs.4,98,32,147). |
| v1.5 | 2026-10-08 | **Step 1 now recovers from a broken Drive connection by itself.** In v1.4 a dropped Drive connection stopped Step 1 with `ValueError: Mountpoint must not already contain files`, and restarting the session did not clear it. Cause: once Drive disconnects, `/content/drive` becomes an ordinary local folder, and any cell run after that wrote into it (case folders, uploads) — files that looked saved but never reached Drive. Colab then refuses to mount Drive onto a folder that has files in it. Step 1 now releases a dead connection, moves any such local leftovers aside to `/content/drive_NOT_SAVED_<time>` (nothing is deleted), lists them so you can re-upload anything that matters, and mounts Drive fresh. (2) **No more silent local saving.** Every cell that writes — the case folder, upload, extract, report, and both utilities — first checks that Drive is really connected and stops with a plain instruction if it is not, so work can no longer land outside Drive. The engine and the workbook are unchanged. |
| v1.4 | 2026-09-29 | **Three additions to the duplicate handling.** (1) **`_DUPLICATES/` review cell** — lists every quarantined file with its size, when it was moved and the identical file still held, plus how many upload attempts were turned away at the door. Nothing is deleted unless the box is ticked, and a file is only deleted if an identical copy is still held, re-checked by checksum at the moment of deletion; a file with no twin is kept and flagged. Each deletion is recorded in the manifest. (2) **Supersession shown in the workbook** — every record replaced by a later export carries a *Record history* column on its `SRC_` sheet naming the file it replaced; a row whose values differ is shaded amber and the differing cell orange, with the earlier value in a hover comment. Rows replaced by identical values are not shaded, so a full re-export does not turn the whole sheet amber. (3) **Restatement warning** — when a later export changes a value the earlier one already gave, Step 7 prints a warning and the workbook gains sheet `09A_RESTATEMENTS`, listing each difference with both values, the size of the movement and the two files. Changes are graded: **INFO** for GSTR-2A (it moves whenever suppliers file or amend, so it is expected), **NOTE** for blank-to-populated (most likely a return filed since), **REVIEW** for anything else — GSTR-3B and challans cannot be revised, so a change there is the one to check. REVIEW changes add a finding and a report caveat. **Which export is newer is not left to chance**: Prime exports carry no date of their own, so order comes from the upload time in the manifest; when two overlapping files arrived in the same upload nothing separates them, and every difference between them is marked ORDER UNCERTAIN, escalated to REVIEW, and shown in the warning, in `09A_RESTATEMENTS` and on the `SRC_` sheet, instead of filename order silently deciding. Also fixed: two identical lines *inside* one file (two equal HSN rows) are no longer collapsed into one. |
| v1.3 | 2026-09-29 | **Re-uploading a file no longer duplicates anything.** Two levels, because the second one was silently corrupting figures. (1) **At upload** — a file byte-identical to one already held is not stored again; the upload cell says so and records it in the manifest as `skipped-duplicate`. Byte-identical copies already sitting in `01_RAW_UPLOADS` from earlier runs are moved to `_DUPLICATES/` on the next run. A file with the same name but *different* content is still kept, under a timestamped name. (2) **At parse** — a LATER EXPORT of a report already held (same records plus a few new ones) is not a byte-duplicate and was being appended, so its overlapping records were counted twice. Tested: re-exporting PAYMENT DETAILS with one extra challan doubled the cash figure from Rs.1,38,83,546 to Rs.2,77,66,892. Every row now carries a natural key — tax period, or period plus HSN line, or the CIN of a challan — and a record already held is REPLACED, never added again. Files are read in upload order so the later export wins. `09_FILE_REGISTER` gains two columns showing records new and records replaced per file. |
| v1.2 | 2026-09-29 | **The GSTIN is now entered in a form, not in code.** Step 2 is a Colab form with text boxes for the GSTIN and the taxpayer's name, validated before anything is created, so a run can no longer end up filed under the example GSTIN. Added: a case picker that lists every taxpayer already on file with their name, file count and last report; `case.json` in each case folder so re-entering a GSTIN recalls the name and profile; `_MANIFEST.csv` in `01_RAW_UPLOADS` recording every file with its checksum, upload time and the GSTIN it was filed under; a banner naming the taxpayer immediately before and after the upload; and a re-file utility at the end for moving files already filed under the wrong GSTIN. |
| v1.1 | 2026-09-29 | Three fixes found by comparing a real Colab run against the reference workbook. (1) **Filename check** — Prime saves reports with spaces (`EXCESS ITC CLAIMED FOR SGST-IGST.xlsx`) but the check expected underscores, so 27 of 33 files wrongly showed *filename does not agree*; filenames are now compared on a normalised form. Detection itself was never affected — the header decides. (2) **Unfiled periods** — a period whose GSTR-3B is *Not-Filed* (Jun-2021, the month of registration) was counted as a nil-turnover period, giving 63 periods / 50 NIL instead of 62 / 49; such periods are now kept out of the period count and the NIL and dormancy tests, and named on the index. (3) **Excess-ITC period count** — was netting SGST against IGST inside a period (30); now counts a period if *any head* is in excess (35), consistent with the head-wise demand basis. No rupee figure changes. |
| v1.0 | 2026-09-29 | First build. Auto-detects 14 GST-Prime report types by header signature. Twelve checks: GSTR-1 vs GSTR-3B with filing status, excess ITC over GSTR-2A head-wise (the demand figure), the same test re-run from independent reports, NIL-period credit, Rule 86B, reverse charge, credit notes tested against the original supply, return filing delay, cash reconciliation, HSN profile, annual return, and a limitation tracker. Output workbook carries 20+ sheets with live formulas, an auto-written findings sheet, narrative report, action points, data-gap register and file register. Verified end to end against a live taxpayer's 29 Prime exports. |

## Step 1 · Mount Drive and see who is already on file

Run this first. It lists every taxpayer already held, so you can re-open an existing case instead of starting a new one. Re-running is harmless.

In [ ]:
import os, sys, re, json, shutil, datetime, hashlib, csv

IN_COLAB = "google.colab" in sys.modules
MOUNT = "/content/drive"
DRIVE_ROOT = MOUNT + "/MyDrive/GST_Prime_Investigation"

def drive_ok():
    """True only when Google Drive is really connected (not a leftover local folder)."""
    if not IN_COLAB:
        return True
    try:
        return os.path.ismount(MOUNT) and os.path.isdir(MOUNT + "/MyDrive")
    except OSError:
        return False

def require_drive():
    """Called at the top of every cell that writes. Stops before anything lands outside Drive."""
    if not drive_ok():
        raise SystemExit("\nGOOGLE DRIVE IS NOT CONNECTED — nothing was written.\n"
                         "Run Step 1 again (it reconnects Drive), then re-run Step 2 and this cell.")

def _mount_drive():
    import subprocess
    from google.colab import drive
    if drive_ok():
        print("Google Drive already connected.")
        return
    # 1. release a connection that has died (harmless if there is none)
    for cmd in (["fusermount", "-u", "-z", MOUNT], ["umount", "-l", MOUNT]):
        try:
            subprocess.run(cmd, capture_output=True, timeout=20)
        except Exception:
            pass
    # 2. anything left in /content/drive now is a LOCAL folder, written while Drive was not
    #    connected. Colab will not mount onto it, so move it aside — never delete it.
    if os.path.isdir(MOUNT) and not os.path.ismount(MOUNT):
        try:
            left = os.listdir(MOUNT)
        except OSError:
            left = None
        if left is None:
            raise SystemExit("The old Drive connection is stuck.\n"
                             "Use the menu: Runtime -> Disconnect and delete runtime, then run Step 1 again.")
        if left:
            stale = "/content/drive_NOT_SAVED_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            try:
                os.rename(MOUNT, stale)
            except OSError as e:
                raise SystemExit(f"Could not clear {MOUNT} ({e}).\n"
                                 "Use the menu: Runtime -> Disconnect and delete runtime, then run Step 1 again.")
            found = [os.path.relpath(os.path.join(r, f), stale)
                     for r, _, fs in os.walk(stale) for f in fs]
            print("=" * 78)
            print("FILES FOUND THAT WERE NEVER SAVED TO GOOGLE DRIVE".center(78))
            print("=" * 78)
            print(f"{len(found)} file(s) were written while Drive was disconnected. They are NOT in your")
            print(f"Drive. They have been moved aside (not deleted) to:\n    {stale}")
            for f in found[:25]:
                print("    -", f)
            if len(found) > 25:
                print(f"    ... and {len(found) - 25} more")
            print("They vanish when this session ends. Any Prime export listed above should be")
            print("uploaded again in Step 6 — duplicates of files already in Drive are skipped anyway.")
            print("=" * 78 + "\n")
    # 3. connect afresh
    drive.mount(MOUNT, force_remount=True)
    if not drive_ok():
        raise SystemExit("Google Drive did not connect. Use Runtime -> Disconnect and delete runtime, "
                         "then run Step 1 again and allow access in the sign-in window.")
    print("Google Drive connected.")

if IN_COLAB:
    _mount_drive()
else:
    DRIVE_ROOT = os.path.abspath("./GST_Prime_Investigation")
    print("Not running in Colab — working locally at", DRIVE_ROOT)
os.makedirs(DRIVE_ROOT, exist_ok=True)

def list_cases(root=DRIVE_ROOT):
    out = []
    for d in sorted(os.listdir(root)):
        p = os.path.join(root, d)
        if not os.path.isdir(p):
            continue
        meta, raw, rep = {}, os.path.join(p, "01_RAW_UPLOADS"), os.path.join(p, "03_REPORTS")
        cf = os.path.join(p, "case.json")
        if os.path.exists(cf):
            try:
                meta = json.load(open(cf, encoding="utf-8"))
            except Exception:
                meta = {}
        nfiles = len([f for f in os.listdir(raw) if f.lower().endswith((".xlsx", ".xls"))]) \
                 if os.path.isdir(raw) else 0
        reports = sorted(os.listdir(rep)) if os.path.isdir(rep) else []
        out.append(dict(gstin=d, name=meta.get("taxpayer_name", ""),
                        files=nfiles, reports=len(reports),
                        last_report=reports[-1] if reports else "",
                        opened=meta.get("last_opened", "")))
    return out

_cases = list_cases()
if not _cases:
    print("No taxpayer on file yet. Go to Step 2 and enter the first GSTIN.")
else:
    print(f"{len(_cases)} taxpayer(s) already on file\n")
    print(f"{'GSTIN':<18}{'TAXPAYER':<42}{'FILES':>6}{'REPORTS':>9}  LAST REPORT")
    print("-" * 110)
    for c in _cases:
        print(f"{c['gstin']:<18}{(c['name'] or '(name not recorded)')[:40]:<42}"
              f"{c['files']:>6}{c['reports']:>9}  {c['last_report'][:46]}")
    print("\nTo add files to one of these, put its GSTIN in Step 2 — the name comes back automatically.")


## Step 2 · Enter the taxpayer

**Type the GSTIN and the name into the boxes on the right, then run the cell.** You do not need to edit any code.

- An existing GSTIN re-opens that case and brings back the name and profile.
- A new GSTIN creates its folder.
- The cell refuses to run on a blank or malformed GSTIN, so files cannot be filed under the wrong taxpayer.

The profile boxes are optional — anything entered there is printed on the index sheet of the report.

In [ ]:
#@title  Taxpayer details  { run: "auto", display-mode: "form" }
#@markdown ### Required
GSTIN         = ""  #@param {type:"string"}
TAXPAYER_NAME = ""  #@param {type:"string"}
#@markdown ### Optional — printed on the index sheet of the report
CONSTITUTION         = ""  #@param {type:"string"}
NATURE_OF_BUSINESS   = ""  #@param {type:"string"}
REGISTERED_ADDRESS   = ""  #@param {type:"string"}
CENTRAL_JURISDICTION = ""  #@param {type:"string"}
STATE_JURISDICTION   = ""  #@param {type:"string"}
DATE_OF_REGISTRATION = ""  #@param {type:"string"}
REGISTRATION_STATUS  = ""  #@param {type:"string"}
#@markdown ---
#@markdown A light red cell in the report means *this value was blank in the source* or
#@markdown *this report was not supplied*. Keep it ticked while investigating; untick it for a
#@markdown clean copy going up to the Commissioner.
MARK_MISSING_LIGHT_RED = True  #@param {type:"boolean"}
REPORT_VERSION = "1.8"  #@param {type:"string"}

# ---- if the form is not available (plain Jupyter), ask at the prompt ----
if not str(GSTIN).strip():
    try:
        GSTIN = input("GSTIN: ").strip()
    except Exception:
        pass
if not str(TAXPAYER_NAME).strip():
    try:
        TAXPAYER_NAME = input("Taxpayer name: ").strip()
    except Exception:
        pass

GSTIN = str(GSTIN).strip().upper()
TAXPAYER_NAME = str(TAXPAYER_NAME).strip()

# ---- validate before anything is created --------------------------------
GSTIN_RE = re.compile(r"^[0-9]{2}[A-Z]{5}[0-9]{4}[A-Z][1-9A-Z]Z[0-9A-Z]$")
if not GSTIN:
    raise ValueError("Enter the GSTIN in the form on the right, then run this cell again.")
if not TAXPAYER_NAME:
    raise ValueError("Enter the taxpayer's name in the form on the right, then run this cell again.")
if not GSTIN_RE.match(GSTIN):
    raise ValueError(
        f"'{GSTIN}' is not a valid GSTIN.\n"
        "It must be 15 characters: 2 digits (state code), 5 letters + 4 digits + 1 letter (PAN), "
        "1 entity digit/letter, the letter Z, then 1 check digit or letter.\n"
        "Example: 33AAXCA1234A1ZX")

print(f"Taxpayer : {TAXPAYER_NAME}")
print(f"GSTIN    : {GSTIN}   (state code {GSTIN[:2]}, PAN {GSTIN[2:12]})")


In [ ]:
# ---- open or create the case folder for this GSTIN ---------------------
require_drive()
CASE_ID  = GSTIN
CASE_DIR = os.path.join(DRIVE_ROOT, CASE_ID)
DIRS = {k: os.path.join(CASE_DIR, v) for k, v in {
    "engine": "00_ENGINE", "raw": "01_RAW_UPLOADS",
    "extract": "02_EXTRACTED", "reports": "03_REPORTS"}.items()}
_is_new = not os.path.isdir(CASE_DIR)
for d in [CASE_DIR] + list(DIRS.values()):
    os.makedirs(d, exist_ok=True)
sys.path.insert(0, DIRS["engine"])

CASE_FILE = os.path.join(CASE_DIR, "case.json")
_prev = {}
if os.path.exists(CASE_FILE):
    try:
        _prev = json.load(open(CASE_FILE, encoding="utf-8"))
    except Exception:
        _prev = {}

# a field left blank in the form keeps whatever was recorded last time
PROFILE = {}
for label, val in [("Constitution", CONSTITUTION), ("Nature of business", NATURE_OF_BUSINESS),
                   ("Registered address", REGISTERED_ADDRESS),
                   ("Central jurisdiction", CENTRAL_JURISDICTION),
                   ("State jurisdiction", STATE_JURISDICTION),
                   ("Date of registration", DATE_OF_REGISTRATION),
                   ("Registration status", REGISTRATION_STATUS)]:
    v = str(val).strip() or str(_prev.get("profile", {}).get(label, "")).strip()
    if v:
        PROFILE[label] = v

if _prev.get("taxpayer_name") and _prev["taxpayer_name"] != TAXPAYER_NAME:
    print(f"NOTE: this GSTIN was previously recorded as '{_prev['taxpayer_name']}'. "
          f"It will now be recorded as '{TAXPAYER_NAME}'.")

json.dump({"gstin": GSTIN, "taxpayer_name": TAXPAYER_NAME, "profile": PROFILE,
           "created": _prev.get("created", datetime.datetime.now().isoformat(timespec="seconds")),
           "last_opened": datetime.datetime.now().isoformat(timespec="seconds")},
          open(CASE_FILE, "w", encoding="utf-8"), indent=1, ensure_ascii=False)

_held = [f for f in os.listdir(DIRS["raw"]) if f.lower().endswith((".xlsx", ".xls"))] \
        if os.path.isdir(DIRS["raw"]) else []
bar = "=" * 78
print(bar)
print(("NEW CASE CREATED" if _is_new else "EXISTING CASE RE-OPENED").center(78))
print(bar)
print(f"  GSTIN            {GSTIN}")
print(f"  Taxpayer         {TAXPAYER_NAME}")
for k, v in PROFILE.items():
    print(f"  {k:<17}{v}")
print(f"  Case folder      {CASE_DIR}")
print(f"  Files already in 01_RAW_UPLOADS: {len(_held)}")
print(bar)
print("Everything uploaded from here on is filed under this GSTIN.")


## Step 3 · Engine — parser

Reads a GST-Prime export, works out which report it is from the header columns, and parses it. Blank cells are kept blank. Duplicates are caught by MD5 of the file.

*Written as `gpe_core.py` and copied into this taxpayer's `00_ENGINE/` folder, so it persists between sessions.*

In [ ]:
%%writefile gpe_core.py
# -*- coding: utf-8 -*-
"""
GST-PRIME PRELIMINARY INVESTIGATION ENGINE  —  core
Ingest, detect, parse and validate GST-Prime Analytica exports.
Every parsed value keeps its provenance: file, sheet, row, column.
"""
import os, re, json, hashlib, datetime, shutil
from collections import OrderedDict
import openpyxl

ENGINE_VERSION = "1.7"

# ----------------------------------------------------------------- helpers
MONTHS = {m: i for i, m in enumerate(
    ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], 1)}

def md5(path, buf=1 << 20):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while True:
            b = f.read(buf)
            if not b: break
            h.update(b)
    return h.hexdigest()

def parse_period(v):
    """'Aug-2026' -> ('Aug-2026', 2026, 8). Returns None if not a tax period."""
    if v is None: return None
    s = str(v).strip()
    m = re.fullmatch(r'([A-Za-z]{3})[-/ ](\d{4})', s)
    if not m: return None
    mon = m.group(1).title()
    if mon not in MONTHS: return None
    return (f"{mon}-{m.group(2)}", int(m.group(2)), MONTHS[mon])

def period_key(p):
    t = parse_period(p)
    return (t[1], t[2]) if t else (9999, 99)

def fy_of(p):
    t = parse_period(p)
    if not t: return None
    _, y, m = t
    return f"{y-1}-{str(y)[2:]}" if m <= 3 else f"{y}-{str(y+1)[2:]}"

def fy_long(fy):           # '2021-22' -> '2021-2022'
    a, b = fy.split('-');  return f"{a}-{a[:2]}{b}"

def norm_fy(v):
    """Accept '2021-2022', '2021-22', 2021 -> '2021-22'."""
    if v is None: return None
    s = str(v).strip()
    m = re.fullmatch(r'(\d{4})\s*-\s*(\d{2,4})', s)
    if not m: return None
    a, b = m.group(1), m.group(2)
    return f"{a}-{b[-2:]}"

_NUM_RE = re.compile(r'^-?[\d,]*\.?\d+$')

def to_num(v):
    """Indian-grouped numerics -> float. BLANK STAYS None. '0' stays 0.0."""
    if v is None: return None
    if isinstance(v, bool): return None
    if isinstance(v, (int, float)): return float(v)
    s = str(v).strip()
    if s == '' or s.lower() in ('nan', 'none', '-', 'na', 'n/a'): return None
    s2 = s.replace(',', '').replace('₹', '').replace('Rs.', '').strip()
    if s2.startswith('(') and s2.endswith(')'): s2 = '-' + s2[1:-1]
    if _NUM_RE.fullmatch(s2):
        try: return float(s2)
        except ValueError: return None
    return None

def to_date(v):
    if v is None: return None
    if isinstance(v, datetime.datetime): return v.date()
    if isinstance(v, datetime.date): return v
    s = str(v).strip()
    for f in ('%d-%m-%Y', '%Y-%m-%d', '%d/%m/%Y', '%Y-%m-%d %H:%M:%S', '%d-%b-%Y'):
        try: return datetime.datetime.strptime(s, f).date()
        except ValueError: pass
    return None

def inr(x, dec=0):
    """Indian digit grouping: 10186350 -> '1,01,86,350'. Used in every narrative line."""
    if x is None: return ''
    neg = x < 0; x = abs(float(x))
    whole = int(round(x)) if dec == 0 else int(x)
    frac = '' if dec == 0 else ('%.*f' % (dec, x - whole))[1:]
    s = str(whole)
    if len(s) > 3:
        last3, rest = s[-3:], s[:-3]
        parts = []
        while len(rest) > 2:
            parts.insert(0, rest[-2:]); rest = rest[:-2]
        if rest: parts.insert(0, rest)
        s = ','.join(parts + [last3])
    return ('-' if neg else '') + s + frac


def n0(v):
    """Blank-safe zero for arithmetic. Use ONLY where blank truly means nil."""
    return 0.0 if v is None else float(v)

# ----------------------------------------------------------------- specs
# Each spec: the columns that must be present for a confident match.
# 'key' is the column that identifies a row (tax period / FY / challan).
REPORT_SPECS = OrderedDict([
 ('R3B_TURNOVER', dict(
    must={'Return Period','OutwardTaxable Supplies(Other Than Zero)','Inward Supplies(Reverse Charge)','Total Turnover'},
    key='Return Period', grain='period', fname=r'TURNOVER_OUTWARD_TAX_SUPPLIES',
    title='GSTR-3B turnover and outward taxable supplies')),
 ('R3B_TAX_PAYABLE', dict(
    must={'Return Period','Outward SGST','Outward CGST','Outward IGST','Outward Total GST','Inward Total GST'},
    key='Return Period', grain='period', fname=r'R3B_TAX_PAYABLE_DETAILS',
    title='GSTR-3B tax payable, outward and inward')),
 ('TAX_PAID', dict(
    must={'Return Period','Filing Status','Total TO','Taxable TO','GST OP Total','GST ITC Total'},
    key='Return Period', grain='period', fname=r'TAX_PAID_DETAILS',
    title='GSTR-3B tax paid summary')),
 ('EXCESS_ITC', dict(
    must={'Return Period','SGSTITC Claimed In R3B','SGSTITC Available In R2A','IGSTITC Claimed In R3B','IGSTITC Available In R2A'},
    key='Return Period', grain='period', fname=r'EXCESS_ITC_CLAIMED',
    title='ITC claimed in GSTR-3B against ITC available in GSTR-2A')),
 ('SGST_SETTLEMENT', dict(
    must={'Return Period','SGST Cash Paid','IGST Paid through SGST ITC','SGST Paid through IGST ITC','SGST Settlement'},
    key='Return Period', grain='period', fname=r'SGST_SETTLEMENT_DETAILS',
    title='SGST settlement details')),
 ('HSN', dict(
    must={'Return Period','HSN Code','Taxable Value','Total GST'},
    key='Return Period', grain='period_multi', fname=r'HSN_LIST',
    title='HSN summary (GSTR-1 Table 12)')),
 ('GSTR9', dict(
    must={'Return Period','Date of Filing'},
    key='Return Period', grain='fy', fname=r'GSTR9_FILING_DETAILS',
    title='GSTR-9 annual return filing details')),
 ('R1_MONTHLY', dict(
    must={'Return Period','No Of Sellers','No Of Invoices','Taxable Value','Total GST'},
    key='Return Period', grain='period', fname=r'R1_FILINGS_IN_THE_YEAR',
    title='GSTR-1 filings, month-wise')),
 ('R2A_MONTHLY', dict(
    must={'Return Period','Seller Count','Invoices Count','Taxable Value','Total GST'},
    key='Return Period', grain='period', fname=r'R2A_FILINGS_IN_THE_YEAR',
    title='GSTR-2A filings, month-wise')),
 ('R3B_FILING', dict(
    must={'Return Period','Date Of Filing','Total Turnover','GST Total','GST Input Tax Credit'},
    key='Return Period', grain='period', fname=r'R3B_PAYMENT_SUMMARY_IN_THE_YEAR',
    title='GSTR-3B payment summary with date of filing')),
 ('PAYMENTS', dict(
    must={'Payment Month','Payment Date','CIN','TOTAL'},
    key='CIN', grain='challan', fname=r'PAYMENT_DETAILS',
    title='Cash payment challans')),
 ('R3B_FY_SUMMARY', dict(
    must={'Financial Year','Payment Count','Total Turnover','GST Total','GST Input Tax Credit'},
    key='Financial Year', grain='fy', fname=r'PAYMENTS_GSTR3B',
    title='GSTR-3B payment summary, FY level')),
 ('R2A_FY_SUMMARY', dict(
    must={'Financial Year','GST R2A Count','Seller Count','Invoices Count','Total GST'},
    key='Financial Year', grain='fy', fname=r'GST-R2A_FILING_SUMMARY',
    title='GSTR-2A filing summary, FY level')),
 ('R7_TDS', dict(
    must={'Return Period','No Of Deductors','SGST','CGST','IGST','Total GST'},
    key='Return Period', grain='period', fname=r'R7_FILINGS_IN_THE_YEAR', master=False,
    title='GSTR-7 — tax deducted at source from this taxpayer under section 51, month-wise')),
 ('R7_DETAIL', dict(
    must={'Deductee GSTIN','Trade Name','Taxable Value','SGST','CGST','IGST','Total GST'},
    key='Return Period', grain='period_multi', fname=r'R7_Filing_Details_For_The_Month', master=False,
    period_from_filename=True,
    title='GSTR-7 filing details for one month — party by party, with taxable value')),
 ('R1_FY_SUMMARY', dict(
    must={'Financial Year','Filing Count','No Of Sellers','No Of Invoices','Total GST'},
    key='Financial Year', grain='fy', fname=r'GST-R1_FILING_SUMMARY',
    title='GSTR-1 filing summary, FY level')),
])

DATE_COLS = {'Date of Filing','Date Of Filing','Payment Date'}
TEXT_COLS = {'Deductee GSTIN','Trade Name','Return Period','Financial Year','Filing Status','CIN','Payment Month','HSN Code','Trend',
             'Trend For Other Than Zero Rated','Trend For Zero Rated'}

# ----------------------------------------------------------------- reader
def _find_header(rows):
    """GST-Prime sheets carry a title block then a header row starting 'SNo'."""
    for i, r in enumerate(rows):
        if r and r[0] is not None and str(r[0]).strip().lower() == 'sno':
            return i
    return None

def _declared_records(rows, hdr_i):
    """'Total Records: 12 (All Amounts Are In INR)' -> 12"""
    for r in rows[:hdr_i]:
        for c in r or ():
            if c is None: continue
            m = re.search(r'Total\s+Records\s*:\s*(\d+)', str(c), re.I)
            if m: return int(m.group(1))
    return None

def read_sheet(path):
    wb = openpyxl.load_workbook(path, data_only=True, read_only=True)
    ws = wb[wb.sheetnames[0]]
    rows = [tuple(r) for r in ws.iter_rows(values_only=True)]
    wb.close()
    hdr_i = _find_header(rows)
    if hdr_i is None:
        return None, None, None, rows
    header = [(str(x).strip() if x is not None else '') for x in rows[hdr_i]]
    while header and header[-1] == '': header.pop()
    return hdr_i, header, _declared_records(rows, hdr_i), rows

def _norm_name(filename):
    """Prime saves 'EXCESS ITC CLAIMED FOR SGST-IGST.xlsx' when downloaded, but the same report
    arrives as 'EXCESS_ITC_CLAIMED_FOR_SGST-IGST.xlsx' when re-uploaded. Compare on a form
    where every run of non-alphanumerics is one underscore, so spaces, brackets and dashes
    can never make a correct filename look wrong."""
    return re.sub(r'[^A-Za-z0-9]+', '_', filename).strip('_')

def detect(header, filename):
    """Return (code, score, why). Header signature decides; filename corroborates."""
    filename = _norm_name(filename)
    hs = set(h for h in header if h)
    best, best_score, why = None, 0.0, ''
    for code, sp in REPORT_SPECS.items():
        must = sp['must']
        if not must <= hs:
            continue
        score = len(must) / max(len(hs), 1)
        fn_ok = bool(re.search(sp['fname'].replace('-', '_'), filename, re.I))
        score += 0.5 if fn_ok else 0.0
        if score > best_score:
            best, best_score = code, score
            why = ('header signature matched' + (' and filename agrees' if fn_ok
                   else ' (FILENAME DOES NOT AGREE — header was trusted)'))
    if best is None:
        for code, sp in REPORT_SPECS.items():
            if re.search(sp['fname'].replace('-', '_'), filename, re.I):
                return None, 0.0, (f'filename looks like {code} but the header does not carry '
                                   f'the expected columns — NOT PARSED, needs a look')
    return best, best_score, why

GSTIN_IN_TEXT = re.compile(r'\b(\d{2}[A-Z0-9]{10}\d[A-Z0-9][A-Z0-9])\b')

def subject_gstin(rows, hdr_i):
    """The GSTIN a Prime export names in its title block ('R7 FILING DETAILS OF: 33...'), if any."""
    for r in rows[:hdr_i]:
        for c in r or ():
            if c is None: continue
            m = GSTIN_IN_TEXT.search(str(c).upper())
            if m: return m.group(1)
    return None

def period_from_filename(filename):
    """'R7_Filing_Details_For_The_Month_Mar-2026.xlsx' -> 'Mar-2026'. Used only for a report that
    carries no period inside it; the register says so."""
    m = re.search(r'([A-Za-z]{3})[-_ ](\d{4})(?!.*[A-Za-z]{3}[-_ ]\d{4})', filename)
    if not m: return None
    t = parse_period(f'{m.group(1)}-{m.group(2)}')
    return t[0] if t else None

def parse_rows(header, rows, hdr_i, code, default_period=None):
    """Rows as dicts. BLANK stays None so 'not filed' never reads as 'nil'."""
    sp = REPORT_SPECS[code]
    out = []
    for ri, r in enumerate(rows[hdr_i + 1:], start=hdr_i + 2):
        if not r or r[0] is None: continue
        if str(r[0]).strip().lower() in ('sub total', 'total', 'grand total'): continue
        d, blanks = {}, []
        for ci, col in enumerate(header):
            if not col: continue
            v = r[ci] if ci < len(r) else None
            if v is None or (isinstance(v, str) and v.strip() == ''):
                d[col] = None; blanks.append(col); continue
            if col in DATE_COLS:      d[col] = to_date(v)
            elif col in TEXT_COLS:
                d[col] = (str(int(v)) if isinstance(v, float) and v.is_integer() else str(v).strip())
            else:                     d[col] = to_num(v)
        d['_row'] = ri
        d['_blank_cols'] = blanks
        if default_period and not d.get('Return Period'):
            d['Return Period'] = default_period
        if sp['grain'] in ('period', 'period_multi'):
            t = parse_period(d.get('Return Period'))
            if not t: continue
            d['_period'], d['_fy'] = t[0], fy_of(t[0])
        elif sp['grain'] == 'fy':
            f = norm_fy(d.get('Financial Year') or d.get('Return Period'))
            if f is None: continue
            d['_fy'] = f
        out.append(d)
    return out

# ----------------------------------------------------------------- restatements
_SKIP_FIELD = re.compile(r'^SNo$|Growth|Trend', re.I)   # position / derived columns, not data

def diff_rows(old, new):
    """Fields whose VALUE differs between two versions of the same record.
    SNo is only the row position in that export, and Growth / Trend are derived, so neither counts.
    A difference under Re.1 is rounding, not a restatement."""
    out = []
    for k, b in new.items():
        if k.startswith('_') or _SKIP_FIELD.search(k):
            continue
        a = old.get(k)
        if a is None and b is None:
            continue
        if isinstance(a, (int, float)) and isinstance(b, (int, float)):
            if abs(a - b) < 1.0:
                continue
        elif a == b:
            continue
        out.append((k, a, b))
    return out

def is_dynamic(code, field):
    """GSTR-2A moves on its own: it changes whenever a supplier files late or amends. A change
    in a 2A figure is therefore expected and says nothing about the taxpayer's own returns."""
    if code in ('R2A_MONTHLY', 'R2A_FY_SUMMARY'):
        return True
    if code == 'EXCESS_ITC' and ('R2A' in field or 'Excess' in field):
        return True
    return False

def classify_change(code, field, old, new):
    """(nature, level, reading). Wording is deliberately cautious: the export alone cannot say
    WHY a value moved, only that it did."""
    if old is None and new is not None:
        nature = 'POPULATED'
    elif old is not None and new is None:
        nature = 'BLANKED'
    else:
        nature = 'CHANGED'
    if is_dynamic(code, field):
        return (nature, 'INFO',
                'GSTR-2A is a dynamic statement and moves when suppliers file late or amend. '
                'Expected. Not a restatement by the taxpayer.')
    if nature == 'POPULATED':
        return (nature, 'NOTE',
                'Blank or absent in the earlier export and present now. The return has most likely '
                'been filed, or the data populated, since the earlier export was taken.')
    if nature == 'BLANKED':
        return (nature, 'REVIEW',
                'Held a value in the earlier export and is blank now. Unusual. Check the later '
                'export was not truncated, then check the portal.')
    return (nature, 'REVIEW',
            'A value already held has changed. GSTR-3B and payment challans cannot be revised, so a '
            'change here is unusual. Establish whether the earlier export was taken before the period '
            'was final, whether the taxpayer amended (GSTR-1 amendments are made through later '
            'returns), or whether Prime refreshed its own data. Verify on the portal before relying '
            'on either figure.')

def key_label(code, row):
    """A human-readable name for a record: the tax period, FY, or challan number."""
    g = REPORT_SPECS[code]['grain']
    if g == 'period':       return str(row.get('_period'))
    if g == 'period_multi':
        if row.get('Deductee GSTIN'): return f"{row.get('_period')} / {row.get('Deductee GSTIN')} / Rs.{row.get('Taxable Value')}"
        return f"{row.get('_period')} / HSN {row.get('HSN Code')}"
    if g == 'fy':           return f"FY {row.get('_fy')}"
    if g == 'challan':      return f"{row.get('Payment Month')} / {row.get('CIN')}"
    return str(row.get('SNo'))

# ----------------------------------------------------------------- ingest
def row_key(code, r):
    """The natural identity of a row, so the same record supplied twice is stored once.

    A byte-identical file is rejected before it is ever parsed. This is the second line of
    defence, for the case that actually corrupts figures: a LATER EXPORT of the same report
    carrying the same records plus a few new ones. Without a row key those overlapping records
    would be appended a second time and every total built on them would roughly double."""
    sp = REPORT_SPECS[code]
    g = sp['grain']
    if g == 'period':
        return ('p', r.get('_period'))
    if g == 'fy':
        return ('f', r.get('_fy'))
    if g == 'period_multi':
        # one period may legitimately carry several HSN lines (or several lines for one party),
        # so the whole line is the key
        return ('m', r.get('_period'), str(r.get('HSN Code') or r.get('Deductee GSTIN')),
                r.get('Taxable Value'), r.get('Total GST'))
    if g == 'challan':
        cin = r.get('CIN')
        if cin:
            return ('c', str(cin).strip())
        return ('c', r.get('Payment Month'), str(r.get('Payment Date')), r.get('TOTAL'))
    return ('x', r.get('SNo'))


class Ingest:
    """Holds every file seen, what it was, and whether it was counted."""
    def __init__(self, case_gstin=None):
        self.case_gstin = (case_gstin or '').strip().upper() or None
        self.register = []     # one row per upload
        self.data = {}         # code -> list of row dicts (deduped, merged)
        self._seen_md5 = {}
        self._index = {}       # code -> {row_key: position in self.data[code]}
        self.changes = []      # every value that a later export changed from an earlier one

    def add(self, path, batch=1):
        name = os.path.basename(path)
        h = md5(path)
        rec = dict(file=name, batch=batch, md5=h, size=os.path.getsize(path),
                   code=None, title=None, rows=0, declared=None,
                   status='', note='', detect_note='')
        if h in self._seen_md5:
            rec.update(status='DUPLICATE - NOT COUNTED',
                       note=f'Byte-identical to {self._seen_md5[h]}. Adds nothing; excluded from every count.')
            self.register.append(rec); return rec
        try:
            hdr_i, header, declared, rows = read_sheet(path)
        except Exception as e:
            rec.update(status='UNREADABLE', note=f'Could not open: {e}')
            self.register.append(rec); return rec
        if hdr_i is None:
            rec.update(status='NOT RECOGNISED', note='No header row beginning "SNo" was found.')
            self.register.append(rec); return rec
        code, score, why = detect(header, name)
        rec['declared'] = declared
        rec['detect_note'] = why
        if code is None:
            rec.update(status='NOT RECOGNISED',
                       note=why or 'Header does not match any known GST-Prime report.')
            self.register.append(rec); return rec
        rec['subject_gstin'] = subject_gstin(rows, hdr_i)
        notes = []
        dperiod = None
        if REPORT_SPECS[code].get('period_from_filename'):
            dperiod = period_from_filename(name)
            if dperiod is None:
                rec.update(code=code, title=REPORT_SPECS[code]['title'], status='NOT RECOGNISED',
                           note='This report carries no tax period inside it, and none could be read from the file name '
                                '(expected e.g. ..._Month_Mar-2026.xlsx). Rename the file with its month and upload again.')
                self.register.append(rec); return rec
            notes.append(f'Tax period {dperiod} taken from the FILE NAME — the export carries no period inside it. '
                         f'Keep the Prime file name unchanged.')
        parsed = parse_rows(header, rows, hdr_i, code, default_period=dperiod)
        rec.update(code=code, title=REPORT_SPECS[code]['title'], rows=len(parsed), status='COUNTED')
        if self.case_gstin and rec['subject_gstin'] and rec['subject_gstin'] != self.case_gstin:
            notes.append(f'WRONG TAXPAYER? This export is titled for GSTIN {rec["subject_gstin"]}, not {self.case_gstin}.')
        if declared is not None and declared != len(parsed):
            notes.append(f'Report header declares {declared} records; {len(parsed)} data rows were '
                         f'found. The parsed rows are used. Verify the export on the portal.')
        self._seen_md5[h] = name

        # ---- merge rows by their natural key: a record already held is REPLACED, never added.
        # A key that repeats INSIDE one file (two identical HSN lines, say) is two real rows, so
        # each occurrence is numbered and matches only the same occurrence in a later export.
        bucket = self.data.setdefault(code, [])
        index = self._index.setdefault(code, {})
        added = replaced = rec_changed = vals_changed = 0
        occ = {}
        if REPORT_SPECS[code]['grain'] == 'period_multi' and bucket:
            # A period carries SEVERAL lines (HSN lines, or one line per deduction). A later export of
            # that period is the whole of it: matching line by line would keep a restated line twice
            # (the old value and the new), and keep a line the later export dropped. So every period
            # this file covers is REPLACED as a block, and the difference is recorded line by line.
            blk = self._replace_period_block(code, name, batch, parsed)
            if blk:
                bucket = self.data[code]; index = self._index[code]
                replaced, added, rec_changed, vals_changed = blk
                notes.append(f'Covers {len({x["_period"] for x in parsed})} period(s) already held from an earlier export; '
                             f'those periods were REPLACED AS A WHOLE by this file ({replaced} line(s) unchanged, '
                             f'{rec_changed} line(s) added or dropped). Nothing is counted twice.'
                             + (' See 09A_RESTATEMENTS.' if rec_changed else ''))
                for x in parsed:
                    row = dict(x, _file=name, _batch=batch)
                    k0 = row_key(code, row); n = occ.get(k0, 0); occ[k0] = n + 1
                    index[k0 + (n,)] = len(bucket); bucket.append(row)
                rec['rows_new'] = added; rec['rows_replaced'] = replaced
                rec['records_changed'] = rec_changed; rec['values_changed'] = vals_changed
                rec['note'] = ' '.join(notes)
                self.register.append(rec); return rec
        for x in parsed:
            row = dict(x, _file=name, _batch=batch)
            k0 = row_key(code, row)
            n = occ.get(k0, 0); occ[k0] = n + 1
            k = k0 + (n,)
            if k in index:
                prev = bucket[index[k]]
                ch = diff_rows(prev, row)
                replaced += 1
                # Two files that arrived in the SAME upload batch give no evidence of which export is
                # newer: they share a timestamp, so the order between them is only filename order.
                uncertain = (prev.get('_batch') is not None and prev.get('_batch') == batch)
                row['_supersedes'] = prev.get('_file')
                row['_history'] = dict(supersedes=prev.get('_file'), changes=ch, uncertain=uncertain)
                if ch:
                    rec_changed += 1
                    vals_changed += len(ch)
                    for f, a0, b0 in ch:
                        nature, level, reading = classify_change(code, f, a0, b0)
                        if uncertain:
                            level = 'REVIEW'
                            reading = ('ORDER UNCERTAIN. Both files arrived in the same upload, so nothing shows which '
                                       'export is newer; the value called "later" is only the one that sorted last by '
                                       'filename. Establish which export was taken more recently, delete the other from '
                                       '01_RAW_UPLOADS and run Step 7 again. ') + reading
                        self.changes.append(dict(
                            code=code, title=REPORT_SPECS[code]['title'], record=key_label(code, row),
                            field=f, old=a0, new=b0, old_file=prev.get('_file'), new_file=name,
                            nature=nature, level=level, reading=reading, order_uncertain=uncertain))
                bucket[index[k]] = row
            else:
                index[k] = len(bucket)
                bucket.append(row)
                added += 1
        rec['rows_new'] = added
        rec['rows_replaced'] = replaced
        rec['records_changed'] = rec_changed
        rec['values_changed'] = vals_changed
        if replaced:
            notes.append(f'{replaced} record(s) in this file were already held from an earlier export '
                         f'and have been REPLACED, not added again. {added} record(s) are new.')
            if vals_changed:
                notes.append(f'{vals_changed} VALUE(S) in {rec_changed} of those records DIFFER from the earlier '
                             f'export — see 09A_RESTATEMENTS.')
        elif added == 0 and parsed:
            notes.append('Every record in this file was already held from an earlier export. '
                         'Nothing was added.')
        rec['note'] = ' '.join(notes)
        self.register.append(rec); return rec

    def _replace_period_block(self, code, name, batch, parsed):
        """Drop every line held for the periods this file covers (from other files) and report the
        line-level difference. Returns None when none of its periods is already held."""
        from collections import Counter
        periods = {x['_period'] for x in parsed}
        bucket = self.data[code]
        old = [r for r in bucket if r.get('_period') in periods]
        if not old:
            return None
        sig = lambda r: row_key(code, r)
        c_old, c_new = Counter(sig(r) for r in old), Counter(sig(x) for x in parsed)
        same = sum((c_old & c_new).values())
        gone, came = c_old - c_new, c_new - c_old
        first = {}
        for r in old: first.setdefault(sig(r), r)
        uncertain = any(r.get('_batch') == batch for r in old)
        n_chg = 0
        for k, n in list(gone.items()) + list(came.items()):
            is_new = k in came
            r = (next(x for x in parsed if sig(x) == k)) if is_new else first[k]
            label = key_label(code, dict(r, _period=r['_period']))
            nature = 'ADDED' if is_new else 'DROPPED'
            level = 'REVIEW' if (not is_new or uncertain) else 'NOTE'
            reading = ('A line present in the later export of this period but not in the earlier one — a return filed or '
                       'amended since, or a corrected line (look for a DROPPED line of the same party in the same period).'
                       if is_new else
                       'A line in the earlier export of this period that the later export no longer carries — amended or '
                       'withdrawn, or a corrected line (look for an ADDED line of the same party in the same period). '
                       'The later export is used.')
            if uncertain:
                reading = ('ORDER UNCERTAIN. Both exports of this period arrived in the same upload, so nothing shows which '
                           'is newer. Delete the one that is not current from 01_RAW_UPLOADS and run Step 7 again. ') + reading
            for _ in range(n):
                n_chg += 1
                self.changes.append(dict(code=code, title=REPORT_SPECS[code]['title'], record=label, field='whole line',
                                         old=None if is_new else 'present', new='present' if is_new else None,
                                         old_file=', '.join(sorted({x['_file'] for x in old})), new_file=name,
                                         nature=nature, level=level, reading=reading, order_uncertain=uncertain))
        self.data[code] = [r for r in bucket if r.get('_period') not in periods]
        idx, occ = {}, {}
        for i, r in enumerate(self.data[code]):
            k0 = row_key(code, r); n = occ.get(k0, 0); occ[k0] = n + 1
            idx[k0 + (n,)] = i
        self._index[code] = idx
        n_new = sum(came.values())
        return same, n_new, n_chg, n_chg

    # ---- access -------------------------------------------------------
    def have(self, code): return code in self.data and len(self.data[code]) > 0

    def by_period(self, code):
        """period -> row. Later files win only if the earlier row is emptier."""
        out = {}
        for r in self.data.get(code, []):
            p = r.get('_period')
            if p is None: continue
            if p not in out or len(r.get('_blank_cols', [])) < len(out[p].get('_blank_cols', [])):
                out[p] = r
        return out

    def by_fy(self, code):
        out = {}
        for r in self.data.get(code, []):
            f = r.get('_fy')
            if f: out[f] = r
        return out

    def periods(self):
        """Every tax period seen in any period-grained report, chronological."""
        s = set()
        for code, sp in REPORT_SPECS.items():
            if sp['grain'] in ('period',) and code in self.data and sp.get('master', True):
                s |= {r['_period'] for r in self.data[code] if r.get('_period')}
        return sorted(s, key=period_key)

    def all_periods(self):
        """Every tax period in any report, including those (GSTR-7) that do not open a row in the
        master. Used for limitation, which runs on every year the evidence touches."""
        s = set(self.periods())
        for code, sp in REPORT_SPECS.items():
            if sp['grain'] in ('period', 'period_multi') and code in self.data:
                s |= {r['_period'] for r in self.data[code] if r.get('_period')}
        return sorted(s, key=period_key)

    def counted(self):  return [r for r in self.register if r['status'] == 'COUNTED']
    def duplicates(self): return [r for r in self.register if r['status'].startswith('DUPLICATE')]
    def problems(self): return [r for r in self.register
                                if r['status'] in ('NOT RECOGNISED', 'UNREADABLE')]


## Step 4 · Engine — checks

The twelve investigation checks. Each one states which reports it needs; if any is missing it returns `available=False` and names what is absent rather than producing a number.

*Written as `gpe_checks.py` and copied into this taxpayer's `00_ENGINE/` folder, so it persists between sessions.*

In [ ]:
%%writefile gpe_checks.py
# -*- coding: utf-8 -*-
"""
GST-PRIME PRELIMINARY INVESTIGATION ENGINE  —  checks
Each check states which reports it needs. If any is absent the check returns
available=False and names what is missing, instead of inventing a result.
"""
import datetime, re
from gpe_core import (REPORT_SPECS, parse_period, period_key, fy_of, fy_long,
                      n0, to_date, inr)

CHECKS_VERSION = "1.2"
FY_ORDER = ['2021-22','2022-23','2023-24','2024-25','2025-26','2026-27',
            '2027-28','2028-29','2029-30']

def _fys(periods):
    seen = []
    for p in periods:
        f = fy_of(p)
        if f and f not in seen: seen.append(f)
    return sorted(seen, key=lambda f: int(f.split('-')[0]))

def _need(ing, *codes):
    miss = [c for c in codes if not ing.have(c)]
    return miss

def due_date(p):
    _, y, m = parse_period(p)
    m += 1
    if m > 12: m, y = 1, y + 1
    return datetime.date(y, m, 20)

# ------------------------------------------------------------------ master
def unfiled_periods(ing):
    """Periods for which GSTR-3B itself is shown as not filed (typically the month of
    registration). They are not tax periods with a return, so they are kept OUT of the
    master, out of every period count, and out of the NIL / dormancy tests. Otherwise a
    period with no return at all is silently counted as a nil-turnover period."""
    tp = ing.by_period('TAX_PAID')
    return sorted([p for p, r in tp.items()
                   if (r.get('Filing Status') or '').strip().lower().startswith('not')],
                  key=period_key)

def build_master(ing):
    """One row per tax period, every figure traced to its source report."""
    skip = set(unfiled_periods(ing))
    per = [p for p in ing.periods() if p not in skip]
    tp, tx, to, it, se = (ing.by_period(c) for c in
        ('TAX_PAID','R3B_TAX_PAYABLE','R3B_TURNOVER','EXCESS_ITC','SGST_SETTLEMENT'))
    rows = []
    for p in per:
        a, b, c, d, e = tp.get(p,{}), tx.get(p,{}), to.get(p,{}), it.get(p,{}), se.get(p,{})
        r2a_blank = ('SGSTITC Available In R2A' in d.get('_blank_cols', []) or
                     'IGSTITC Available In R2A' in d.get('_blank_cols', []))
        rows.append(dict(
            period=p, fy=fy_of(p),
            filing_status=a.get('Filing Status'),
            turnover=n0(a.get('Total TO')), taxable=n0(a.get('Taxable TO')),
            out_sgst=n0(b.get('Outward SGST')), out_cgst=n0(b.get('Outward CGST')),
            out_igst=n0(b.get('Outward IGST')), out_total=n0(b.get('Outward Total GST')),
            op_total=n0(a.get('GST OP Total')),
            rcm_value=n0(c.get('Inward Supplies(Reverse Charge)')),
            itc=n0(a.get('GST ITC Total')),
            sgst_claim=n0(d.get('SGSTITC Claimed In R3B')), sgst_avail=n0(d.get('SGSTITC Available In R2A')),
            sgst_excess=n0(d.get('SGSTExcess Claimed')),
            igst_claim=n0(d.get('IGSTITC Claimed In R3B')), igst_avail=n0(d.get('IGSTITC Available In R2A')),
            igst_excess=n0(d.get('IGSTExcess Claimed')),
            cash_sgst=n0(a.get('SGST Cash Paid')), settlement=n0(a.get('SGST Settlement')),
            gross_sgst=n0(a.get('Gross SGST Collected')),
            r2a_not_available=r2a_blank,
        ))
    for r in rows:
        r['rcm_tax'] = r['op_total'] - r['out_total']
        r['nil'] = (r['turnover'] == 0)
        r['itc_in_nil'] = r['itc'] if r['nil'] else 0.0
    return rows

# ------------------------------------------------------- 1. GSTR-1 vs 3B
def chk_r1_vs_r3b(ing, master):
    miss = _need(ing, 'R1_MONTHLY', 'R3B_TAX_PAYABLE', 'TAX_PAID')
    if miss: return dict(available=False, missing=miss)
    r1 = ing.by_period('R1_MONTHLY')
    rows, short, notfiled, noreturn = [], 0.0, [], []
    nf_to = nf_tax = 0.0
    for m in master:
        p = m['period']; row = r1.get(p)
        r3b_absent = (m['filing_status'] or '').strip().lower().startswith('not')
        if r3b_absent:
            # No GSTR-3B either: this is a period with no return of any kind,
            # not a GSTR-1 gap in a month that was otherwise reported.
            st, v, t = 'NO RETURN OF ANY KIND', None, None
            noreturn.append(p)
        elif row is None:
            st, v, t = 'NOT FILED - no return on record', None, None
        elif row.get('Total GST') is None:
            st, v, t = 'NOT FILED - row blank', None, None
        elif n0(row.get('Total GST')) == 0 and n0(row.get('Taxable Value')) == 0:
            st, v, t = 'FILED - NIL', 0.0, 0.0
        else:
            st, v, t = 'FILED - WITH DATA', n0(row.get('Taxable Value')), n0(row.get('Total GST'))
        if st.startswith('NOT FILED'):
            notfiled.append(p); nf_to += m['turnover']; nf_tax += m['out_total']
        d = None if t is None else t - m['out_total']
        if d and d > 1: short += d
        rows.append(dict(period=p, fy=m['fy'], status=st, r1_value=v, r3b_value=m['turnover'],
                         r1_tax=t, r3b_tax=m['out_total'], diff=d,
                         short=max(0.0, d) if d else 0.0,
                         sellers=None if row is None else row.get('No Of Sellers'),
                         invoices=None if row is None else row.get('No Of Invoices')))
    return dict(available=True, rows=rows, short_total=short, not_filed=notfiled,
                not_filed_turnover=nf_to, not_filed_tax=nf_tax, no_return=noreturn)

# ------------------------------------------------- 2. excess ITC (demand)
def chk_excess_itc(ing, master):
    miss = _need(ing, 'EXCESS_ITC')
    if miss: return dict(available=False, missing=miss)
    rows, net, gross = [], 0.0, 0.0
    for m in master:
        s, i = m['sgst_excess'], m['igst_excess']
        tot = s * 2 + i                      # CGST mirrors SGST on intra-State supply
        pos = max(0.0, s) * 2 + max(0.0, i)
        net += tot; gross += pos
        rows.append(dict(period=m['period'], fy=m['fy'],
                         sgst_claim=m['sgst_claim'], sgst_avail=m['sgst_avail'], sgst_excess=s,
                         cgst_excess=s, igst_claim=m['igst_claim'], igst_avail=m['igst_avail'],
                         igst_excess=i, total=tot, positive=pos,
                         r2a_not_available=m['r2a_not_available']))
    return dict(available=True, rows=rows, net=net, gross=gross,
                months_excess=sum(1 for r in rows if r['sgst_excess'] > 0.5 or r['igst_excess'] > 0.5),
                months_excess_sgst=sum(1 for r in rows if r['sgst_excess'] > 0.5),
                months_excess_igst=sum(1 for r in rows if r['igst_excess'] > 0.5))

# --------------------------------------- 3. ITC vs 2A, independent route
def chk_itc_vs_2a(ing, master):
    miss = _need(ing, 'R2A_MONTHLY', 'TAX_PAID')
    if miss: return dict(available=False, missing=miss)
    r2 = ing.by_period('R2A_MONTHLY')
    rows, gap, untested = [], 0.0, 0.0
    for m in master:
        a = r2.get(m['period'])
        if a is None:
            rows.append(dict(period=m['period'], fy=m['fy'], present=False, sellers=None,
                             invoices=None, taxable=None, avail=None, itc=m['itc'], gap=None))
            untested += m['itc']; continue
        av = n0(a.get('Total GST')); g = m['itc'] - av; gap += g
        rows.append(dict(period=m['period'], fy=m['fy'], present=True,
                         sellers=a.get('Seller Count'), invoices=a.get('Invoices Count'),
                         taxable=a.get('Taxable Value'), avail=av, itc=m['itc'], gap=g))
    ann = {}
    if ing.have('R2A_FY_SUMMARY') and ing.have('R3B_FY_SUMMARY'):
        f2, f3 = ing.by_fy('R2A_FY_SUMMARY'), ing.by_fy('R3B_FY_SUMMARY')
        for f in set(f2) | set(f3):
            ann[f] = dict(avail=n0(f2.get(f, {}).get('Total GST')),
                          itc=n0(f3.get(f, {}).get('GST Input Tax Credit')))
            ann[f]['gap'] = ann[f]['itc'] - ann[f]['avail']
    return dict(available=True, rows=rows, gap=gap, untested_itc=untested, annual=ann)

# ------------------------------------------------------------ 4. dormancy
def chk_dormancy(ing, master):
    if not ing.have('TAX_PAID'): return dict(available=False, missing=['TAX_PAID'])
    run = 0; runs = []; cur = []
    for m in master:
        if m['nil']:
            cur.append(m['period']); run += 1
        else:
            if len(cur) >= 3: runs.append((cur[0], cur[-1], len(cur)))
            cur, run = [], 0
    if len(cur) >= 3: runs.append((cur[0], cur[-1], len(cur)))
    longest = max(runs, key=lambda x: x[2]) if runs else None
    itc_nil = sum(m['itc_in_nil'] for m in master)
    after = 0.0
    if longest:
        k = period_key(longest[0])
        after = sum(m['itc_in_nil'] for m in master if period_key(m['period']) >= k)
    return dict(available=True, runs=runs, longest=longest,
                itc_in_nil_all=itc_nil, itc_in_nil_from_longest=after,
                nil_months=sum(1 for m in master if m['nil']))

# ----------------------------------------------------------- 5. Rule 86B
def chk_rule86b(ing, master, threshold=5_000_000, floor=0.01):
    if not ing.have('TAX_PAID'): return dict(available=False, missing=['TAX_PAID'])
    rows, flagged = [], []
    for m in master:
        cash = m['cash_sgst'] * 2            # CGST cash taken as equal; IGST cash not captured
        pct = (cash / m['op_total']) if m['op_total'] > 0 else None
        hit = (m['taxable'] > threshold and m['op_total'] > 0 and pct is not None and pct < floor)
        if hit: flagged.append(m['period'])
        rows.append(dict(period=m['period'], taxable=m['taxable'], op=m['op_total'],
                         cash_sgst=m['cash_sgst'], cash_proxy=cash, pct=pct,
                         applicable=m['taxable'] > threshold, flagged=hit))
    return dict(available=True, rows=rows, flagged=flagged)

# ----------------------------------------------------------------- 6. RCM
def chk_rcm(ing, master):
    miss = _need(ing, 'R3B_TURNOVER', 'R3B_TAX_PAYABLE', 'TAX_PAID')
    if miss: return dict(available=False, missing=miss)
    rows = []; declared = 0.0; tax = 0.0; last = None; gap_periods = []
    for m in master:
        rate = (m['rcm_tax'] / m['rcm_value']) if m['rcm_value'] else None
        declared += m['rcm_value']; tax += m['rcm_tax']
        if m['rcm_value'] > 0: last = m['period']
        if m['rcm_value'] > 0 and abs(m['rcm_tax']) < 1:
            gap_periods.append((m['period'], m['rcm_value']))
        rows.append(dict(period=m['period'], value=m['rcm_value'], op=m['op_total'],
                         out=m['out_total'], tax=m['rcm_tax'], rate=rate))
    since = 0
    if last:
        k = period_key(last)
        since = sum(1 for m in master if period_key(m['period']) > k)
    return dict(available=True, rows=rows, declared=declared, tax=tax,
                last_period=last, silent_periods=since, value_without_tax=gap_periods)

# ------------------------- 7. credit notes with no original supply (NEW)
def chk_credit_notes(ing, master):
    """A negative outward declaration is tested against every positive supply
    ever declared under the same HSN, and against total positive turnover."""
    if not ing.have('R3B_TAX_PAYABLE'): return dict(available=False, missing=['R3B_TAX_PAYABLE'])
    pos_turnover = sum(m['turnover'] for m in master if m['turnover'] > 0)
    hsn_pos, hsn_rows = {}, ing.data.get('HSN', [])
    for h in hsn_rows:
        code = str(h.get('HSN Code'))
        v = n0(h.get('Taxable Value'))
        if v > 0: hsn_pos.setdefault(code, []).append((h['_period'], v, n0(h.get('Total GST'))))
    events = []
    for m in master:
        if m['turnover'] >= 0 and m['out_total'] >= 0: continue
        codes = [str(h.get('HSN Code')) for h in hsn_rows
                 if h.get('_period') == m['period'] and n0(h.get('Taxable Value')) < 0]
        matched = {c: hsn_pos.get(c, []) for c in codes}
        events.append(dict(period=m['period'], fy=m['fy'], turnover=m['turnover'],
                           tax=m['out_total'], op_total=m['op_total'], hsn=codes,
                           prior_positive_same_hsn=matched,
                           pct_of_all_positive_turnover=(abs(m['turnover'])/pos_turnover
                                                         if pos_turnover else None)))
    # negative declared in GSTR-1 but not carried into 3B
    r1only = []
    if ing.have('R1_MONTHLY'):
        r1 = ing.by_period('R1_MONTHLY')
        for m in master:
            row = r1.get(m['period'])
            if row is None or row.get('Total GST') is None: continue
            t = n0(row.get('Total GST'))
            if t < -1 and m['out_total'] >= -1:
                r1only.append(dict(period=m['period'], r1_tax=t,
                                   r1_value=n0(row.get('Taxable Value')),
                                   r3b_tax=m['out_total'],
                                   inconsistent=(n0(row.get('Taxable Value')) >= 0),
                                   docs=row.get('No Of Invoices')))
    return dict(available=True, events=events, r1_only=r1only,
                positive_turnover_total=pos_turnover)

# -------------------------------------------------------- 8. return filing
def chk_return_filing(ing, master, r1res):
    if not ing.have('R3B_FILING'): return dict(available=False, missing=['R3B_FILING'])
    fl = ing.by_period('R3B_FILING')
    rows, late, delay = [], [], 0
    r1map = {r['period']: r['status'] for r in r1res['rows']} if r1res.get('available') else {}
    r2 = ing.by_period('R2A_MONTHLY') if ing.have('R2A_MONTHLY') else {}
    for m in master:
        f = fl.get(m['period'], {})
        d = f.get('Date Of Filing')
        dd = (d - due_date(m['period'])).days if d else None
        if dd and dd > 0: late.append((m['period'], d, dd)); delay += dd
        rows.append(dict(period=m['period'], fy=m['fy'], filed=d, due=due_date(m['period']),
                         delay=max(0, dd) if dd is not None else None,
                         r1_status=r1map.get(m['period'], 'not tested'),
                         r2a_present=m['period'] in r2,
                         turnover=m['turnover'], op=m['op_total']))
    return dict(available=True, rows=rows, late=late, total_delay=delay)

# ----------------------------------------------------------------- 9. cash
def chk_cash(ing, master):
    if not ing.have('PAYMENTS'): return dict(available=False, missing=['PAYMENTS'])
    pay = sorted(ing.data['PAYMENTS'], key=lambda r: (r.get('Payment Date') or datetime.date(1900,1,1)))
    tot = dict(SGST=0.0, CGST=0.0, IGST=0.0, TOTAL=0.0)
    by_fy = {}
    for r in pay:
        for k in tot: tot[k] += n0(r.get(k))
        d = r.get('Payment Date')
        if d:
            f = f"{d.year}-{str(d.year+1)[2:]}" if d.month > 3 else f"{d.year-1}-{str(d.year)[2:]}"
            by_fy[f] = by_fy.get(f, 0.0) + n0(r.get('TOTAL'))
    op = sum(m['op_total'] for m in master)
    itc = sum(m['itc'] for m in master)
    return dict(available=True, challans=pay, totals=tot, by_fy=by_fy,
                output_tax=op, itc=itc,
                cash_pct=(tot['TOTAL']/op if op else None),
                count=len(pay))

# ---------------------------------------------------------- 10. HSN drift
def chk_hsn(ing):
    if not ing.have('HSN'): return dict(available=False, missing=['HSN'])
    rows = sorted(ing.data['HSN'], key=lambda r: period_key(r['_period']))
    seq = []
    for r in rows:
        seq.append((r['_period'], str(r.get('HSN Code')), n0(r.get('Taxable Value')), n0(r.get('Total GST'))))
    codes = []
    for _, c, _, _ in seq:
        if c not in codes: codes.append(c)
    return dict(available=True, rows=seq, codes=codes)

# ------------------------------------------------------ 11. annual return
def chk_annual(ing, master):
    filed = {r['_fy']: r.get('Date of Filing') for r in ing.data.get('GSTR9', [])}
    by_fy = {}
    for m in master:
        by_fy[m['fy']] = by_fy.get(m['fy'], 0.0) + m['turnover']
    out = []
    for f in sorted(by_fy, key=lambda x: int(x.split('-')[0])):
        t = by_fy[f]
        out.append(dict(fy=f, turnover=t, gstr9_required=t > 20_000_000,
                        gstr9c_required=t > 50_000_000,
                        gstr9_filed=filed.get(f), on_record=f in filed))
    return dict(available=ing.have('GSTR9'), rows=out,
                missing=[] if ing.have('GSTR9') else ['GSTR9'])

# ----------------------------------------------------------- 12. FY summary
def fy_summary(master):
    out = {}
    for m in master:
        a = out.setdefault(m['fy'], dict(periods=0, nil=0, turnover=0.0, op=0.0, itc=0.0,
                                         cash=0.0, gross_sgst=0.0, sgst_exc=0.0,
                                         igst_exc=0.0, itc_nil=0.0))
        a['periods'] += 1; a['nil'] += 1 if m['nil'] else 0
        a['turnover'] += m['turnover']; a['op'] += m['op_total']; a['itc'] += m['itc']
        a['cash'] += m['cash_sgst']; a['gross_sgst'] += m['gross_sgst']
        a['sgst_exc'] += m['sgst_excess']; a['igst_exc'] += m['igst_excess']
        a['itc_nil'] += m['itc_in_nil']
    return out

# ------------------------------------------------------- 13. limitation
def limitation(fys, today=None):
    """Indicative only. Extension notifications must be checked separately."""
    today = today or datetime.date.today()
    out = []
    for f in sorted(fys, key=lambda x: int(x.split('-')[0])):
        end = int(f.split('-')[0]) + 1
        ar_due = datetime.date(end, 12, 31)                 # GSTR-9 due date
        s73_order = ar_due.replace(year=ar_due.year + 3)
        s73_scn = s73_order - datetime.timedelta(days=92)
        s74_order = ar_due.replace(year=ar_due.year + 5)
        s74_scn = s74_order - datetime.timedelta(days=183)
        is74a = end >= 2025                                  # FY 2024-25 onwards: s.73(12)/74(12) stop, s.74A runs
        m42 = ar_due.month + 42
        s74a_scn = datetime.date(ar_due.year + (m42 - 1) // 12, (m42 - 1) % 12 + 1, 30 if ((m42 - 1) % 12 + 1) in (4, 6, 9, 11) else 31) \
                   if ((m42 - 1) % 12 + 1) != 2 else datetime.date(ar_due.year + (m42 - 1) // 12, 2, 28)
        s74a_order = s74a_scn.replace(year=s74a_scn.year + 1)
        out.append(dict(fy=f, annual_return_due=ar_due, is74a=is74a,
                        s74a_scn_by=s74a_scn if is74a else None, s74a_order_by=s74a_order if is74a else None,
                        s74a_open=(today <= s74a_scn) if is74a else None,
                        s73_scn_by=s73_scn, s73_order_by=s73_order,
                        s73_open=today <= s73_scn,
                        s74_scn_by=s74_scn, s74_order_by=s74_order,
                        s74_open=today <= s74_scn,
                        regime='Section 74A applies (FY 2024-25 onwards)' if end >= 2025
                               else 'Sections 73 / 74 apply'))
    return out

# ------------------------------------------------- 14. GSTR-7 TDS (NEW v1.5)
R3B_CODES = ('TAX_PAID', 'R3B_TAX_PAYABLE', 'R3B_TURNOVER', 'R3B_FILING')
TDS_RATE = 0.02                 # s.51(1): 1% CGST + 1% SGST, or 2% IGST, of the value of supply
RULE86B_LIMIT = 5_000_000       # Rule 86B: taxable supply in a month above Rs.50 lakh
GSTR9_LIMIT, GSTR9C_LIMIT = 20_000_000, 50_000_000

def chk_tds(ing, master, today=None):
    """GSTR-7: tax deducted at source from THIS taxpayer by Government departments, local
    authorities and Government agencies (s.51, Notification 50/2018-CT). The deductor pays the
    supplier the contract value less 2%, and the 2% is credited to the supplier's electronic cash
    ledger (Rule 87(9)). So every rupee of TDS proves a payment received for a taxable supply of
    50 times that amount, excluding the tax on the invoice (s.51(2)).

    What that proves, and what it does not:
      * It is a FLOOR on the taxpayer's turnover with Government. Supplies to anyone else, and
        Government contracts below Rs.2.5 lakh, carry no TDS and are not in it.
      * Payment month is not invoice month. For services (works contract is a service) the
        time of supply is the earlier of invoice and payment (s.13(2)), and for goods the invoice
        is issued on removal (s.31(1)), so a supply paid for by month M must have been declared
        by month M at the latest. The firm test is therefore CUMULATIVE within the financial year,
        not month against month. The one exception is an advance received for goods, which is
        not taxable on receipt (Notification 66/2017-CT).
    """
    if not ing.have('R7_TDS'):
        return dict(available=False, missing=['R7_TDS'])
    r7 = ing.by_period('R7_TDS')
    have_3b = any(ing.have(c) for c in R3B_CODES)
    have_r1 = ing.have('R1_MONTHLY')
    mm = {m['period']: m for m in master}
    unfiled = set(unfiled_periods(ing))
    r1 = ing.by_period('R1_MONTHLY') if have_r1 else {}
    rows = []
    cum_tds_val, cum_3b = {}, {}
    # 3B cumulated from the first period of each FY, so a supply invoiced before the first TDS month
    # is still counted in the taxpayer's favour.
    master_by_fy = {}
    for m in master:
        master_by_fy.setdefault(m['fy'], []).append(m)
    for p in sorted(r7, key=period_key):
        x = r7[p]; f = fy_of(p)
        sg, cg, ig, ce = (x.get(k) for k in ('SGST', 'CGST', 'IGST', 'CESS'))
        tot = x.get('Total GST')
        heads = n0(sg) + n0(cg) + n0(ig)
        # Total GST is used where present: Prime rounds each head to the rupee but totals the
        # unrounded heads, so the total is the more exact figure. Cess is not deducted under s.51.
        value = ((n0(tot) - n0(ce)) if tot is not None else heads) / TDS_RATE
        cum_tds_val[f] = cum_tds_val.get(f, 0.0) + value
        # 3B position for this period
        today_ = today or datetime.date.today()
        if not have_3b:
            st3 = None
        elif p not in mm and due_date(p) >= today_:
            st3 = 'NOT YET DUE'          # GSTR-3B for this month is due on the 20th of the next month
        elif p in unfiled:
            st3 = 'NOT FILED'
        elif p in mm:
            st3 = 'FILED - NIL' if mm[p]['turnover'] == 0 else 'FILED'
        else:
            st3 = 'NO GSTR-3B ON RECORD'
        m = mm.get(p)
        cum = None
        if have_3b:
            cum = sum(z['taxable'] for z in master_by_fy.get(f, []) if period_key(z['period']) <= period_key(p))
        # GSTR-1 position
        if not have_r1:
            st1, v1 = None, None
        else:
            rr = r1.get(p)
            _, y_, m_ = parse_period(p)
            r1_due = datetime.date(y_ + (m_ == 12), (m_ % 12) + 1, 11)   # monthly GSTR-1: 11th of the next month
            if (rr is None or rr.get('Total GST') is None) and r1_due >= today_:
                st1, v1 = 'NOT YET DUE', None
            elif rr is None or rr.get('Total GST') is None:
                st1, v1 = 'NOT FILED', None
            else:
                v1 = n0(rr.get('Taxable Value'))
                st1 = 'FILED - NIL' if v1 == 0 and n0(rr.get('Total GST')) == 0 else 'FILED'
        obs = []
        if tot is not None and abs(heads + n0(ce) - n0(tot)) > 1.0:
            obs.append(f'Heads add to {heads + n0(ce):,.0f} but Total GST reads {n0(tot):,.0f}')
        if n0(sg) != n0(cg):
            obs.append('CGST and SGST deducted differ — check the deductor-wise detail')
        if st3 in ('NOT FILED', 'NO GSTR-3B ON RECORD'):
            obs.append('TDS deducted but no GSTR-3B for the month')
        elif st3 == 'FILED - NIL':
            obs.append('GSTR-3B filed NIL in a month in which the deductors paid for supplies')
        if m is not None and n0(cg) > m['out_cgst'] + 1:
            obs.append('CGST deducted at 1% exceeds the entire CGST declared in GSTR-3B')
        if st1 in ('NOT FILED',):
            obs.append('No GSTR-1 for a month with TDS payments — B2B invoices to the deductors not on record')
        if value > RULE86B_LIMIT:
            obs.append('Value paid for exceeds Rs.50 lakh: Rule 86B threshold crossed on supplies to the deductors alone')
        rows.append(dict(period=p, fy=f, deductors=x.get('No Of Deductors'), sgst=sg, cgst=cg, igst=ig, cess=ce,
                         total=tot, value=value, cum_value=cum_tds_val[f],
                         r3b_status=st3, r3b_taxable=(m['taxable'] if m else None), cum_3b=cum,
                         cum_short=(max(0.0, cum_tds_val[f] - cum) if cum is not None and st3 != 'NOT YET DUE' else None),
                         r3b_out_cgst=(m['out_cgst'] if m else None),
                         r1_status=st1, r1_value=v1, obs=obs))
    # financial-year roll-up
    fys = {}
    for r in rows:
        a = fys.setdefault(r['fy'], dict(months=0, tds=0.0, sgst=0.0, value=0.0, first=r['period'], last=r['period'],
                                         over50=[]))
        a['months'] += 1; a['tds'] += r['value'] * TDS_RATE
        a['sgst'] += n0(r['sgst']); a['value'] += r['value']; a['last'] = r['period']
        if r['value'] > RULE86B_LIMIT: a['over50'].append(r['period'])
    for f, a in fys.items():
        ms = master_by_fy.get(f, [])
        a['r3b_taxable'] = sum(z['taxable'] for z in ms) if have_3b else None
        a['r3b_months'] = len(ms)
        a['itc'] = sum(z['itc'] for z in ms) if have_3b else None
        a['cash_sgst'] = sum(z['cash_sgst'] for z in ms) if ing.have('TAX_PAID') else None
        a['r1_value'] = (sum(n0(r1[p].get('Taxable Value')) for p in r1 if fy_of(p) == f) if have_r1 else None)
        # a month whose return is not yet due is left out of the comparison, not counted as a shortfall
        a['value_due'] = sum(r['value'] for r in rows if r['fy'] == f and r['r3b_status'] != 'NOT YET DUE')
        a['value_due_r1'] = sum(r['value'] for r in rows if r['fy'] == f and r['r1_status'] != 'NOT YET DUE')
        a['short_3b'] = (max(0.0, a['value_due'] - a['r3b_taxable']) if a['r3b_taxable'] is not None else None)
        a['short_r1'] = (max(0.0, a['value_due_r1'] - a['r1_value']) if a['r1_value'] is not None else None)
        a['max_cum_short'] = max([r['cum_short'] or 0.0 for r in rows if r['fy'] == f], default=0.0) if have_3b else None
        a['gstr9'] = a['value'] > GSTR9_LIMIT
        a['gstr9c'] = a['value'] > GSTR9C_LIMIT
        # TDS sits in the cash ledger; SGST cash used in 3B below the SGST TDS credited means the
        # TDS was not absorbed — a balance building up, and a refund claim to watch for.
        a['tds_sgst_unabsorbed'] = (max(0.0, a['sgst'] - a['cash_sgst']) if a['cash_sgst'] is not None else None)
    no3b = [r['period'] for r in rows if r['r3b_status'] in ('NOT FILED', 'NO GSTR-3B ON RECORD')]
    nil3b = [r['period'] for r in rows if r['r3b_status'] == 'FILED - NIL']
    cgst_over = [r['period'] for r in rows if r['r3b_out_cgst'] is not None and n0(r['cgst']) > r['r3b_out_cgst'] + 1]
    no_r1 = [r['period'] for r in rows if r['r1_status'] == 'NOT FILED']
    total_mismatch = [r['period'] for r in rows if any(o.startswith('Heads add') for o in r['obs'])]
    deductor_max = max([n0(r['deductors']) for r in rows], default=0)

    # ---- the tests, each with its legal basis and whether the data held lets it run
    def rs(v): return f"Rs.{inr(v)}"
    fyl = sorted(fys, key=lambda f: int(f.split('-')[0]))
    T = []
    short_fy = [(f, fys[f]['short_3b']) for f in fyl if (fys[f]['short_3b'] or 0) > 1]
    cum_fy = [(f, fys[f]['max_cum_short']) for f in fyl if (fys[f]['max_cum_short'] or 0) > 1]
    T.append(('Value the deductors paid for against taxable turnover declared in GSTR-3B — month, cumulative and FY',
              'Sections 12, 13(2), 31 and 39; Rule 61. A supply paid for must have been declared.',
              'GSTR-3B: TAX PAID DETAILS (and R3B TAX PAYABLE)', have_3b,
              (('SHORTFALL: ' + '; '.join(f'FY {f} {rs(v)}' for f, v in short_fy) + '. ' if short_fy else
                'No FY shortfall: GSTR-3B taxable turnover covers the value the deductors paid for. ')
               + ('Peak cumulative shortfall within the year: ' + '; '.join(f'FY {f} {rs(v)}' for f, v in cum_fy) + '.'
                  if cum_fy else '')) if have_3b else 'Not run.'))
    T.append(('Months in which TDS was deducted but GSTR-3B was not filed or was filed NIL',
              'Section 39; section 46 (FORM GSTR-3A); section 62 best-judgment assessment of a non-filer.',
              'GSTR-3B: TAX PAID DETAILS', have_3b,
              ((f'No GSTR-3B: {", ".join(no3b)}. ' if no3b else '') + (f'GSTR-3B NIL: {", ".join(nil3b)}. ' if nil3b else '')
               + ('None — every month with TDS has a GSTR-3B with turnover.' if not (no3b or nil3b) else ''))
              if have_3b else 'Not run.'))
    T.append(('CGST deducted at 1% against the whole CGST declared in GSTR-3B for the month',
              'Sections 39 and 51. At any rate of 2% or more, tax declared cannot be below tax deducted at 1%.',
              'R3B TAX PAYABLE DETAILS', ing.have('R3B_TAX_PAYABLE'),
              ((f'CGST deducted EXCEEDS CGST declared in: {", ".join(cgst_over)}.' if cgst_over else 'No month where it does.')
               if ing.have('R3B_TAX_PAYABLE') else 'Not run.')))
    T.append(('GSTR-1 filed for every month with TDS, and its taxable value against the value paid for (FY)',
              'Section 37 and Rule 59 (B2B invoices to each deductor GSTIN); Rule 59(6). Section 47 late fee.',
              'R1 FILINGS IN THE YEAR', have_r1,
              (((f'No GSTR-1: {", ".join(no_r1)}. ' if no_r1 else 'GSTR-1 on record for every TDS month. ')
                + '; '.join(f'FY {f} shortfall against GSTR-1 {rs(fys[f]["short_r1"])}'
                            for f in fyl if (fys[f]['short_r1'] or 0) > 1)) if have_r1 else 'Not run.')))
    T.append(('TDS credited to the cash ledger against cash actually used in GSTR-3B (SGST head)',
              'Rule 87(9); section 49. Unused TDS builds a cash balance that can be claimed as refund under section 54 '
              '(excess balance in the electronic cash ledger).',
              'TAX PAID DETAILS', ing.have('TAX_PAID'),
              ('; '.join(f'FY {f}: SGST deducted {rs(fys[f]["sgst"])}, SGST cash used {rs(fys[f]["cash_sgst"])}, '
                         f'not absorbed {rs(fys[f]["tds_sgst_unabsorbed"])}' for f in fyl) + '.'
               if ing.have('TAX_PAID') else 'Not run.')))
    T.append(('ITC availed against GSTR-2A in the months the deductors paid for supplies',
              'Section 16(2)(aa), Rule 36(4); Rule 88D. A supplier paid by deductors who discharges output tax '
              'wholly from ITC while TDS sits unused is the pattern behind cash-ledger refund claims.',
              'EXCESS ITC CLAIMED, R2A FILINGS', ing.have('EXCESS_ITC') or ing.have('R2A_MONTHLY'),
              ('See 13_EXCESS_ITC and 19A_ITC_vs_2A for the same months.'
               if (ing.have('EXCESS_ITC') or ing.have('R2A_MONTHLY')) else 'Not run.')))
    T.append(('Annual return thresholds on receipts subject to TDS alone',
              'Section 44 and Rule 80: GSTR-9 above Rs.2 crore aggregate turnover (below it, optional where the year\'s notification exempts); self-certified GSTR-9C above Rs.5 crore.',
              'GSTR-7 alone', True,
              '; '.join(f'FY {f} ({fys[f]["first"]} to {fys[f]["last"]}): {rs(fys[f]["value"])} — '
                        + ('GSTR-9C required' if fys[f]['gstr9c'] else 'GSTR-9 required' if fys[f]['gstr9']
                           else 'below Rs.2 crore on these receipts alone') for f in fyl) + '.'))
    T.append(('E-invoicing and QRMP eligibility',
              'Rule 48(4) with Notification 13/2020-CT as amended (e-invoice once aggregate turnover in any preceding FY '
              'exceeds Rs.5 crore); Rule 61A (QRMP only up to Rs.5 crore).',
              'GSTR-7 alone', True,
              '; '.join(f'FY {f}: {rs(fys[f]["value"])} from the deductors alone'
                        + (' — above Rs.5 crore: e-invoicing from the next FY, QRMP not available'
                           if fys[f]['value'] > GSTR9C_LIMIT else
                           (' in a part year — close to Rs.5 crore; watch the rest of the year'
                            if fys[f]['value'] > 0.75 * GSTR9C_LIMIT else '')) for f in fyl) + '.'))
    over = [p for f in fyl for p in fys[f]['over50']]
    T.append(('Rule 86B — months in which the deductors alone paid for more than Rs.50 lakh of supplies',
              'Rule 86B (not more than 99% of output tax from ITC, subject to the proviso). TDS credit sits in the cash '
              'ledger, so TDS actually USED counts as cash.',
              'GSTR-7 alone; TAX PAID DETAILS to test', True,
              (f'Threshold crossed on receipts subject to TDS alone in {", ".join(over)}. Whether the 1% was met depends on the '
               f'cash and TDS actually used — test on the electronic cash ledger.' if over else 'No month above Rs.50 lakh.')))
    T.append(('Place of supply — intra-State or inter-State deduction',
              'Section 51(1) and its proviso: IGST is deducted on an inter-State supply; nothing is deducted where the '
              "supplier's location and the place of supply are both in a State other than the recipient's registration.", 'GSTR-7 alone', True,
              ('IGST was deducted in some months — inter-State supplies; check GSTR-1 shows them as inter-State.'
               if any(n0(r['igst']) > 0 for r in rows) else
               'Only CGST and SGST deducted: every deductor is in the same State as the supply. GSTR-1 should show these '
               'as intra-State B2B supplies.')))

    return dict(available=True, rows=rows, fy=fys, have_3b=have_3b, have_r1=have_r1, tests=T,
                have_tax_paid=ing.have('TAX_PAID'), have_2a=ing.have('R2A_MONTHLY') or ing.have('EXCESS_ITC'),
                no_3b=no3b, nil_3b=nil3b, cgst_over=cgst_over, no_r1=no_r1, total_mismatch=total_mismatch,
                interstate=any(n0(r['igst']) > 0 for r in rows), deductor_max=deductor_max,
                total_value=sum(r['value'] for r in rows), total_tds=sum(a['tds'] for a in fys.values()),
                first=rows[0]['period'] if rows else None, last=rows[-1]['period'] if rows else None)

# ------------------------------------- 15. GSTR-7 party-wise detail (NEW v1.7)
PAN_HOLDER = {'P': 'Individual / proprietor', 'F': 'Firm / LLP', 'H': 'HUF', 'C': 'Company',
              'A': 'Association of persons', 'T': 'Trust', 'B': 'Body of individuals',
              'L': 'Local authority', 'J': 'Artificial juridical person', 'G': 'Government', 'K': 'Krish (unused)'}
# who may deduct under s.51(1)(a)-(d) read with Notification 50/2018-CT
ELIGIBLE = {'G': 'ELIGIBLE TYPE — Government', 'L': 'ELIGIBLE TYPE — local authority'}
VERIFY = {'C': 'VERIFY — a company can deduct only if it is a public sector undertaking or a Government-controlled body',
          'J': 'VERIFY — eligible only if an authority / board set up by statute or Government',
          'A': 'VERIFY — eligible only if a society established by Government or a local authority',
          'T': 'VERIFY — eligible only if a society / body established by Government',
          'B': 'VERIFY — eligible only if a body established by Government'}
TAN_BODY = re.compile(r'^[A-Z]{4}\d{5}[A-Z]$')
PAN_BODY = re.compile(r'^[A-Z]{5}\d{4}[A-Z]$')

def gstin_profile(g):
    """What a GSTIN says about itself, on its face. Position 14 is 'Z' for an ordinary registration;
    a tax-deductor registration (REG-07, section 24(vi)) carries 'D' there. Positions 3-12 are the
    PAN (or, for most Government deductors, the TAN); the 4th character of a PAN is the holder type."""
    g = (g or '').strip().upper()
    if len(g) != 15: return dict(gstin=g, reg='INVALID LENGTH', body='', holder='', eligibility='VERIFY — malformed GSTIN')
    body, c14 = g[2:12], g[13]
    reg = {'Z': 'Ordinary registration', 'D': 'Tax-deductor registration (REG-07)', 'C': 'Tax-collector registration'}.get(
        c14, f'Unusual 14th character "{c14}"')
    if TAN_BODY.match(body):
        return dict(gstin=g, reg=reg, body='TAN', holder='TAN-based (typical of a Government DDO)',
                    eligibility='ELIGIBLE TYPE — TAN-based deductor (confirm it is a notified deductor)')
    if PAN_BODY.match(body):
        h = body[3]
        el = ELIGIBLE.get(h) or VERIFY.get(h) or ('NOT A PERMITTED DEDUCTOR ON THE FACE OF THE PAN — holder type "' +
                                                    PAN_HOLDER.get(h, f'PAN type {h}') +
                                                    '" is not within section 51(1) or Notification 50/2018-CT')
        return dict(gstin=g, reg=reg, body='PAN', holder=PAN_HOLDER.get(h, f'PAN type {h}'), eligibility=el)
    return dict(gstin=g, reg=reg, body='?', holder='Neither PAN nor TAN pattern', eligibility='VERIFY — malformed body')

def chk_tds_detail(ing, case_gstin=None):
    """The party-wise GSTR-7 detail, one Prime export per month. Tested for: who the listed parties are
    (registration type and PAN holder type, against who section 51 allows to deduct); agreement with the
    month-wise summary; rows just above the Rs.2.5 lakh deduction threshold; and which months are missing."""
    if not ing.have('R7_DETAIL'):
        return dict(available=False, missing=['R7_DETAIL'])
    rows = sorted(ing.data['R7_DETAIL'], key=lambda r: (period_key(r['_period']), r.get('SNo') or 0))
    subj = sorted({x.get('subject_gstin') for x in ing.register if x.get('code') == 'R7_DETAIL' and x.get('subject_gstin')})
    parties = {}
    for r in rows:
        g = (r.get('Deductee GSTIN') or '').strip().upper()
        p = parties.setdefault(g, dict(names=[], months=[], lines=0, value=0.0, tds=0.0, **gstin_profile(g)))
        nm = r.get('Trade Name')
        if nm and nm not in p['names']: p['names'].append(nm)
        if r['_period'] not in p['months']: p['months'].append(r['_period'])
        p['lines'] += 1; p['value'] += n0(r.get('Taxable Value')); p['tds'] += n0(r.get('Total GST'))
    # month reconciliation with the summary
    summ = ing.by_period('R7_TDS') if ing.have('R7_TDS') else {}
    months = []
    for m in sorted({r['_period'] for r in rows}, key=period_key):
        mr = [r for r in rows if r['_period'] == m]
        s = summ.get(m)
        tds = sum(n0(r.get('Total GST')) for r in mr)
        months.append(dict(period=m, lines=len(mr), parties=len({r.get('Deductee GSTIN') for r in mr}),
                           value=sum(n0(r.get('Taxable Value')) for r in mr), tds=tds,
                           s_parties=(s or {}).get('No Of Deductors'), s_tds=(s or {}).get('Total GST'),
                           agrees=(None if s is None else
                                   abs(tds - n0(s.get('Total GST'))) <= 1.0 + 0.5 * len(mr)
                                   and int(n0(s.get('No Of Deductors'))) == len({r.get('Deductee GSTIN') for r in mr}))))
    missing = sorted(set(summ) - {m['period'] for m in months}, key=period_key)
    # rate check line by line: each head should be 1% of the taxable value (2% IGST)
    off_rate = []
    near = []
    for r in rows:
        v = n0(r.get('Taxable Value'))
        if v and abs(n0(r.get('CGST')) - v * 0.01) > 1.0 and abs(n0(r.get('IGST')) - v * 0.02) > 1.0:
            off_rate.append((r['_period'], r.get('Deductee GSTIN'), v, n0(r.get('CGST'))))
        if 250_000 < v <= 260_000:
            near.append((r['_period'], r.get('Deductee GSTIN'), v))
    lst = list(parties.values())
    all_d = all(p['reg'].startswith('Tax-deductor') for p in lst)
    subj_z = bool(subj) and all(x[13] == 'Z' for x in subj)
    if all_d and subj_z:
        role = ('The listed parties hold TAX-DEDUCTOR registrations (14th character D) and the GSTIN the export is titled '
                'for holds an ordinary registration (14th character Z). A GSTR-7 can be filed only from a deductor '
                'registration, so the LISTED PARTIES ARE THE DEDUCTORS and the case taxpayer is the DEDUCTEE. Prime heads '
                'the column "Deductee GSTIN"; the summary report counts the same parties as "No Of Deductors". Confirm '
                'one GSTIN on the portal (Search Taxpayer shows the registration type).')
        role_code = 'DEDUCTORS'
    else:
        role = ('The export does not settle on its face whether the listed parties are the deductors or the deductees. '
                'Check each GSTIN on the portal before relying on the eligibility column.')
        role_code = 'UNSETTLED'
    bad = [p for p in lst if p['eligibility'].startswith('NOT A PERMITTED')]
    total_v = sum(p['value'] for p in lst)
    return dict(available=True, rows=rows, parties=sorted(lst, key=lambda p: -p['value']), months=months,
                missing_months=missing, subject=subj,
                subject_mismatch=[x for x in subj if case_gstin and x != case_gstin.upper()],
                role=role, role_code=role_code, ineligible=bad,
                ineligible_value=sum(p['value'] for p in bad), ineligible_tds=sum(p['tds'] for p in bad),
                total_value=total_v, total_tds=sum(p['tds'] for p in lst),
                top_share=(max((p['value'] for p in lst), default=0) / total_v if total_v else None),
                off_rate=off_rate, near_threshold=near)

def run_all(ing):
    m = build_master(ing)
    r1 = chk_r1_vs_r3b(ing, m)
    res = dict(
        master=m, fy=fy_summary(m),
        r1_vs_r3b=r1,
        excess_itc=chk_excess_itc(ing, m),
        itc_vs_2a=chk_itc_vs_2a(ing, m),
        dormancy=chk_dormancy(ing, m),
        rule86b=chk_rule86b(ing, m),
        rcm=chk_rcm(ing, m),
        credit_notes=chk_credit_notes(ing, m),
        filing=chk_return_filing(ing, m, r1),
        cash=chk_cash(ing, m),
        hsn=chk_hsn(ing),
        annual=chk_annual(ing, m),
        tds=chk_tds(ing, m),
        tds_detail=chk_tds_detail(ing, getattr(ing, 'case_gstin', None)),
        limitation=limitation(_fys(ing.all_periods())),
        unfiled=unfiled_periods(ing),
    )
    return res


## Step 5 · Engine — workbook builder

Writes the source sheets verbatim, then every computed sheet as live Excel formulas reading them, then the findings, report, action points, data gaps and limitation sheets.

*Written as `gpe_build.py` and copied into this taxpayer's `00_ENGINE/` folder, so it persists between sessions.*

In [ ]:
%%writefile gpe_build.py
# -*- coding: utf-8 -*-
"""
GST-PRIME PRELIMINARY INVESTIGATION ENGINE  —  workbook builder
Source sheets are written verbatim first. Every computed sheet then reads them
with live Excel formulas, so any value can be traced back to a source cell.
"""
import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill
from openpyxl.comments import Comment
from openpyxl.utils import get_column_letter
from gpe_core import REPORT_SPECS, period_key, fy_of, n0, inr
from gpe_checks import due_date

BUILD_VERSION = "1.6"
BOLD = Font(bold=True)
THIN = Side(style='thin')
BD = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
MISSING_FILL = PatternFill('solid', fgColor='FFD9D9')   # light red — data not available
SUPERSEDED_FILL = PatternFill('solid', fgColor='FFF2CC')  # light amber — record replaced by a later export
CHANGED_FILL = PatternFill('solid', fgColor='F8CBAD')     # orange — a value that differs from the earlier export
NUM = '#,##0'

def _q(s):  return f"'{s}'"

def _frank(f):
    """Findings order: revenue first; among equal amounts, by risk."""
    r = f.get('r', '')
    k = 0 if r.startswith('VERY HIGH') else 1 if r.startswith('HIGH') else 2 if r.startswith('MEDIUM') else 3
    return (-f['a'], k)

class Builder:
    def __init__(self, ing, R, gstin, name, mark_missing_light_red=True,
                 extra_profile=None):
        self.ing, self.R = ing, R
        self.gstin, self.name = gstin, name
        self.mark = mark_missing_light_red
        self.profile = extra_profile or {}
        self.wb = Workbook(); self.wb.remove(self.wb.active)
        self.src_rows = {}          # code -> last data row
        self.periods = [m['period'] for m in R['master']]

    # ---------------------------------------------------------- utilities
    def sheet(self, title):  return self.wb.create_sheet(title)

    def head(self, ws, r, hdr, widths=None):
        for j, h in enumerate(hdr, 1):
            c = ws.cell(row=r, column=j, value=h); c.font = BOLD; c.border = BD
            c.alignment = Alignment(wrap_text=True, vertical='top', horizontal='center')
        if widths:
            for i, w in enumerate(widths, 1): ws.column_dimensions[get_column_letter(i)].width = w
        ws.freeze_panes = ws.cell(row=r + 1, column=1)

    def box(self, ws, r0, r1, c0, c1, numfrom=None):
        for r in range(r0, r1 + 1):
            for c in range(c0, c1 + 1):
                x = ws.cell(row=r, column=c); x.border = BD
                if numfrom and c >= numfrom and not isinstance(x.value, (datetime.date, datetime.datetime)):
                    x.number_format = NUM

    @staticmethod
    def _safe(v):
        """Stop a descriptive string that starts with '=' being read as a formula."""
        return ('Formula: ' + v[1:].strip()) if isinstance(v, str) and v.startswith('=') and ' ' in v[:14] else v

    def note(self, ws, r, text, bold=False, col=1):
        c = ws.cell(row=r, column=col, value=text)
        c.alignment = Alignment(wrap_text=True, vertical='top')
        if bold: c.font = BOLD
        return r + 1

    def na(self, ws, r, c, text='NOT YET AVAILABLE'):
        x = ws.cell(row=r, column=c, value=text)
        if self.mark: x.fill = MISSING_FILL
        x.alignment = Alignment(horizontal='center')
        return x

    def unavailable(self, ws, res, what):
        ws['A1'] = what; ws['A1'].font = BOLD
        r = self.note(ws, 3, 'THIS TEST COULD NOT BE RUN.', bold=True)
        r = self.note(ws, r, 'The following GST-Prime report(s) have not been supplied: '
                             + ', '.join(REPORT_SPECS[m]['title'] + f'  [{m}]'
                                         for m in res.get('missing', [])))
        r = self.note(ws, r, 'Upload the report and run the notebook again. Nothing has been '
                             'estimated or assumed in its place.')
        if self.mark: ws.cell(row=3, column=1).fill = MISSING_FILL
        ws.column_dimensions['A'].width = 130

    # ---------------------------------------------------- 09A restatements
    def build_restatements(self):
        ch = self.ing.changes
        counted = self.ing.counted()
        n_repl = sum(r.get('rows_replaced', 0) for r in counted)
        n_recchg = sum(r.get('records_changed', 0) for r in counted)
        ws = self.sheet('09A_RESTATEMENTS')
        ws['A1'] = 'WHAT A LATER EXPORT CHANGED FROM AN EARLIER ONE'; ws['A1'].font = BOLD
        ws['A2'] = ('When a later Prime export carries a record already held, the later record replaces the earlier one. '
                    'Where the two differ, the difference is listed here with both values, so a restatement is never '
                    'absorbed silently. The later value is the one used everywhere in this workbook.')
        r = 4
        ws.cell(row=r, column=1, value='Records REPLACED by a later export').font = BOLD
        ws.cell(row=r, column=4, value=n_repl); r += 1
        ws.cell(row=r, column=1, value='   of which the values were IDENTICAL').font = BOLD
        ws.cell(row=r, column=4, value=n_repl - n_recchg); r += 1
        ws.cell(row=r, column=1, value='   of which one or more values DIFFER').font = BOLD
        ws.cell(row=r, column=4, value=n_recchg); r += 2
        if not ch:
            self.note(ws, r, 'No value in any later export differs from the earlier export it replaced.' if n_repl
                             else 'No record was supplied more than once. Nothing was replaced.', bold=True)
            ws.column_dimensions['A'].width = 60
            return
        order = {'REVIEW': 0, 'NOTE': 1, 'INFO': 2}
        rows = sorted(ch, key=lambda c: (order[c['level']], c['code'], str(c['record']), c['field']))
        ws.cell(row=r, column=1, value='SUMMARY').font = BOLD; r += 1
        H = ['Level','What it means','Values','Records']
        for j, h in enumerate(H, 1):
            x = ws.cell(row=r, column=j, value=h); x.font = BOLD; x.border = BD
        meaning = {'REVIEW': 'A value the taxpayer reported, or a challan, differs from the earlier export. Verify on the portal before relying on either.',
                   'NOTE': 'Blank in the earlier export and populated now — most likely a return filed since.',
                   'INFO': 'GSTR-2A moved, as it does whenever a supplier files late or amends. Expected.'}
        r += 1; sum0 = r
        det0 = sum0 + 3 + 3          # detail header row is computed below; formulas use whole-column ranges instead
        for lv in ('REVIEW', 'NOTE', 'INFO'):
            ws.cell(row=r, column=1, value=lv)
            ws.cell(row=r, column=2, value=meaning[lv]).alignment = Alignment(wrap_text=True, vertical='top')
            ws.cell(row=r, column=3, value=sum(1 for c in ch if c['level'] == lv))
            ws.cell(row=r, column=4, value=len({(c['code'], c['record']) for c in ch if c['level'] == lv}))
            for j in range(1, 5): ws.cell(row=r, column=j).border = BD
            if lv == 'REVIEW' and self.mark and any(c['level'] == 'REVIEW' for c in ch):
                ws.cell(row=r, column=1).fill = MISSING_FILL
            r += 1
        r += 1
        ws.cell(row=r, column=1, value='EVERY DIFFERENCE').font = BOLD; r += 1
        H = ['Level','Report','Record','Field','Value in the EARLIER export','Value in the LATER export',
             'Difference (later minus earlier)','Nature','Which is newer?','File labelled earlier',
             'File labelled later','Reading']
        self.head(ws, r, H, [10,40,26,26,22,22,22,12,26,42,42,88])
        hdr = r; r += 1
        for c in rows:
            vals = [c['level'], c['title'], c['record'], c['field'], c['old'], c['new'], None, c['nature'],
                    ('UNCERTAIN — same upload' if c.get('order_uncertain') else 'Later upload batch'),
                    c['old_file'], c['new_file'], c['reading']]
            for j, v in enumerate(vals, 1):
                x = ws.cell(row=r, column=j, value=v); x.border = BD
                x.alignment = Alignment(wrap_text=True, vertical='top')
            ws.cell(row=r, column=7, value=f'=IF(AND(ISNUMBER(E{r}),ISNUMBER(F{r})),F{r}-E{r},"")')
            for j in (5, 6, 7): ws.cell(row=r, column=j).number_format = NUM
            if c['level'] == 'REVIEW' and self.mark: ws.cell(row=r, column=1).fill = MISSING_FILL
            if c.get('order_uncertain') and self.mark: ws.cell(row=r, column=9).fill = MISSING_FILL
            r += 1
        r += 1
        n_unc = sum(1 for c in ch if c.get('order_uncertain'))
        if n_unc:
            r = self.note(ws, r, f'ORDER UNCERTAIN for {n_unc} value(s). The two files arrived in the same upload, so the '
                                 f'upload gives no evidence of which is newer, and Prime exports carry no date of their own. '
                                 f'The value called "later" is only the one that sorted last by filename. Establish which '
                                 f'export was taken more recently, delete the other from 01_RAW_UPLOADS and run again.', bold=True)
        r = self.note(ws, r, 'WHY THIS MATTERS. A later export that changes a value already held may mean the taxpayer amended '
                             'something, or that Prime refreshed its own data, or that the earlier export was taken before the '
                             'period was final. The export cannot say which. GSTR-3B and payment challans cannot be revised, so a '
                             'REVIEW difference in either is unusual and should be checked on the portal before either figure is '
                             'quoted in a notice.', bold=True)
        self.note(ws, r, 'The later value is the one used in every computed sheet. If the earlier value is the correct one, '
                         'remove the later file from 01_RAW_UPLOADS and run again.')

    # ------------------------------------------------------ source sheets
    def build_sources(self):
        order = [c for c in REPORT_SPECS if self.ing.have(c)]
        for code in order:
            sp = REPORT_SPECS[code]
            rows = self.ing.data[code]
            if sp['grain'] in ('period', 'period_multi'):
                rows = sorted(rows, key=lambda r: (period_key(r['_period']), r.get('SNo') or 0))
            elif sp['grain'] == 'fy':
                rows = sorted(rows, key=lambda r: r.get('_fy') or '')
            cols = [k for k in rows[0].keys() if not k.startswith('_')]
            for r in rows:
                for k in r:
                    if not k.startswith('_') and k not in cols: cols.append(k)
            ws = self.sheet(f'SRC_{code}'[:31])
            ws.cell(row=1, column=1, value='Source file')
            for j, c in enumerate(cols, 2): ws.cell(row=1, column=j, value=c)
            HIST = len(cols) + 2                       # last column: record history
            ws.cell(row=1, column=HIST, value='Record history')
            col_of = {c: j for j, c in enumerate(cols, 2)}
            n_super = n_chg = 0
            for i, r in enumerate(rows, 2):
                ws.cell(row=i, column=1, value=r.get('_file'))
                for j, c in enumerate(cols, 2):
                    v = r.get(c)
                    cell = ws.cell(row=i, column=j, value=v)
                    if isinstance(v, (datetime.date, datetime.datetime)):
                        cell.number_format = 'DD-MM-YYYY'
                    elif v is None and self.mark and c not in ('CESS',):
                        cell.fill = MISSING_FILL
                h = r.get('_history')
                if h:
                    n_super += 1
                    ch = h.get('changes') or []
                    txt = ((f'ORDER UNCERTAIN (same upload). ' if h.get('uncertain') and ch else '')
                           + f'REPLACES the record held from {h.get("supersedes")}. '
                           + (f'{len(ch)} value(s) differ: ' + '; '.join(f'{f}: {a0} -> {b0}' for f, a0, b0 in ch)
                              if ch else 'No value differs.'))
                    hc = ws.cell(row=i, column=HIST, value=txt)
                    hc.alignment = Alignment(wrap_text=True, vertical='top')
                    if ch:
                        n_chg += 1
                        if self.mark:
                            for j in range(1, HIST + 1):
                                if ws.cell(row=i, column=j).fill != MISSING_FILL:
                                    ws.cell(row=i, column=j).fill = SUPERSEDED_FILL
                    for f, a0, b0 in ch:
                        j = col_of.get(f)
                        if j:
                            cc = ws.cell(row=i, column=j)
                            if self.mark: cc.fill = CHANGED_FILL
                            cc.comment = Comment(f'Earlier export ({h.get("supersedes")}) held: {a0}\n'
                                                 f'This value comes from {r.get("_file")}.', 'Engine')
            self.head(ws, 1, ['Source file'] + cols + ['Record history'],
                      [34] + [16] * len(cols) + [58])
            self.box(ws, 2, 1 + len(rows), 1, HIST, numfrom=3)
            self.src_rows[code] = 1 + len(rows)
            self.col_index = None
            r = 3 + len(rows)
            self.note(ws, r, f'{sp["title"]}. Reproduced exactly as received — no value altered, '
                             f'no blank filled in. A cell shaded light red was BLANK in the source, '
                             f'which is not the same as nil.', bold=True)
            if n_super:
                self.note(ws, r + 1, f'{n_super} row(s) come from a LATER export and REPLACED a record held from an earlier one '
                                     f'(see the Record history column). {n_chg} of them differ from the record they replaced and '
                                     f'are shaded amber; an orange cell is the value that differs — hover over it for the earlier '
                                     f'value. The other {n_super - n_chg} were replaced by identical values and are not shaded. '
                                     f'Every difference is listed in 09A_RESTATEMENTS.', bold=True)
        return order

    def src_col(self, code, name):
        """Column letter of a named column on the SRC sheet for `code`, or None if that
        report was never supplied and so has no source sheet."""
        title = f'SRC_{code}'[:31]
        if title not in self.wb.sheetnames:
            return None
        ws = self.wb[title]
        for c in range(1, ws.max_column + 1):
            if ws.cell(row=1, column=c).value == name: return get_column_letter(c)
        return None

    def lookup(self, code, colname, keycol, key_ref):
        """Live INDEX/MATCH into a source sheet; 0 when the report was not supplied at all,
        so a missing report leaves an honest zero rather than breaking the workbook."""
        if code not in self.src_rows:
            return 0
        s = _q(f'SRC_{code}'[:31]); last = self.src_rows[code]
        col = self.src_col(code, colname); kc = self.src_col(code, keycol)
        if col is None or kc is None: return 0
        return (f'=IFERROR(INDEX({s}!${col}$2:${col}${last},'
                f'MATCH({key_ref},{s}!${kc}$2:${kc}${last},0)),0)')

    # ------------------------------------------------------------ master
    def build_master(self):
        ws = self.sheet('10_MONTHLY_MASTER')
        if not self.R['master']:
            self.master_last = 3
            return self.unavailable(ws, dict(missing=['TAX_PAID', 'R3B_TAX_PAYABLE', 'R3B_TURNOVER']),
                                    'MONTHLY MASTER — no GSTR-3B report supplied, so there is no tax period to build it on')
        ws['A1'] = ('MONTHLY MASTER — one row per tax period, chronological. Every figure below '
                    'row 3 is a live formula reading the SRC_ sheets.')
        ws['A1'].font = BOLD
        H = ['Return Period','FY','Filing status','Total turnover (3B)','Taxable turnover (3B)',
             'Outward SGST','Outward CGST','Outward IGST','Outward total tax (3.1(a))',
             'GST OP Total (incl. RCM)','Implied RCM / other liability','RCM inward supply value',
             'ITC availed','SGST ITC claimed 3B','SGST available 2A','SGST excess',
             'IGST ITC claimed 3B','IGST available 2A','IGST excess','SGST cash paid',
             'SGST settlement','Gross SGST collected','NIL turnover','ITC availed in NIL period',
             'GSTR-2A data availability']
        self.head(ws, 3, H, [13,9,13]+[15]*19+[11,16,20])
        P = lambda r: f'$A{r}'
        for i, m in enumerate(self.R['master'], 4):
            ws.cell(row=i, column=1, value=m['period'])
            ws.cell(row=i, column=2, value=m['fy'])
            ws.cell(row=i, column=3, value=self.lookup('TAX_PAID','Filing Status','Return Period',P(i)))
            ws.cell(row=i, column=4, value=self.lookup('TAX_PAID','Total TO','Return Period',P(i)))
            ws.cell(row=i, column=5, value=self.lookup('TAX_PAID','Taxable TO','Return Period',P(i)))
            for col, nm in zip(range(6,10), ['Outward SGST','Outward CGST','Outward IGST','Outward Total GST']):
                ws.cell(row=i, column=col, value=self.lookup('R3B_TAX_PAYABLE', nm, 'Return Period', P(i)))
            ws.cell(row=i, column=10, value=self.lookup('TAX_PAID','GST OP Total','Return Period',P(i)))
            ws.cell(row=i, column=11, value=f'=J{i}-I{i}')
            ws.cell(row=i, column=12, value=self.lookup('R3B_TURNOVER','Inward Supplies(Reverse Charge)','Return Period',P(i)))
            ws.cell(row=i, column=13, value=self.lookup('TAX_PAID','GST ITC Total','Return Period',P(i)))
            for col, nm in zip(range(14,20), ['SGSTITC Claimed In R3B','SGSTITC Available In R2A','SGSTExcess Claimed',
                                              'IGSTITC Claimed In R3B','IGSTITC Available In R2A','IGSTExcess Claimed']):
                ws.cell(row=i, column=col, value=self.lookup('EXCESS_ITC', nm, 'Return Period', P(i)))
            for col, nm in zip(range(20,23), ['SGST Cash Paid','SGST Settlement','Gross SGST Collected']):
                ws.cell(row=i, column=col, value=self.lookup('TAX_PAID', nm, 'Return Period', P(i)))
            ws.cell(row=i, column=23, value=f'=IF(D{i}=0,"NIL","")')
            ws.cell(row=i, column=24, value=f'=IF(W{i}="NIL",M{i},0)')
            c = ws.cell(row=i, column=25, value='2A NOT AVAILABLE' if m['r2a_not_available'] else '')
            if m['r2a_not_available'] and self.mark: c.fill = MISSING_FILL
        last = 3 + len(self.R['master'])
        ws.cell(row=last+1, column=1, value='TOTAL').font = BOLD
        for c in list(range(4,23)) + [24]:
            L = get_column_letter(c)
            ws.cell(row=last+1, column=c, value=f'=SUM({L}4:{L}{last})').font = BOLD
        self.box(ws, 4, last+1, 1, 25, numfrom=4)
        for r in range(4, last+2):
            for c in (3, 23, 25): ws.cell(row=r, column=c).number_format = 'General'
        self.master_last = last
        return last

    def m(self, col, row):  return f"'10_MONTHLY_MASTER'!{col}{row}"

    # --------------------------------------------------------- FY summary
    def build_fy(self):
        ws = self.sheet('11_FY_SUMMARY')
        if not self.R['master']:
            self.fy_last = 4
            return self.unavailable(ws, dict(missing=['TAX_PAID', 'R3B_TAX_PAYABLE']),
                                    'FINANCIAL-YEAR SUMMARY OF GSTR-3B')
        ws['A1'] = 'FINANCIAL-YEAR SUMMARY — SUMIFS over 10_MONTHLY_MASTER'; ws['A1'].font = BOLD
        H = ['FY','Tax periods','NIL periods','Turnover (3B)','Output tax (incl. RCM)','ITC availed',
             'ITC as % of output tax','SGST cash paid','Net SGST to exchequer','SGST excess over 2A',
             'IGST excess over 2A','ITC availed in NIL periods']
        self.head(ws, 3, H, [10,11,11,18,18,16,13,15,17,16,16,18])
        M = _q('10_MONTHLY_MASTER'); L = self.master_last
        fys = sorted(self.R['fy'], key=lambda x: int(x.split('-')[0]))
        for i, f in enumerate(fys, 4):
            ws.cell(row=i, column=1, value=f)
            ws.cell(row=i, column=2, value=f'=COUNTIF({M}!$B$4:$B${L},$A{i})')
            ws.cell(row=i, column=3, value=f'=COUNTIFS({M}!$B$4:$B${L},$A{i},{M}!$W$4:$W${L},"NIL")')
            for col, src in zip([4,5,6,8,9,10,11,12], ['D','J','M','T','V','P','S','X']):
                ws.cell(row=i, column=col,
                        value=f'=SUMIF({M}!$B$4:$B${L},$A{i},{M}!${src}$4:${src}${L})')
            ws.cell(row=i, column=7, value=f'=IFERROR(F{i}/E{i},"")')
        r = 4 + len(fys)
        ws.cell(row=r, column=1, value='TOTAL').font = BOLD
        for c in [2,3,4,5,6,8,9,10,11,12]:
            LL = get_column_letter(c); ws.cell(row=r, column=c, value=f'=SUM({LL}4:{LL}{r-1})').font = BOLD
        ws.cell(row=r, column=7, value=f'=IFERROR(F{r}/E{r},"")').font = BOLD
        self.box(ws, 4, r, 1, 12, numfrom=2)
        for i in range(4, r+1): ws.cell(row=i, column=7).number_format = '0.0%'
        self.fy_last = r
        self.note(ws, r+2, 'Net SGST to exchequer is Gross SGST Collected as computed by GST-Prime '
                           '(cash plus settlement). A negative figure means the State bore a net outflow that year.')
        return r

    # --------------------------------------------- 13 excess ITC (demand)
    def build_excess(self):
        res = self.R['excess_itc']; ws = self.sheet('13_EXCESS_ITC')
        if not res.get('available'): return self.unavailable(ws, res, 'ITC AVAILED AGAINST ITC AVAILABLE IN GSTR-2A')
        ws['A1'] = 'ITC AVAILED IN GSTR-3B AGAINST ITC AVAILABLE IN GSTR-2A — head by head, month by month'
        ws['A1'].font = BOLD
        ws['A2'] = 'THIS IS THE DEMAND FIGURE. Heads are compared separately because credit in one head is not set off against a shortfall in another.'
        H = ['Return Period','FY','SGST claimed 3B','SGST available 2A','SGST excess',
             'CGST excess (mirror of SGST)','IGST claimed 3B','IGST available 2A','IGST excess',
             'Total excess (S+C+I)','Positive excess only','2A availability']
        self.head(ws, 4, H, [13,9]+[17]*9+[18])
        L = self.master_last
        for i, m in enumerate(self.R['master'], 5):
            mr = i - 1
            ws.cell(row=i, column=1, value=f'={self.m("A", mr)}')
            ws.cell(row=i, column=2, value=f'={self.m("B", mr)}')
            for col, src in zip([3,4,5,7,8,9], ['N','O','P','Q','R','S']):
                ws.cell(row=i, column=col, value=f'={self.m(src, mr)}')
            ws.cell(row=i, column=6, value=f'=E{i}')
            ws.cell(row=i, column=10, value=f'=E{i}+F{i}+I{i}')
            ws.cell(row=i, column=11, value=f'=MAX(0,E{i})+MAX(0,F{i})+MAX(0,I{i})')
            c = ws.cell(row=i, column=12, value='2A NOT AVAILABLE' if m['r2a_not_available'] else '')
            if m['r2a_not_available'] and self.mark: c.fill = MISSING_FILL
        r = 4 + len(self.R['master']) + 1
        ws.cell(row=r, column=1, value='TOTAL').font = BOLD
        for c in range(3, 12):
            LL = get_column_letter(c); ws.cell(row=r, column=c, value=f'=SUM({LL}5:{LL}{r-1})').font = BOLD
        self.box(ws, 5, r, 1, 12, numfrom=3)
        for i in range(5, r+1): ws.cell(row=i, column=12).number_format = 'General'
        self.excess_net_cell = f"'13_EXCESS_ITC'!J{r}"
        self.excess_gross_cell = f"'13_EXCESS_ITC'!K{r}"
        s = r + 2
        ws.cell(row=s, column=1, value='FINANCIAL-YEAR ROLL-UP').font = BOLD
        H2 = ['FY','SGST excess (net)','CGST excess (net)','IGST excess (net)','Total excess (net)','Total excess (gross)']
        self.head(ws, s+1, H2)
        fys = sorted(self.R['fy'], key=lambda x: int(x.split('-')[0]))
        for i, f in enumerate(fys, s+2):
            ws.cell(row=i, column=1, value=f)
            ws.cell(row=i, column=2, value=f'=SUMIF($B$5:$B${r-1},$A{i},$E$5:$E${r-1})')
            ws.cell(row=i, column=3, value=f'=B{i}')
            ws.cell(row=i, column=4, value=f'=SUMIF($B$5:$B${r-1},$A{i},$I$5:$I${r-1})')
            ws.cell(row=i, column=5, value=f'=B{i}+C{i}+D{i}')
            ws.cell(row=i, column=6, value=f'=SUMIF($B$5:$B${r-1},$A{i},$K$5:$K${r-1})')
        r2 = s + 2 + len(fys)
        ws.cell(row=r2, column=1, value='TOTAL').font = BOLD
        for c in range(2, 7):
            LL = get_column_letter(c); ws.cell(row=r2, column=c, value=f'=SUM({LL}{s+2}:{LL}{r2-1})').font = BOLD
        self.box(ws, s+2, r2, 1, 6, numfrom=2)
        n = r2 + 2
        n = self.note(ws, n, 'NET is the sum of all periods, positive and negative. GROSS counts only the periods of over-claim. '
                             'A short claim in one period followed by a larger claim in the next is ordinarily only timing, '
                             'so NET is the defensible exposure and GROSS the outer limit.', bold=True)
        n = self.note(ws, n, 'CGST is not reported by the source statement. It is taken as equal to SGST, which holds for '
                             'intra-State supply. Confirm from the electronic credit ledger.')
        self.note(ws, n, 'Legal handles: section 16(2)(aa) and Rule 36(4); Rule 88D / FORM DRC-01C; Rule 86A where the credit is found ineligible.')

    # ----------------------------------------------- 17 GSTR-1 vs GSTR-3B
    def build_r1(self):
        res = self.R['r1_vs_r3b']; ws = self.sheet('17_R1_vs_R3B')
        if not res.get('available'): return self.unavailable(ws, res, 'GSTR-1 COMPARED WITH GSTR-3B')
        ws['A1'] = 'GSTR-1 COMPARED WITH GSTR-3B — every tax period'; ws['A1'].font = BOLD
        H = ['Return Period','FY','GSTR-1 filing status','Recipients','Documents','GSTR-1 taxable value',
             'GSTR-3B turnover','Difference in value','GSTR-1 tax','GSTR-3B outward tax (3.1(a))',
             'Difference in tax','Short declared in 3B','Remark']
        self.head(ws, 3, H, [13,9,27,11,11,18,18,17,17,19,17,18,72])
        for i, row in enumerate(res['rows'], 4):
            mr = i
            ws.cell(row=i, column=1, value=row['period']); ws.cell(row=i, column=2, value=row['fy'])
            c = ws.cell(row=i, column=3, value=row['status'])
            if row['status'].startswith('NOT FILED') and self.mark: c.fill = MISSING_FILL
            for col, k in zip([4,5], ['sellers','invoices']):
                ws.cell(row=i, column=col, value=row[k] if row[k] is not None else '')
            ws.cell(row=i, column=6, value=row['r1_value'] if row['r1_value'] is not None else '')
            ws.cell(row=i, column=7, value=f'={self.m("D", mr)}')
            ws.cell(row=i, column=8, value=f'=IF(F{i}="","",F{i}-G{i})')
            ws.cell(row=i, column=9, value=row['r1_tax'] if row['r1_tax'] is not None else '')
            ws.cell(row=i, column=10, value=f'={self.m("I", mr)}')
            ws.cell(row=i, column=11, value=f'=IF(I{i}="","",I{i}-J{i})')
            ws.cell(row=i, column=12, value=f'=IF(K{i}="",0,MAX(0,K{i}))')
            rm = ''
            if row['status'] == 'NO RETURN OF ANY KIND':
                rm = 'No GSTR-3B either. Period of registration or a period with no return of any kind.'
            elif row['status'].startswith('NOT FILED'):
                rm = (f'GSTR-1 NOT ON RECORD. GSTR-3B nonetheless declares turnover of '
                      f'Rs.{inr(row["r3b_value"])} and outward tax of Rs.{inr(row["r3b_tax"])}. '
                      f'No invoice-level data exists on the portal for this period.'
                      if row['r3b_value'] or row['r3b_tax'] else
                      'GSTR-1 NOT ON RECORD. GSTR-3B for the period is nil.')
            elif row['diff'] is not None and row['diff'] > 1:
                rm = (f'Tax of Rs.{inr(row["diff"])} declared in GSTR-1 and not carried into GSTR-3B. '
                      + ('GSTR-3B shows NIL turnover against a GSTR-1 supply.' if row['r3b_value'] == 0
                         else 'Turnover carried into GSTR-3B but the tax short-declared.'))
            elif row['r1_tax'] is not None and row['r1_tax'] < -1:
                rm = ('GSTR-1 carries a NEGATIVE tax value — credit notes. '
                      + ('TAXABLE VALUE IS NOT NEGATIVE: the return is internally inconsistent on its face.'
                         if (row['r1_value'] or 0) >= 0 else 'Not carried into GSTR-3B.'))
            elif row['diff'] is not None and row['diff'] < -1:
                rm = 'GSTR-3B tax exceeds GSTR-1 — usually the reverse-charge liability of Table 3.1(d). Reconcile.'
            c = ws.cell(row=i, column=13, value=rm); c.alignment = Alignment(wrap_text=True, vertical='top')
        r = 3 + len(res['rows']) + 1
        ws.cell(row=r, column=1, value='TOTAL').font = BOLD
        for c in [6,7,8,9,10,11,12]:
            LL = get_column_letter(c); ws.cell(row=r, column=c, value=f'=SUM({LL}4:{LL}{r-1})').font = BOLD
        self.box(ws, 4, r, 1, 13, numfrom=4)
        for i in range(4, r+1): ws.cell(row=i, column=3).number_format = 'General'
        self.short_cell = f"'17_R1_vs_R3B'!L{r}"
        n = r + 1
        ws.cell(row=n, column=1, value='Periods with NO GSTR-1 on record').font = BOLD
        ws.cell(row=n, column=3, value=f'=COUNTIF(C4:C{r-1},"NOT FILED*")').font = BOLD
        ws.cell(row=n+1, column=1, value='Turnover declared in GSTR-3B for those periods').font = BOLD
        ws.cell(row=n+1, column=7, value=f'=SUMIF(C4:C{r-1},"NOT FILED*",G4:G{r-1})').font = BOLD
        ws.cell(row=n+2, column=1, value='Outward tax declared in GSTR-3B for those periods').font = BOLD
        ws.cell(row=n+2, column=10, value=f'=SUMIF(C4:C{r-1},"NOT FILED*",J4:J{r-1})').font = BOLD
        for rr in (n, n+1, n+2):
            for cc in (3,7,10): ws.cell(row=rr, column=cc).number_format = NUM
        self.nf_count_cell = f"'17_R1_vs_R3B'!C{n}"
        self.nf_to_cell = f"'17_R1_vs_R3B'!G{n+1}"
        self.nf_tax_cell = f"'17_R1_vs_R3B'!J{n+2}"
        self.note(ws, n+4, 'Compared against GSTR-3B Table 3.1(a) and not against GST OP Total, because OP Total '
                           'includes the reverse-charge liability of Table 3.1(d), which has no counterpart in GSTR-1.')
        self.note(ws, n+5, 'A positive difference is self-assessed tax within the Explanation to section 75(12), recoverable '
                           'under section 79 without a show cause notice. Intimation route: Rule 88C / FORM DRC-01B.')

    # ------------------------------------- 19B credit notes without supply
    def build_credit_notes(self):
        res = self.R['credit_notes']; ws = self.sheet('19B_CREDIT_NOTES')
        if not res.get('available'): return self.unavailable(ws, res, 'CREDIT NOTES TESTED AGAINST THE ORIGINAL SUPPLY')
        ws['A1'] = 'CREDIT NOTES TESTED AGAINST EVERY POSITIVE SUPPLY EVER DECLARED UNDER THE SAME HSN'
        ws['A1'].font = BOLD
        ws['A2'] = ('Section 34(1) requires a credit note to be referable to an identified original tax invoice. '
                    'Section 34(2) allows the reduction in liability only up to 30 November following the end of the '
                    'financial year of that supply.')
        H = ['Return Period','FY','Negative turnover declared','Negative tax declared','HSN code(s)',
             'Positive supply ever declared under the same HSN','As % of ALL positive turnover ever declared','Assessment']
        self.head(ws, 4, H, [13,9,22,20,14,46,22,80])
        r = 5
        if not res['events']:
            self.note(ws, r, 'No tax period declares a negative outward supply. Nothing to test.', bold=True); r += 2
        for e in res['events']:
            ws.cell(row=r, column=1, value=e['period']); ws.cell(row=r, column=2, value=e['fy'])
            ws.cell(row=r, column=3, value=e['turnover']); ws.cell(row=r, column=4, value=e['tax'])
            ws.cell(row=r, column=5, value=', '.join(e['hsn']) if e['hsn'] else 'not reported')
            found = []
            for code, lst in e['prior_positive_same_hsn'].items():
                found.append(f'{code}: NONE — no positive supply under this HSN in any period'
                             if not lst else
                             f'{code}: ' + '; '.join(f'{p} Rs.{inr(v)}' for p, v, _ in lst))
            cell = ws.cell(row=r, column=6, value='\n'.join(found) if found else 'HSN not reported for this period')
            cell.alignment = Alignment(wrap_text=True, vertical='top')
            if any('NONE' in x for x in found) and self.mark: cell.fill = MISSING_FILL
            ws.cell(row=r, column=7, value=e['pct_of_all_positive_turnover'] or 0).number_format = '0.0%'
            none_found = any('NONE' in x for x in found)
            a = ('NO ORIGINAL SUPPLY FOUND. The taxpayer has never declared a positive outward supply under this HSN '
                 'in any tax period on record. A credit note under section 34(1) must be referable to an identified '
                 'original tax invoice. Call for the invoice, the credit note and the contract.'
                 if none_found else
                 'Positive supply under the same HSN does exist. Check the value, the tax head and the section 34(2) '
                 'time limit against the original invoice.')
            c = ws.cell(row=r, column=8, value=a); c.alignment = Alignment(wrap_text=True, vertical='top')
            r += 1
        self.box(ws, 5, max(r-1, 5), 1, 8, numfrom=3)
        r += 1
        r = self.note(ws, r, 'NEGATIVE TAX DECLARED IN GSTR-1 BUT NOT CARRIED INTO GSTR-3B', bold=True)
        H2 = ['Return Period','GSTR-1 tax','GSTR-1 taxable value','GSTR-3B outward tax','Documents','Assessment']
        self.head(ws, r, H2, [13,18,20,19,12,96]); hdr = r; r += 1
        if not res['r1_only']:
            self.note(ws, r, 'None.'); r += 1
        for x in res['r1_only']:
            ws.cell(row=r, column=1, value=x['period']); ws.cell(row=r, column=2, value=x['r1_tax'])
            ws.cell(row=r, column=3, value=x['r1_value']); ws.cell(row=r, column=4, value=x['r3b_tax'])
            ws.cell(row=r, column=5, value=x['docs'])
            if x['inconsistent']:
                imp = abs(x['r1_tax']) / 0.18
                a = (f'INTERNALLY INCONSISTENT ON ITS FACE. The tax column is negative while the taxable value column '
                     f'is not. At 18% the implied credit-note value is about Rs.{inr(imp)}, none of which appears in the '
                     f'taxable value column. The return cannot be read from a summary — draw the full GSTR-1 table by '
                     f'table (B2B, CDNR, Table 12) from the portal.')
            else:
                a = ('Credit notes declared in GSTR-1 and not carried into GSTR-3B. The liability was therefore not '
                     'reduced in GSTR-3B. Establish what the credit notes relate to.')
            c = ws.cell(row=r, column=6, value=a); c.alignment = Alignment(wrap_text=True, vertical='top')
            r += 1
        self.box(ws, hdr+1, max(r-1, hdr+1), 1, 6, numfrom=2)
        self.note(ws, r+1, 'The percentage in column G is measured against every rupee of positive turnover the taxpayer '
                           'has declared across the whole record, which is the test of whether a credit note is proportionate '
                           'to the business that was actually done.')

    # ---------------------------------------- 14 / 15 / 16 / 18 / 19 / 19A
    def build_dormancy(self):
        res = self.R['dormancy']; ws = self.sheet('14_DORMANCY_ITC')
        if not res.get('available'): return self.unavailable(ws, res, 'ITC AVAILED IN NIL-TURNOVER PERIODS')
        ws['A1'] = 'ITC AVAILED IN TAX PERIODS DECLARING NIL OUTWARD SUPPLY'; ws['A1'].font = BOLD
        self.head(ws, 3, ['Return Period','FY','Turnover (3B)','Output tax (3B)','ITC availed','NIL flag',
                          'ITC availed in NIL period','Consecutive NIL count'], [13,9,18,16,16,10,20,18])
        for i, m in enumerate(self.R['master'], 4):
            mr = i
            for col, src in zip([1,2,3,4,5,6,7], ['A','B','D','J','M','W','X']):
                ws.cell(row=i, column=col, value=f'={self.m(src, mr)}')
            ws.cell(row=i, column=8, value='=IF(F4="NIL",1,0)' if i == 4 else f'=IF(F{i}="NIL",H{i-1}+1,0)')
        r = 3 + len(self.R['master']) + 1
        ws.cell(row=r, column=1, value='TOTAL').font = BOLD
        for c in [3,4,5,7]:
            LL = get_column_letter(c); ws.cell(row=r, column=c, value=f'=SUM({LL}4:{LL}{r-1})').font = BOLD
        ws.cell(row=r, column=8, value=f'=MAX(H4:H{r-1})').font = BOLD
        self.box(ws, 4, r, 1, 8, numfrom=3)
        for i in range(4, r+1): ws.cell(row=i, column=6).number_format = 'General'
        self.dorm_cell = None
        n = r + 2
        if res['longest']:
            a, b, c = res['longest']
            ws.cell(row=n, column=1, value=f'Longest unbroken NIL run: {c} tax periods, {a} to {b}.').font = BOLD
            ws.cell(row=n+1, column=1, value=f'ITC availed in NIL periods from {a} onwards').font = BOLD
            ws.cell(row=n+1, column=7, value=res['itc_in_nil_from_longest']).number_format = NUM
            self.dorm_cell = f"'14_DORMANCY_ITC'!G{n+1}"
            n += 2
        n = self.note(ws, n+1, 'Section 16(1) allows credit only on inputs and input services used or intended to be used '
                               'in the course or furtherance of business. Also section 17(5) for blocked credits, and Rule 86A.')
        self.note(ws, n, 'Credit accumulated through a dormancy is what later discharges liability without cash. '
                         'Compare with sheet 19.')

    def build_86b(self):
        res = self.R['rule86b']; ws = self.sheet('15_RULE_86B')
        if not res.get('available'): return self.unavailable(ws, res, 'RULE 86B TEST')
        ws['A1'] = 'RULE 86B — AT LEAST 1 PER CENT OF OUTPUT TAX TO BE PAID IN CASH'; ws['A1'].font = BOLD
        self.head(ws, 3, ['Return Period','Taxable turnover','Output tax','SGST cash paid','Cash proxy (SGST x 2)',
                          'Cash as % of output tax','Rule applicable? (taxable supply > Rs.50 lakh)','Prima facie result'],
                  [13,18,16,15,17,16,22,24])
        for i, m in enumerate(self.R['master'], 4):
            mr = i
            ws.cell(row=i, column=1, value=f'={self.m("A", mr)}')
            ws.cell(row=i, column=2, value=f'={self.m("E", mr)}')
            ws.cell(row=i, column=3, value=f'={self.m("J", mr)}')
            ws.cell(row=i, column=4, value=f'={self.m("T", mr)}')
            ws.cell(row=i, column=5, value=f'=D{i}*2')
            ws.cell(row=i, column=6, value=f'=IFERROR(IF(C{i}<=0,"",E{i}/C{i}),"")')
            ws.cell(row=i, column=7, value=f'=IF(B{i}>5000000,"YES","No")')
            ws.cell(row=i, column=8, value=f'=IF(G{i}="YES",IF(C{i}>0,IF(E{i}/C{i}<0.01,"BELOW 1% - EXAMINE",""),""),"")')
        r = 3 + len(self.R['master'])
        self.box(ws, 4, r, 1, 8, numfrom=2)
        for i in range(4, r+1):
            ws.cell(row=i, column=6).number_format = '0.00%'
            for c in (7, 8): ws.cell(row=i, column=c).number_format = 'General'
        n = self.note(ws, r+2, 'Periods flagged: ' + (', '.join(res['flagged']) if res['flagged'] else 'none'), bold=True)
        n = self.note(ws, n, 'ASSUMPTION: the source statement gives cash paid under the SGST head only. CGST cash is taken '
                             'as equal and IGST cash is not captured at all. Re-test on the electronic cash ledger, and '
                             'cross-check against the challans in sheet 19.')
        self.note(ws, n, 'Exceptions in the proviso to be ruled out first: income tax of more than Rs.1 lakh paid in each of '
                         'the two preceding financial years; refund of more than Rs.1 lakh on zero-rated or inverted duty; '
                         'cumulative discharge of 1% in cash from the start of the financial year; and the status of the person.')

    def build_rcm(self):
        res = self.R['rcm']; ws = self.sheet('16_RCM_CHECK')
        if not res.get('available'): return self.unavailable(ws, res, 'REVERSE CHARGE')
        ws['A1'] = 'REVERSE CHARGE — INWARD SUPPLIES DECLARED AND TAX IMPLIED BY THE RETURNS'; ws['A1'].font = BOLD
        self.head(ws, 3, ['Return Period','FY','RCM inward value (3.1(d))','GST OP Total','Outward tax (3.1(a))',
                          'Implied RCM liability','Implied rate','Remark'], [13,9,22,16,18,20,12,62])
        for i, m in enumerate(self.R['master'], 4):
            mr = i
            for col, src in zip([1,2,3,4,5,6], ['A','B','L','J','I','K']):
                ws.cell(row=i, column=col, value=f'={self.m(src, mr)}')
            ws.cell(row=i, column=7, value=f'=IFERROR(IF(C{i}=0,"",F{i}/C{i}),"")')
            rm = ''
            if m['rcm_value'] > 0 and abs(m['rcm_tax']) < 1:
                rm = (f'RCM inward value of Rs.{inr(m["rcm_value"])} declared but NO RCM tax implied by the return. '
                      f'Tax of about Rs.{inr(m["rcm_value"]*0.05)} at 5% appears not to have been discharged.')
            elif m['rcm_value'] > 0 and abs(m['rcm_tax'] / m['rcm_value'] - 0.05) > 0.002:
                rm = f'Implied rate {m["rcm_tax"]/m["rcm_value"]*100:.2f}% against 5.00% elsewhere. Reconcile.'
            elif m['period'] == res['last_period']:
                rm = 'Last period in which any reverse-charge inward supply was declared.'
            c = ws.cell(row=i, column=8, value=rm); c.alignment = Alignment(wrap_text=True, vertical='top')
        r = 3 + len(self.R['master']) + 1
        ws.cell(row=r, column=1, value='TOTAL').font = BOLD
        for c in [3,4,5,6]:
            LL = get_column_letter(c); ws.cell(row=r, column=c, value=f'=SUM({LL}4:{LL}{r-1})').font = BOLD
        self.box(ws, 4, r, 1, 8, numfrom=3)
        for i in range(4, r+1): ws.cell(row=i, column=7).number_format = '0.00%'
        n = r + 2
        if res['silent_periods']:
            n = self.note(ws, n, f'NO reverse-charge inward supply has been declared in any of the {res["silent_periods"]} '
                                 f'tax periods after {res["last_period"]}, although ITC was availed in most of them.', bold=True)
        self.note(ws, n, 'Sections 9(3) and 9(4) with Notification 13/2017-CT(R). Section 31(3)(f) self-invoicing. '
                         'For a working unit, nil RCM on goods transport, legal services, security services, director '
                         'remuneration and renting of immovable property over a long stretch requires explanation.')

    def build_filing(self):
        res = self.R['filing']; ws = self.sheet('18_RETURN_FILING')
        if not res.get('available'): return self.unavailable(ws, res, 'RETURN FILING RECORD')
        ws['A1'] = 'RETURN FILING RECORD — DATE OF FILING AGAINST THE STATUTORY DUE DATE'; ws['A1'].font = BOLD
        self.head(ws, 3, ['Return Period','FY','GSTR-3B date of filing','Statutory due date','Delay in days',
                          'GSTR-1 status','GSTR-2A report present?','3B turnover','3B output tax'],
                  [13,9,20,20,12,27,18,16,16])
        for i, row in enumerate(res['rows'], 4):
            mr = i
            ws.cell(row=i, column=1, value=row['period']); ws.cell(row=i, column=2, value=row['fy'])
            c = ws.cell(row=i, column=3, value=row['filed']); c.number_format = 'DD-MM-YYYY'
            if row['filed'] is None and self.mark: c.fill = MISSING_FILL
            c = ws.cell(row=i, column=4, value=row['due']); c.number_format = 'DD-MM-YYYY'
            ws.cell(row=i, column=5, value=f'=IF(C{i}="","",MAX(0,C{i}-D{i}))')
            c = ws.cell(row=i, column=6, value=f"='17_R1_vs_R3B'!C{i}")
            c2 = ws.cell(row=i, column=7, value='yes' if row['r2a_present'] else 'NOT SUPPLIED')
            if not row['r2a_present'] and self.mark: c2.fill = MISSING_FILL
            ws.cell(row=i, column=8, value=f'={self.m("D", mr)}')
            ws.cell(row=i, column=9, value=f'={self.m("J", mr)}')
        r = 3 + len(res['rows']) + 1
        ws.cell(row=r, column=1, value='TOTAL').font = BOLD
        for c in [5,8,9]:
            LL = get_column_letter(c); ws.cell(row=r, column=c, value=f'=SUM({LL}4:{LL}{r-1})').font = BOLD
        ws.cell(row=r+1, column=1, value='Tax periods filed after the due date').font = BOLD
        ws.cell(row=r+1, column=5, value=f'=COUNTIF(E4:E{r-1},">0")').font = BOLD
        self.box(ws, 4, r+1, 1, 9, numfrom=5)
        for i in range(4, r+2):
            for c in (3,4): ws.cell(row=i, column=c).number_format = 'DD-MM-YYYY'
            for c in (6,7): ws.cell(row=i, column=c).number_format = 'General'
        self.delay_cell = f"'18_RETURN_FILING'!E{r}"
        n = self.note(ws, r+3, 'The due date is taken as the 20th of the month following the tax period, the monthly filer '
                               'date. If the taxpayer was on QRMP in any period, that period must be re-set before late fee '
                               'is computed. Section 47(1) late fee; section 50 interest where tax was paid late.', bold=True)
        self.note(ws, n, 'Where GSTR-1 is not filed, the recipients of those supplies could not have taken credit through '
                         'GSTR-2B, so the identity of the recipients has to come from the invoices themselves. Section 37 read with Rule 59.')

    def build_cash(self):
        res = self.R['cash']; ws = self.sheet('19_CASH_RECON')
        if not res.get('available'): return self.unavailable(ws, res, 'CASH ACTUALLY PAID TO THE EXCHEQUER')
        ws['A1'] = 'CASH ACTUALLY PAID TO THE EXCHEQUER — every challan since registration'; ws['A1'].font = BOLD
        self.head(ws, 3, ['Payment month','Payment date','CIN','SGST','CGST','IGST','TOTAL','Running total'],
                  [15,14,26,15,15,15,16,16])
        last = self.src_rows.get('PAYMENTS')
        s = _q('SRC_PAYMENTS')
        for i, p in enumerate(res['challans'], 4):
            ws.cell(row=i, column=1, value=p.get('Payment Month'))
            c = ws.cell(row=i, column=2, value=p.get('Payment Date')); c.number_format = 'DD-MM-YYYY'
            ws.cell(row=i, column=3, value=p.get('CIN'))
            for col, k in zip([4,5,6,7], ['SGST','CGST','IGST','TOTAL']):
                ws.cell(row=i, column=col, value=n0(p.get(k)))
            ws.cell(row=i, column=8, value=f'=SUM($G$4:$G{i})')
        r = 3 + len(res['challans']) + 1
        ws.cell(row=r, column=3, value='TOTAL CASH EVER PAID').font = BOLD
        for c in range(4, 8):
            LL = get_column_letter(c); ws.cell(row=r, column=c, value=f'=SUM({LL}4:{LL}{r-1})').font = BOLD
        self.box(ws, 4, r, 1, 8, numfrom=4)
        for i in range(4, r+1):
            ws.cell(row=i, column=2).number_format = 'DD-MM-YYYY'
            for c in (1, 3): ws.cell(row=i, column=c).number_format = 'General'
        self.cash_cell = f"'19_CASH_RECON'!G{r}"
        s0 = r + 2
        ws.cell(row=s0, column=1, value='CASH AGAINST DECLARED LIABILITY').font = BOLD
        F = self.fy_last
        pairs = [('Number of challans since registration', f'=COUNTA(C4:C{r-1})'),
                 ('Total output tax declared in GSTR-3B', f"='11_FY_SUMMARY'!E{F}"),
                 ('Total ITC availed in GSTR-3B', f"='11_FY_SUMMARY'!F{F}"),
                 ('Total cash actually paid, all heads', f'=G{r}'),
                 ('Cash as a proportion of output tax declared', f'=IFERROR(B{s0+4}/B{s0+2},"")'),
                 ('Liability discharged otherwise than in cash', f'=B{s0+2}-B{s0+4}'),
                 ('ITC availed as a proportion of output tax', f'=IFERROR(B{s0+3}/B{s0+2},"")')]
        for i, (a, b) in enumerate(pairs, s0+1):
            ws.cell(row=i, column=1, value=a); ws.cell(row=i, column=2, value=b).number_format = NUM
        ws.cell(row=s0+5, column=2).number_format = '0.0%'
        ws.cell(row=s0+7, column=2).number_format = '0.0%'
        self.box(ws, s0+1, s0+len(pairs), 1, 2)
        t = s0 + len(pairs) + 2
        ws.cell(row=t, column=1, value='CASH BY FINANCIAL YEAR OF THE CHALLAN').font = BOLD
        self.head(ws, t+1, ['FY of challan','Cash paid'], [22, 18])
        for i, (f, v) in enumerate(sorted(res['by_fy'].items()), t+2):
            ws.cell(row=i, column=1, value=f); ws.cell(row=i, column=2, value=v).number_format = NUM
        self.box(ws, t+2, t+1+len(res['by_fy']), 1, 2, numfrom=2)
        self.note(ws, t+3+len(res['by_fy']), 'Payment Month is the month of the challan, which discharges the preceding tax '
                  'period. A run of very small challans is ordinarily late fee, not tax — check against the delays in sheet 18.',
                  bold=True)

    def build_itc_2a(self):
        res = self.R['itc_vs_2a']; ws = self.sheet('19A_ITC_vs_2A')
        if not res.get('available'): return self.unavailable(ws, res, 'ITC AGAINST GSTR-2A — INDEPENDENT ROUTE')
        ws['A1'] = 'INDEPENDENT CORROBORATION — ITC AVAILED AGAINST GSTR-2A, EVERY TAX PERIOD'; ws['A1'].font = BOLD
        ws['A2'] = ('Built from the GSTR-2A reports and the GSTR-3B payment summaries, neither of which feeds sheet 13. '
                    'Two independent routes to the same conclusion. Sheet 13 remains the demand figure because it compares '
                    'head by head; this sheet nets the heads within a period.')
        self.head(ws, 4, ['Return Period','FY','2A report present?','2A sellers','2A invoices','2A taxable value',
                          '2A credit available','ITC availed in GSTR-3B','GAP','Positive gap only','Remark'],
                  [13,9,16,12,13,18,20,19,18,16,66])
        s = _q('SRC_R2A_MONTHLY'); last = self.src_rows.get('R2A_MONTHLY')
        for i, row in enumerate(res['rows'], 5):
            mr = i - 1
            ws.cell(row=i, column=1, value=row['period']); ws.cell(row=i, column=2, value=row['fy'])
            c = ws.cell(row=i, column=3, value='yes' if row['present'] else 'NOT IN DATA')
            if not row['present'] and self.mark: c.fill = MISSING_FILL
            for col, k in zip([4,5,6,7], ['sellers','invoices','taxable','avail']):
                ws.cell(row=i, column=col, value=row[k] if row[k] is not None else '')
            ws.cell(row=i, column=8, value=f'={self.m("M", mr)}')
            ws.cell(row=i, column=9, value=f'=IF($C{i}="NOT IN DATA","",H{i}-G{i})')
            ws.cell(row=i, column=10, value=f'=IF(I{i}="",0,MAX(0,I{i}))')
            rm = ''
            if not row['present']:
                rm = ('No GSTR-2A report in the data for this period. ' +
                      ('ITC availed is nil, so nothing turns on it.' if row['itc'] == 0
                       else f'Rs.{inr(row["itc"])} of credit availed cannot be tested.'))
            elif row['gap'] and row['gap'] > 100000:
                rm = (f'Only {inr(row["sellers"])} suppliers and {inr(row["invoices"])} invoices in 2A carrying '
                      f'Rs.{inr(row["avail"])}, against Rs.{inr(row["itc"])} availed.')
            elif row['gap'] and row['gap'] < -100000:
                rm = 'Credit available but not taken, or credit reversed in this period. Ordinarily timing.'
            c = ws.cell(row=i, column=11, value=rm); c.alignment = Alignment(wrap_text=True, vertical='top')
        r = 4 + len(res['rows']) + 1
        ws.cell(row=r, column=1, value='TOTAL').font = BOLD
        for c in range(4, 11):
            LL = get_column_letter(c); ws.cell(row=r, column=c, value=f'=SUM({LL}5:{LL}{r-1})').font = BOLD
        self.box(ws, 5, r, 1, 11, numfrom=4)
        for i in range(5, r+1): ws.cell(row=i, column=3).number_format = 'General'
        t = r + 2
        ws.cell(row=t, column=1, value='FINANCIAL-YEAR ROLL-UP, AND THE SAME TEST RUN AGAIN OFF THE ANNUAL SUMMARIES').font = BOLD
        self.head(ws, t+1, ['FY','2A credit available (monthly reports)','ITC availed in periods that HAVE a 2A report',
                            'GAP (monthly route)','ITC availed in periods with NO 2A report — not testable',
                            '2A credit available (annual summary)','ITC availed (annual summary)','GAP (annual route)',
                            'Do the two routes agree?'],
                  [13,22,24,18,24,22,22,18,18])
        fys = sorted(self.R['fy'], key=lambda x: int(x.split('-')[0]))
        ann = res.get('annual', {})
        for i, f in enumerate(fys, t+2):
            ws.cell(row=i, column=1, value=f)
            ws.cell(row=i, column=2, value=f'=SUMIFS($G$5:$G${r-1},$B$5:$B${r-1},$A{i},$C$5:$C${r-1},"yes")')
            ws.cell(row=i, column=3, value=f'=SUMIFS($H$5:$H${r-1},$B$5:$B${r-1},$A{i},$C$5:$C${r-1},"yes")')
            ws.cell(row=i, column=4, value=f'=C{i}-B{i}')
            ws.cell(row=i, column=5, value=f'=SUMIFS($H$5:$H${r-1},$B$5:$B${r-1},$A{i},$C$5:$C${r-1},"NOT IN DATA")')
            if f in ann:
                ws.cell(row=i, column=6, value=ann[f]['avail']); ws.cell(row=i, column=7, value=ann[f]['itc'])
                ws.cell(row=i, column=8, value=f'=G{i}-F{i}')
                ws.cell(row=i, column=9, value=f'=IF(ABS((D{i}+E{i})-H{i})<100,"yes","CHECK")')
            else:
                for c in (6,7,8): self.na(ws, i, c, '-')
                ws.cell(row=i, column=9, value='annual summary not supplied')
        r2 = t + 2 + len(fys)
        ws.cell(row=r2, column=1, value='TOTAL').font = BOLD
        for c in range(2, 9):
            LL = get_column_letter(c); ws.cell(row=r2, column=c, value=f'=SUM({LL}{t+2}:{LL}{r2-1})').font = BOLD
        self.box(ws, t+2, r2, 1, 9, numfrom=2)
        for i in range(t+2, r2+1): ws.cell(row=i, column=9).number_format = 'General'
        self.note(ws, r2+2, 'Column E isolates credit availed in periods for which no GSTR-2A report exists. The monthly route '
                            'cannot test it; the annual summary silently includes it. Keeping it in its own column is what '
                            'makes the two routes reconcile.', bold=True)

    def build_hsn(self):
        res = self.R['hsn']; ws = self.sheet('12_HSN_PROFILE')
        if not res.get('available'): return self.unavailable(ws, res, 'HSN PROFILE')
        ws['A1'] = 'HSN PROFILE — WHAT THE TAXPAYER SAYS IT SUPPLIES, IN THE ORDER IT SAID IT'; ws['A1'].font = BOLD
        self.head(ws, 3, ['Return Period','FY','HSN code','Taxable value','Total GST','Note'], [13,9,14,18,18,70])
        for i, (p, code, v, t) in enumerate(res['rows'], 4):
            ws.cell(row=i, column=1, value=p); ws.cell(row=i, column=2, value=fy_of(p))
            ws.cell(row=i, column=3, value=code); ws.cell(row=i, column=4, value=v); ws.cell(row=i, column=5, value=t)
            n = 'NEGATIVE — credit note. See 19B_CREDIT_NOTES.' if v < 0 else ''
            c = ws.cell(row=i, column=6, value=n); c.alignment = Alignment(wrap_text=True, vertical='top')
        r = 3 + len(res['rows']) + 1
        ws.cell(row=r, column=1, value='TOTAL').font = BOLD
        for c in (4, 5):
            LL = get_column_letter(c); ws.cell(row=r, column=c, value=f'=SUM({LL}4:{LL}{r-1})').font = BOLD
        self.box(ws, 4, r, 1, 6, numfrom=4)
        for i in range(4, r+1): ws.cell(row=i, column=3).number_format = 'General'
        n = self.note(ws, r+2, 'HSN codes in the order they first appear: ' + ' -> '.join(res['codes']), bold=True)
        self.note(ws, n, 'A drift away from the registered nature of business, and in particular the appearance of goods '
                         'HSNs in a services taxpayer, points to disposal of capital goods. Section 18(6) with Rule 44(6): '
                         'tax is the higher of the credit taken less the prescribed reduction, or the tax on transaction value.')

    def build_limitation(self):
        ws = self.sheet('24_LIMITATION')
        ws['A1'] = 'LIMITATION — INDICATIVE OUTER DATES BY FINANCIAL YEAR'; ws['A1'].font = BOLD
        ws['A2'] = ('COMPUTED FROM THE STATUTE ALONE. Extension notifications are NOT applied and several have been the '
                    'subject of litigation. Verify every date against the notifications in force before acting on it.')
        self.head(ws, 4, ['FY','Annual return due','Section 73 — SCN by','Section 73 — order by','Section 73 still open?',
                          'Section 74 — SCN by','Section 74 — order by','Section 74 still open?','Regime',
                          'Section 74A — SCN by (42 months)','Section 74A — order by (12 months, +6 extendable)',
                          'Section 74A still open?'],
                  [10,18,18,18,18,18,18,18,40,20,22,16])
        for i, l in enumerate(self.R['limitation'], 5):
            ws.cell(row=i, column=1, value=l['fy'])
            c = ws.cell(row=i, column=2, value=l['annual_return_due']); c.number_format = 'DD-MM-YYYY'
            if l.get('is74a'):
                for col in (3, 4, 5, 6, 7, 8):
                    ws.cell(row=i, column=col, value='not applicable')
                for col, k in zip([10, 11], ['s74a_scn_by', 's74a_order_by']):
                    c = ws.cell(row=i, column=col, value=l[k]); c.number_format = 'DD-MM-YYYY'
                ws.cell(row=i, column=12, value='OPEN' if l['s74a_open'] else 'TIME-BARRED')
            else:
                for col, k in zip([3,4,6,7], ['s73_scn_by','s73_order_by','s74_scn_by','s74_order_by']):
                    c = ws.cell(row=i, column=col, value=l[k]); c.number_format = 'DD-MM-YYYY'
                c = ws.cell(row=i, column=5, value='OPEN' if l['s73_open'] else 'TIME-BARRED')
                if not l['s73_open'] and self.mark: c.fill = MISSING_FILL
                ws.cell(row=i, column=8, value='OPEN' if l['s74_open'] else 'TIME-BARRED')
                for col in (10, 11, 12): ws.cell(row=i, column=col, value='not applicable')
            ws.cell(row=i, column=9, value=l['regime'])
        r = 4 + len(self.R['limitation'])
        self.box(ws, 5, r, 1, 12)
        for i in range(5, r+1):
            for c in (3, 4, 5, 6, 7, 8, 9, 10, 11, 12):
                if not isinstance(ws.cell(row=i, column=c).value, datetime.date):
                    ws.cell(row=i, column=c).number_format = 'General'
        n = self.note(ws, r+2, 'Where section 73 is time-barred, a demand for that year can be raised only under section 74, '
                               'which requires fraud, wilful misstatement or suppression of facts to be alleged and made out '
                               'on evidence. That decision has to be taken before the evidence is gathered, not after.', bold=True)
        self.note(ws, n, 'Section 74A applies to demands for FY 2024-25 onwards (sections 73(12) and 74(12) confine 73 and 74 to '
                         'FY 2023-24 and earlier): notice within 42 months of the due date of the annual return (s.74A(2)), order '
                         'within 12 months of the notice, extendable by up to 6 months (s.74A(10)). One limitation for both the '
                         'non-fraud and the fraud case; the difference lies in the penalty.')


    # ------------------------------------------------- 19C GSTR-7 TDS (v1.5)
    def _src_range(self, code, fy, colname):
        """Contiguous rows of an SRC_ sheet that belong to one FY (SRC sheets are sorted by period),
        as an A1 range, so an FY total stays a live SUM over the source cells."""
        if code not in self.src_rows: return None
        rows = sorted(self.ing.data[code], key=lambda r: (period_key(r['_period']), r.get('SNo') or 0))
        idx = [i for i, r in enumerate(rows, 2) if r.get('_fy') == fy]
        col = self.src_col(code, colname)
        if not idx or col is None: return None
        return f"{_q(f'SRC_{code}'[:31])}!{col}{idx[0]}:{col}{idx[-1]}"

    def _master_range(self, fy, col, upto=None):
        rows = [i for i, m in enumerate(self.R['master'], 4)
                if m['fy'] == fy and (upto is None or period_key(m['period']) <= period_key(upto))]
        if not rows: return None
        return f"'10_MONTHLY_MASTER'!{col}{rows[0]}:{col}{rows[-1]}"

    def build_tds(self):
        res = self.R['tds']; ws = self.sheet('19C_TDS_GSTR7')
        if not res.get('available'):
            return self.unavailable(ws, res, 'TAX DEDUCTED AT SOURCE BY GOVERNMENT (GSTR-7)')
        ws['A1'] = ('TAX DEDUCTED AT SOURCE FROM THIS TAXPAYER BY GOVERNMENT DEDUCTORS (GSTR-7) — '
                    'what the deductors paid for, against what the taxpayer declared'); ws['A1'].font = BOLD
        ws['A2'] = ('Section 51 CGST Act with Notification 50/2018-CT: a Government department, local authority or Government '
                    'agency paying a supplier under a contract above Rs.2.5 lakh deducts 1% CGST + 1% SGST (or 2% IGST) of the '
                    'value of supply, excluding the tax on the invoice (s.51(2)), files GSTR-7 (Rule 66), and the amount is '
                    'credited to the supplier\'s electronic cash ledger (Rule 87(9)). TDS / 2% is therefore the value of '
                    'supplies the deductors have PAID for — a floor on turnover, not the whole of it.')
        ws['A2'].alignment = Alignment(wrap_text=True, vertical='top')
        ws.merge_cells('A2:T2'); ws.row_dimensions[2].height = 48
        ws['A3'] = 'Rate of tax ASSUMED for the indicative figure only — confirm from the contracts and invoices'
        ws['A3'].font = BOLD
        rc = ws['H3']; rc.value = 0.18; rc.number_format = '0%'; rc.font = BOLD; rc.border = BD
        self.tds_rate_cell = "'19C_TDS_GSTR7'!$H$3"
        H = ['Return Period','FY','No. of deductors','SGST deducted','CGST deducted','IGST deducted','Cess',
             'Total GST (as reported)','Rounding check (heads + cess - total)','VALUE PAID FOR by the deductors (=(total-cess)/2%)',
             'Cumulative value paid for, in the FY','GSTR-3B status for the month','GSTR-3B taxable turnover, the month',
             'Cumulative GSTR-3B taxable turnover, FY to this month','CUMULATIVE SHORTFALL (value paid for less declared)',
             'GSTR-3B outward CGST, the month','CGST deducted exceeds CGST declared?','GSTR-1 taxable value, the month',
             'Above Rs.50 lakh (Rule 86B)?','Observation']
        self.head(ws, 5, H, [12,9,10,13,13,12,8,14,12,18,18,20,17,18,18,15,13,16,12,70])
        P = lambda r: f'$A{r}'
        mrow = {m['period']: i for i, m in enumerate(self.R['master'], 4)}
        ML = getattr(self, 'master_last', 3)
        M = _q('10_MONTHLY_MASTER')
        first = 6
        for i, x in enumerate(res['rows'], first):
            ws.cell(row=i, column=1, value=x['period'])
            ws.cell(row=i, column=2, value=x['fy'])
            for col, nm in zip(range(3, 9), ['No Of Deductors','SGST','CGST','IGST','CESS','Total GST']):
                ws.cell(row=i, column=col, value=self.lookup('R7_TDS', nm, 'Return Period', P(i)))
            ws.cell(row=i, column=9, value=f'=D{i}+E{i}+F{i}+G{i}-H{i}')
            ws.cell(row=i, column=10, value=f'=(H{i}-G{i})/0.02')
            ws.cell(row=i, column=11, value=f'=SUMIF($B${first}:B{i},B{i},$J${first}:J{i})')
            if res['have_3b']:
                c = ws.cell(row=i, column=12, value=x['r3b_status'])
                if x['r3b_status'] not in ('FILED', 'NOT YET DUE') and self.mark: c.fill = MISSING_FILL
                if x['period'] in mrow:
                    ws.cell(row=i, column=13, value=f'=INDEX({M}!$E$4:$E${ML},MATCH($A{i},{M}!$A$4:$A${ML},0))')
                    ws.cell(row=i, column=16, value=f'=INDEX({M}!$G$4:$G${ML},MATCH($A{i},{M}!$A$4:$A${ML},0))')
                    ws.cell(row=i, column=17, value=f'=IF(E{i}>P{i}+1,"YES","")')
                else:
                    self.na(ws, i, 13, 'NO 3B'); self.na(ws, i, 16, 'NO 3B')
                rng = self._master_range(x['fy'], 'E', upto=x['period'])
                ws.cell(row=i, column=14, value=f'=SUM({rng})' if rng else 0)
                if x['r3b_status'] == 'NOT YET DUE':
                    self.na(ws, i, 15, 'NOT YET DUE')
                else:
                    ws.cell(row=i, column=15, value=f'=MAX(0,K{i}-N{i})')
            else:
                for col in (12, 13, 14, 15, 16, 17): self.na(ws, i, col, '3B NOT SUPPLIED')
            if res['have_r1']:
                if x['r1_status'] in ('NOT FILED', 'NOT YET DUE'):
                    self.na(ws, i, 18, x['r1_status'])
                else:
                    ws.cell(row=i, column=18, value=self.lookup('R1_MONTHLY', 'Taxable Value', 'Return Period', P(i)))
            else:
                self.na(ws, i, 18, 'R1 NOT SUPPLIED')
            ws.cell(row=i, column=19, value=f'=IF(J{i}>5000000,"YES","")')
            ws.cell(row=i, column=20, value='; '.join(x['obs']))
        last = first + len(res['rows']) - 1
        t = last + 1
        ws.cell(row=t, column=1, value='TOTAL').font = BOLD
        for c in (4, 5, 6, 7, 8, 10):
            L = get_column_letter(c); ws.cell(row=t, column=c, value=f'=SUM({L}{first}:{L}{last})').font = BOLD
        self.box(ws, first, t, 1, 20, numfrom=3)
        for r in range(first, t + 1):
            for c in (1, 2, 12, 17, 19, 20):
                ws.cell(row=r, column=c).number_format = 'General'
                ws.cell(row=r, column=c).alignment = Alignment(wrap_text=True, vertical='top')
        self.tds_first, self.tds_last = first, last

        # ---- financial-year roll-up
        r0 = t + 3
        ws.cell(row=r0 - 1, column=1, value='FINANCIAL-YEAR POSITION').font = BOLD
        FH = ['FY','Months with TDS','TDS deducted','VALUE PAID FOR by the deductors','GSTR-3B months on record',
              'GSTR-3B taxable turnover, months on record','SHORTFALL against GSTR-3B (months not yet due left out)','Indicative tax on the shortfall at the assumed rate',
              'GSTR-1 taxable value','SHORTFALL against GSTR-1 (months not yet due left out)','SGST deducted (credited to cash ledger)',
              'SGST paid in cash in GSTR-3B','SGST TDS NOT ABSORBED (balance building up)','ITC availed in GSTR-3B',
              'GSTR-9 required on receipts subject to TDS alone (> Rs.2 crore)?','GSTR-9C required (> Rs.5 crore)?']
        for j, h in enumerate(FH, 1):
            x = ws.cell(row=r0, column=j, value=h); x.font = BOLD; x.border = BD
            x.alignment = Alignment(wrap_text=True, vertical='top', horizontal='center')
        ws.row_dimensions[r0].height = 75
        B = f'$B${first}:$B${last}'
        fys = sorted(res['fy'], key=lambda f: int(f.split('-')[0]))
        for i, f in enumerate(fys, r0 + 1):
            a = res['fy'][f]
            ws.cell(row=i, column=1, value=f)
            ws.cell(row=i, column=2, value=f'=COUNTIF({B},$A{i})')
            ws.cell(row=i, column=3, value=f'=SUMIF({B},$A{i},$H${first}:$H${last})-SUMIF({B},$A{i},$G${first}:$G${last})')
            ws.cell(row=i, column=4, value=f'=SUMIF({B},$A{i},$J${first}:$J${last})')
            if res['have_3b']:
                ws.cell(row=i, column=5, value=f'=COUNTIF({M}!$B$4:$B${ML},$A{i})')
                ws.cell(row=i, column=6, value=f'=SUMIF({M}!$B$4:$B${ML},$A{i},{M}!$E$4:$E${ML})')
                ws.cell(row=i, column=7, value=f'=MAX(0,SUMIFS($J${first}:$J${last},{B},$A{i},$L${first}:$L${last},"<>NOT YET DUE")-F{i})')
                ws.cell(row=i, column=8, value=f'=G{i}*$H$3')
                ws.cell(row=i, column=14, value=f'=SUMIF({M}!$B$4:$B${ML},$A{i},{M}!$M$4:$M${ML})')
            else:
                for c in (5, 6, 7, 8, 14): self.na(ws, i, c, '3B NOT SUPPLIED')
            if res['have_r1']:
                rng = self._src_range('R1_MONTHLY', f, 'Taxable Value')
                ws.cell(row=i, column=9, value=f'=SUM({rng})' if rng else 0)
                ws.cell(row=i, column=10, value=f'=MAX(0,SUMIFS($J${first}:$J${last},{B},$A{i},$R${first}:$R${last},"<>NOT YET DUE")-I{i})')
            else:
                for c in (9, 10): self.na(ws, i, c, 'R1 NOT SUPPLIED')
            ws.cell(row=i, column=11, value=f'=SUMIF({B},$A{i},$D${first}:$D${last})')
            if res['have_tax_paid']:
                ws.cell(row=i, column=12, value=f'=SUMIF({M}!$B$4:$B${ML},$A{i},{M}!$T$4:$T${ML})')
                ws.cell(row=i, column=13, value=f'=MAX(0,K{i}-L{i})')
            else:
                for c in (12, 13): self.na(ws, i, c, '3B NOT SUPPLIED')
            ws.cell(row=i, column=15, value=f'=IF(D{i}>20000000,"YES","not on these receipts alone")')
            ws.cell(row=i, column=16, value=f'=IF(D{i}>50000000,"YES","not on these receipts alone")')
        fl = r0 + len(fys)
        self.box(ws, r0 + 1, fl, 1, 16, numfrom=2)
        for r in range(r0 + 1, fl + 1):
            for c in (1, 15, 16): ws.cell(row=r, column=c).number_format = 'General'
        self.tds_fy_rows = {f: i for i, f in enumerate(fys, r0 + 1)}

        # ---- the tests, their legal basis, and whether each could be run
        r = fl + 3
        ws.cell(row=r - 1, column=1, value='THE CROSS-CHECKS — legal basis, and whether each could be run on the data held').font = BOLD
        TH = ['No.','Test','Provision','Reports it needs','Status','Result']
        for j, h in enumerate(TH, 1):
            x = ws.cell(row=r, column=j, value=h); x.font = BOLD; x.border = BD
        ws.merge_cells(start_row=r, start_column=6, end_row=r, end_column=20)
        r += 1
        for k, (test, prov, needs, ok, result) in enumerate(res['tests'], 1):
            vals = [k, test, prov, needs, 'RUN' if ok else 'NOT RUN — upload the reports named', result]
            for j, v in enumerate(vals, 1):
                c = ws.cell(row=r, column=j, value=v); c.border = BD
                c.alignment = Alignment(wrap_text=True, vertical='top')
            if not ok and self.mark: ws.cell(row=r, column=5).fill = MISSING_FILL
            ws.merge_cells(start_row=r, start_column=6, end_row=r, end_column=20)
            ws.row_dimensions[r].height = 62
            r += 1
        r += 1
        r = self.note(ws, r, 'HOW TO READ THE SHORTFALL. Payment month is not invoice month, so a single month can show a '
                             'shortfall that is only timing. The firm test is the CUMULATIVE column: for services (works contract '
                             'is a service) the time of supply is the earlier of invoice and payment (s.13(2)), and for goods the '
                             'invoice is issued on removal (s.31(1)), so a supply the deductors have paid for by a month must have been '
                             'declared by that month. Exception: an advance for goods is not taxed on receipt (Notification '
                             '66/2017-CT) — rule it out on the contract before relying on a shortfall.', bold=True)
        r = self.note(ws, r, 'The value paid for is a FLOOR. Supplies to buyers who do not deduct, and contracts of Rs.2.5 lakh or '
                             'less, carry no TDS and are not in it. A GSTR-3B turnover above it proves nothing; one below it is a '
                             'shortfall on business with the deductors alone.')
        self.note(ws, r, 'The rate in H3 is an assumption used only for the indicative tax in column H of the FY block. It is not '
                         'carried into 20_FINDINGS. Works contracts for Government are ordinarily 18% under entry 3 of Notification '
                         '11/2017-CT(Rate) as amended, but the rate turns on the contract — confirm it before any figure is quoted.')
        ws.freeze_panes = 'C6'


    # ------------------------------------------ 19D GSTR-7 party-wise (v1.7)
    def build_tds_parties(self):
        res = self.R.get('tds_detail') or {}
        ws = self.sheet('19D_TDS_PARTIES')
        if not res.get('available'):
            return self.unavailable(ws, res, 'GSTR-7 PARTY-WISE DETAIL — who deducted, on what value')
        ws['A1'] = 'GSTR-7 PARTY-WISE DETAIL — who the parties are, whether section 51 lets them deduct, and what was deducted'
        ws['A1'].font = BOLD
        ws['A2'] = ('Export titled for GSTIN: ' + (', '.join(res['subject']) or 'not stated')
                    + ('   — DOES NOT MATCH THE CASE GSTIN' if res['subject_mismatch'] else ''))
        ws['A2'].font = BOLD
        if res['subject_mismatch'] and self.mark: ws['A2'].fill = MISSING_FILL
        ws['A3'] = 'WHO IS WHO. ' + res['role']
        ws['A3'].alignment = Alignment(wrap_text=True, vertical='top'); ws.merge_cells('A3:K3')
        ws.row_dimensions[3].height = 62
        ws['A4'] = ('WHO MAY DEDUCT. Section 51(1) read with Notification 50/2018-CT: (a) a department or establishment of '
                    'the Central or a State Government; (b) a local authority; (c) a Governmental agency; (d) an authority, '
                    'board or body set up by Parliament, a State Legislature or a Government with 51% or more Government '
                    'equity or control; a society established by a Government or local authority; a public sector '
                    'undertaking. A private proprietor, firm or company outside these is not a deductor, and a tax-deductor '
                    'registration granted to one is itself irregular.')
        ws['A4'].alignment = Alignment(wrap_text=True, vertical='top'); ws.merge_cells('A4:K4')
        ws.row_dimensions[4].height = 62
        S = _q('SRC_R7_DETAIL'); last = self.src_rows['R7_DETAIL']
        cg = self.src_col('R7_DETAIL', 'Deductee GSTIN'); cv = self.src_col('R7_DETAIL', 'Taxable Value')
        ct = self.src_col('R7_DETAIL', 'Total GST'); cp = self.src_col('R7_DETAIL', 'Return Period')
        rg = lambda c: f'{S}!${c}$2:${c}${last}'
        r = 6
        H = ['GSTIN listed','Trade name','Registration type (GSTIN position 14)','PAN / TAN holder type (PAN position 4)',
             'Can it deduct under section 51?','Months listed','Lines','Taxable value','TDS (Total GST)','Share of value',
             'Note']
        self.head(ws, r, H, [18,26,24,24,58,22,7,15,13,9,40]); ws.freeze_panes = None
        p0 = r + 1
        for i, p in enumerate(res['parties'], p0):
            vals = [p['gstin'], ', '.join(p['names']), p['reg'], p['holder'], p['eligibility'], ', '.join(p['months'])]
            for j, v in enumerate(vals, 1): ws.cell(row=i, column=j, value=v)
            ws.cell(row=i, column=7, value=f'=COUNTIF({rg(cg)},$A{i})')
            ws.cell(row=i, column=8, value=f'=SUMIF({rg(cg)},$A{i},{rg(cv)})')
            ws.cell(row=i, column=9, value=f'=SUMIF({rg(cg)},$A{i},{rg(ct)})')
            if p['eligibility'].startswith('NOT A PERMITTED') and self.mark:
                ws.cell(row=i, column=5).fill = MISSING_FILL
        pl = p0 + len(res['parties']) - 1
        for i in range(p0, pl + 1):
            ws.cell(row=i, column=10, value=f'=IFERROR(H{i}/H${pl+1},"")').number_format = '0.0%'
        ws.cell(row=pl + 1, column=1, value='TOTAL').font = BOLD
        for c in (7, 8, 9):
            L = get_column_letter(c); ws.cell(row=pl + 1, column=c, value=f'=SUM({L}{p0}:{L}{pl})').font = BOLD
        self.box(ws, p0, pl + 1, 1, 11, numfrom=7)
        for i in range(p0, pl + 2):
            for c in range(1, 12): ws.cell(row=i, column=c).alignment = Alignment(wrap_text=True, vertical='top')
            ws.cell(row=i, column=10).number_format = '0.0%'
        # ---- month reconciliation with the summary report
        r = pl + 4
        ws.cell(row=r - 1, column=1, value='MONTH BY MONTH — the detail against the summary report (R7 FILINGS IN THE YEAR)').font = BOLD
        H = ['Return Period','Lines','Parties','Taxable value (detail)','TDS (detail)','Parties per summary',
             'TDS per summary','TDS difference','Value implied by the summary (TDS / 2%)','Agrees?']
        for j, h in enumerate(H, 1):
            x = ws.cell(row=r, column=j, value=h); x.font = BOLD; x.border = BD
            x.alignment = Alignment(wrap_text=True, vertical='top', horizontal='center')
        m0 = r + 1
        for i, m in enumerate(res['months'], m0):
            ws.cell(row=i, column=1, value=m['period'])
            ws.cell(row=i, column=2, value=f'=COUNTIF({rg(cp)},$A{i})')
            ws.cell(row=i, column=3, value=m['parties'])
            ws.cell(row=i, column=4, value=f'=SUMIF({rg(cp)},$A{i},{rg(cv)})')
            ws.cell(row=i, column=5, value=f'=SUMIF({rg(cp)},$A{i},{rg(ct)})')
            if 'R7_TDS' in self.src_rows:
                ws.cell(row=i, column=6, value=self.lookup('R7_TDS', 'No Of Deductors', 'Return Period', f'$A{i}'))
                ws.cell(row=i, column=7, value=self.lookup('R7_TDS', 'Total GST', 'Return Period', f'$A{i}'))
                ws.cell(row=i, column=8, value=f'=E{i}-G{i}')
                ws.cell(row=i, column=9, value=f'=G{i}/0.02')
                ws.cell(row=i, column=10, value=f'=IF(AND(ABS(H{i})<=1+0.5*B{i},C{i}=F{i}),"YES","NO — CHECK")')
            else:
                for c in (6, 7, 8, 9, 10): self.na(ws, i, c, 'SUMMARY NOT SUPPLIED')
        ml = m0 + len(res['months']) - 1
        self.box(ws, m0, ml, 1, 10, numfrom=2)
        r = ml + 2
        if res['missing_months']:
            r = self.note(ws, r, 'DETAIL NOT YET UPLOADED for ' + ', '.join(res['missing_months']) + ' — months the summary '
                          'shows TDS for. Export "R7 Filing Details For The Month" for each and upload under the same GSTIN.',
                          bold=True)
            if self.mark: ws.cell(row=r - 1, column=1).fill = MISSING_FILL
        if res['near_threshold']:
            r = self.note(ws, r + 1, f'{len(res["near_threshold"])} line(s) carry a taxable value just above the Rs.2.5 lakh '
                          f'contract value at which section 51 deduction begins: '
                          + '; '.join(f'{p} {g} Rs.{inr(v)}' for p, g, v in res['near_threshold'])
                          + '. The threshold applies to the value of the contract, not to each invoice — establish what contract '
                            'each line was deducted under.')
        if res['off_rate']:
            r = self.note(ws, r + 1, 'Lines where tax deducted is not 1% + 1% (or 2% IGST) of the taxable value: '
                          + '; '.join(f'{p} {g} value Rs.{inr(v)} CGST Rs.{inr(c)}' for p, g, v, c in res['off_rate']), bold=True)
        else:
            r = self.note(ws, r + 1, 'Every line was deducted at 1% CGST + 1% SGST of its taxable value (to the rupee).')
        ws.column_dimensions['A'].width = 18

    # ------------------------------------------------------- 20 FINDINGS
    def findings(self):
        R = self.R; F = []
        cn = R['credit_notes']
        if cn.get('available'):
            for e in cn['events']:
                none_found = any(not v for v in e['prior_positive_same_hsn'].values()) or not e['hsn']
                F.append(dict(
                    t=f'Credit note of Rs.{inr(abs(e["turnover"]))} in {e["period"]}'
                      + (' with no traceable original supply' if none_found else ''),
                    w=(f'GSTR-3B for {e["period"]} declares taxable turnover of MINUS Rs.{inr(abs(e["turnover"]))} and '
                       f'outward tax of MINUS Rs.{inr(abs(e["tax"]))}'
                       + (f', under HSN {", ".join(e["hsn"])}' if e['hsn'] else '') + '. '
                       + ('The taxpayer has NEVER declared a positive outward supply under that HSN in any tax period '
                          'on record. ' if none_found else '')
                       + f'It equals {e["pct_of_all_positive_turnover"]*100:.1f}% of every rupee of positive turnover '
                         f'declared since registration.'),
                    a=abs(e['tax']), s='19B_CREDIT_NOTES; 12_HSN_PROFILE',
                    p=('Section 34(1) and 34(2) — a credit note must be referable to an identified original tax invoice '
                       'and the reduction is available only up to 30 November following the end of the financial year of '
                       'that supply. Section 15.'),
                    r='VERY HIGH'))
            for x in cn['r1_only']:
                F.append(dict(
                    t=f'{x["period"]}: credit note of Rs.{inr(abs(x["r1_tax"]))} in GSTR-1, not carried into GSTR-3B',
                    w=(f'GSTR-1 declares tax of MINUS Rs.{inr(abs(x["r1_tax"]))} against a taxable value of '
                       f'{"PLUS" if x["r1_value"]>=0 else "MINUS"} Rs.{inr(abs(x["r1_value"]))}'
                       + (f' on {inr(x["docs"])} documents' if x['docs'] else '') + '. '
                       + ('The two figures cannot both be right: the return is internally inconsistent on its face. '
                          f'At 18% the implied credit-note value is about Rs.{inr(abs(x["r1_tax"])/0.18)}, none of which '
                          'appears in the taxable value column. ' if x['inconsistent'] else '')
                       + 'Nothing was carried into GSTR-3B, so no amount is demanded here. The entry is listed because '
                         'of what it points to.'),
                    a=0.0, s='19B_CREDIT_NOTES; 17_R1_vs_R3B',
                    p='Section 34, section 37, section 70. Also establish whether any audit, scrutiny or intelligence '
                      'proceeding was on foot in that period.',
                    r='VERY HIGH — lead'))
        ex = R['excess_itc']; i2 = R['itc_vs_2a']
        if ex.get('available'):
            corr = ''
            if i2.get('available') and i2.get('annual'):
                worst = max(i2['annual'].items(), key=lambda kv: kv[1]['gap'], default=None)
                if worst and worst[1]['gap'] > 0:
                    corr = (f' INDEPENDENTLY CORROBORATED: for FY {worst[0]} the GSTR-2A summary shows credit available '
                            f'of Rs.{inr(worst[1]["avail"])} against ITC availed of Rs.{inr(worst[1]["itc"])} — a gap of '
                            f'Rs.{inr(worst[1]["gap"])}, reached from reports that do not feed this computation.')
            F.append(dict(
                t='Excess input tax credit availed over GSTR-2A',
                w=(f'ITC availed in GSTR-3B exceeds credit available in GSTR-2A, in at least one head, in '
                   f'{ex["months_excess"]} of the {len(ex["rows"])} tax periods (SGST in {ex["months_excess_sgst"]}, '
                   f'IGST in {ex["months_excess_igst"]}). Net excess after setting off the periods of short claim is '
                   f'Rs.{inr(ex["net"])}; gross, counting only periods of over-claim, Rs.{inr(ex["gross"])}.' + corr),
                a=ex['net'], s='13_EXCESS_ITC; 19A_ITC_vs_2A',
                p='Section 16(2)(aa) and Rule 36(4). Rule 88D / FORM DRC-01C. Rule 86A for blocking.',
                r='VERY HIGH'))
        r1 = R['r1_vs_r3b']
        if r1.get('available'):
            det = [x for x in r1['rows'] if x['short'] > 1]
            if det:
                F.append(dict(
                    t='Output tax declared in GSTR-1 but not discharged in GSTR-3B',
                    w=('; '.join(f'{x["period"]} Rs.{inr(x["short"])}' for x in det)
                       + f'. Rs.{inr(r1["short_total"])} in all.'),
                    a=r1['short_total'], s='17_R1_vs_R3B',
                    p=('Section 75(12) with Explanation — self-assessed tax includes tax payable on outward supplies '
                       'declared in GSTR-1 but not included in GSTR-3B; recoverable under section 79 without a show '
                       'cause notice. Rule 88C / FORM DRC-01B.'),
                    r='HIGH — directly recoverable'))
            if r1['not_filed']:
                F.append(dict(
                    t=f'GSTR-1 not filed for {len(r1["not_filed"])} tax periods',
                    w=(f'No GSTR-1 is on record for {", ".join(r1["not_filed"])}. GSTR-3B was filed for every one of them. '
                       f'Rs.{inr(r1["not_filed_turnover"])} of turnover and Rs.{inr(r1["not_filed_tax"])} of outward tax '
                       f'therefore sit in GSTR-3B with NO invoice-level record on the portal, and no way for any recipient '
                       f'to have taken credit through GSTR-2B.'),
                    a=0.0, s='17_R1_vs_R3B; 18_RETURN_FILING',
                    p='Section 37 read with Rule 59. Section 47(1) late fee.',
                    r='VERY HIGH — evidentiary' if r1['not_filed_turnover'] > 0 else 'MEDIUM'))
        dm = R['dormancy']
        if dm.get('available') and dm['longest'] and dm['longest'][2] >= 6:
            a, b, c = dm['longest']
            F.append(dict(
                t=f'Credit availed through a {c}-period dormancy',
                w=(f'NIL outward supply in {c} consecutive tax periods, {a} to {b}. ITC continued to be availed — '
                   f'Rs.{inr(dm["itc_in_nil_from_longest"])} in NIL periods from {a} onwards. Across the whole record '
                   f'{dm["nil_months"]} periods are NIL.'),
                a=dm['itc_in_nil_from_longest'], s='14_DORMANCY_ITC',
                p='Section 16(1) — credit admissible only on inputs used or intended to be used in the course or '
                  'furtherance of business. Section 17(5). Rule 86A.',
                r='HIGH'))
        rc = R['rcm']
        if rc.get('available') and (rc['silent_periods'] >= 12 or rc['value_without_tax']):
            amt = sum(v * 0.05 for _, v in rc['value_without_tax'])
            F.append(dict(
                t='Reverse charge liability not declared',
                w=(f'Reverse-charge inward supplies totalling Rs.{inr(rc["declared"])} were declared up to '
                   f'{rc["last_period"]}, with tax of Rs.{inr(rc["tax"])} implied by the returns. '
                   f'No reverse-charge inward supply has been declared in any of the {rc["silent_periods"]} tax periods '
                   f'since, although ITC was availed in most of them.'
                   + (''.join(f' {p} declares RCM inward value of Rs.{inr(v)} with no RCM tax implied at all.'
                              for p, v in rc['value_without_tax']))),
                a=amt, s='16_RCM_CHECK',
                p='Sections 9(3) and 9(4) with Notification 13/2017-CT(R). Section 31(3)(f).',
                r='MEDIUM — quantum to be established'))
        ca = R['cash']
        if ca.get('available'):
            F.append(dict(
                t='Cash actually contributed to the exchequer',
                w=(f'{ca["count"]} challans in all, Rs.{inr(ca["totals"]["TOTAL"])} (SGST Rs.{inr(ca["totals"]["SGST"])}, '
                   f'CGST Rs.{inr(ca["totals"]["CGST"])}, IGST Rs.{inr(ca["totals"]["IGST"])}) against output tax declared '
                   f'of Rs.{inr(ca["output_tax"])}'
                   + (f' — {ca["cash_pct"]*100:.1f}%.' if ca['cash_pct'] is not None else '.')
                   + f' ITC availed over the same period is Rs.{inr(ca["itc"])}'
                   + (f', an availment ratio of {ca["itc"]/ca["output_tax"]*100:.1f}%.' if ca['output_tax'] else '.')),
                a=0.0, s='19_CASH_RECON; 11_FY_SUMMARY', p='Rule 86B. Section 49.', r='HIGH — context'))
        rb = R['rule86b']
        if rb.get('available') and rb['flagged']:
            F.append(dict(
                t='Rule 86B — output tax discharged almost wholly from credit',
                w=('Periods below the 1% cash floor on taxable supply above Rs.50 lakh: '
                   + ', '.join(f'{x["period"]} ({x["pct"]*100:.2f}% on taxable supply of Rs.{inr(x["taxable"])})'
                               for x in rb['rows'] if x['flagged']) + '.'),
                a=0.0, s='15_RULE_86B; 19_CASH_RECON',
                p='Rule 86B, subject to the exceptions in the proviso.',
                r='MEDIUM — prima facie, cash ledger to be verified'))
        fl = R['filing']
        if fl.get('available') and fl['late']:
            F.append(dict(
                t=f'GSTR-3B filed late in {len(fl["late"])} tax periods',
                w=('Total delay ' + str(fl['total_delay']) + ' days: '
                   + ', '.join(f'{p} ({d} days)' for p, _, d in fl['late']) + '.'),
                a=0.0, s='18_RETURN_FILING', p='Section 47(1) and 47(2). Section 50 where tax was paid late.', r='LOW'))
        an = R['annual']
        need = [x for x in an['rows'] if x['gstr9_required'] and not x['on_record']]
        if need:
            F.append(dict(
                t='Annual return and reconciliation statement not on record',
                w=('; '.join(f'FY {x["fy"]} turnover Rs.{inr(x["turnover"])} — GSTR-9 required'
                             + (' and GSTR-9C required' if x['gstr9c_required'] else '') + ', not evidenced'
                             for x in need) + '.'),
                a=0.0, s='SRC_GSTR9; 11_FY_SUMMARY', p='Section 44 read with Rule 80. Section 47(2).',
                r='MEDIUM — compliance'))
        hs = R['hsn']
        if hs.get('available') and len(hs['codes']) >= 3:
            F.append(dict(
                t='Line of business as declared through HSN codes',
                w=('HSN reporting moves ' + ' -> '.join(hs['codes']) + ' over the period. A drift of this kind, and in '
                   'particular goods HSNs appearing in a services taxpayer, points to disposal of capital goods.'),
                a=0.0, s='12_HSN_PROFILE',
                p='Section 18(6) with Rule 44(6). Section 25 and Rule 19 for amendment of registration particulars.',
                r='MEDIUM'))
        F += self._tds_findings()
        F += self._tds_detail_findings()
        rev = [c for c in self.ing.changes if c['level'] == 'REVIEW']
        if rev:
            reps = sorted({c['title'] for c in rev})
            recs = sorted({(c['code'], c['record']) for c in rev})
            F.append(dict(
                t='A later export changed values already held (possible restatement)',
                w=(f'{len(rev)} value(s) across {len(recs)} record(s) in {len(reps)} report(s) differ between an earlier and a '
                   f'later Prime export: ' + '; '.join(f'{c["record"]} {c["field"]} {c["old"]} -> {c["new"]}' for c in rev[:6])
                   + (f'; and {len(rev)-6} more' if len(rev) > 6 else '') + '. The later value is the one used. '
                   + (f'{sum(1 for c in rev if c.get("order_uncertain"))} of these come from files uploaded together, so which is '
                      f'newer cannot be told. ' if any(c.get('order_uncertain') for c in rev) else '')
                   + 'GSTR-3B and payment challans cannot be revised, so a change in either is unusual. It may mean the '
                   'taxpayer amended something, that Prime refreshed its data, or that the earlier export was taken before '
                   'the period was final.'),
                a=0.0, s='09A_RESTATEMENTS',
                p='Verify on the portal before either figure is quoted in a notice. Section 70 / 71 to call for the returns and records.',
                r='MEDIUM — verify'))
        return F

    def _tds_findings(self):
        t = self.R.get('tds') or {}
        if not t.get('available'): return []
        F = []; fys = sorted(t['fy'], key=lambda f: int(f.split('-')[0]))
        rate = 0.18
        F.append(dict(
            t=f'Tax deducted under section 51 on supplies by this taxpayer of at least Rs.{inr(t["total_value"])} ({t["first"]} to {t["last"]})',
            w=(f'GSTR-7 shows tax deducted at source of Rs.{inr(t["total_tds"])} by up to {int(t["deductor_max"])} '
               f'deductors a month. At 2% that is payment for taxable supplies of: '
               + '; '.join(f'FY {f} Rs.{inr(t["fy"][f]["value"])} over {t["fy"][f]["months"]} month(s) '
                           f'({t["fy"][f]["first"]} to {t["fy"][f]["last"]})' for f in fys)
               + '. ' + ('Only CGST and SGST were deducted, so every deductor is in the same State. ' if not t['interstate'] else '')
               + 'This is a floor on turnover: supplies to buyers who do not deduct, and contracts of Rs.2.5 lakh or less, '
                 'carry no TDS.'),
            a=0.0, s='19C_TDS_GSTR7; SRC_R7_TDS',
            p='Section 51 with Notification 50/2018-CT; Rule 66 (GSTR-7); Rule 87(9) (credit to the cash ledger).',
            r='BASELINE — the floor every return is tested against'))
        if not t['have_3b'] or not t['have_r1']:
            miss = ([] if t['have_3b'] else ['GSTR-3B (TAX PAID DETAILS, R3B TAX PAYABLE DETAILS, TURNOVER OUTWARD TAX SUPPLIES, '
                                              'R3B PAYMENT SUMMARY)'])
            miss += [] if t['have_r1'] else ['GSTR-1 (R1 FILINGS IN THE YEAR, HSN LIST)']
            F.append(dict(
                t='The GSTR-7 figures could not yet be tested against the taxpayer\'s own returns',
                w=('The following are not in the case file, so the tests that compare what the deductors paid for with what '
                   'the taxpayer declared were NOT RUN: ' + '; '.join(miss) + '. Upload them under the same GSTIN and run '
                   'again. Sheet 19C lists each test, its provision and the report it needs.'),
                a=0.0, s='19C_TDS_GSTR7; 23_DATA_GAPS', p='—', r='PENDING DATA'))
        if t['no_3b']:
            v = sum(r['value'] for r in t['rows'] if r['period'] in t['no_3b'])
            F.append(dict(
                t=f'No GSTR-3B for {len(t["no_3b"])} month(s) in which the deductors paid for supplies',
                w=(f'TDS was deducted in {", ".join(t["no_3b"])}, proving payment for supplies of Rs.{inr(v)}, but no GSTR-3B '
                   f'is on record for those months.'),
                a=0.0, s='19C_TDS_GSTR7',
                p='Section 39. Section 46 notice in FORM GSTR-3A; section 62 best-judgment assessment on the GSTR-7 value. '
                  'Section 47 late fee, section 50 interest.',
                r='VERY HIGH — non-filer'))
        if t['nil_3b']:
            v = sum(r['value'] for r in t['rows'] if r['period'] in t['nil_3b'])
            F.append(dict(
                t=f'GSTR-3B filed NIL in {len(t["nil_3b"])} month(s) in which the deductors paid for supplies',
                w=(f'{", ".join(t["nil_3b"])}: GSTR-3B declares no outward supply while the deductors paid for supplies of '
                   f'Rs.{inr(v)} and deducted tax on them.'),
                a=0.0, s='19C_TDS_GSTR7; 10_MONTHLY_MASTER',
                p='Sections 39, 73 / 74 / 74A (FY 2024-25 onwards), 50, 122.', r='VERY HIGH'))
        sh = [(f, t['fy'][f]) for f in fys if (t['fy'][f]['short_3b'] or 0) > 1 or (t['fy'][f]['max_cum_short'] or 0) > 1]
        if sh:
            F.append(dict(
                t='Turnover declared in GSTR-3B is below what the deductors alone paid for',
                w=('; '.join(f'FY {f}: the deductors paid for Rs.{inr(a["value"])}, GSTR-3B declares taxable turnover of '
                             f'Rs.{inr(a["r3b_taxable"])} over {a["r3b_months"]} month(s) on record — shortfall Rs.{inr(a["short_3b"])}'
                             + (f', peak cumulative shortfall within the year Rs.{inr(a["max_cum_short"])}'
                                if (a['max_cum_short'] or 0) > (a['short_3b'] or 0) + 1 else '')
                             + f' (tax at 18% would be about Rs.{inr((a["short_3b"] or 0) * rate)} — indicative only, rate to be '
                               f'confirmed from the contracts)' for f, a in sh)
                   + '. Not counted in the total below, because the rate is assumed.'),
                a=0.0, s='19C_TDS_GSTR7 (FY position)',
                p='Sections 12, 13(2), 31 and 39 (time and declaration of supply); 74A for FY 2024-25 onwards; section 50 interest.',
                r='VERY HIGH — suppression on supplies to the deductors'))
        if t['cgst_over']:
            F.append(dict(
                t=f'CGST deducted at 1% exceeds the whole CGST declared, in {len(t["cgst_over"])} month(s)',
                w=f'{", ".join(t["cgst_over"])}. At any rate of 2% or more the tax declared cannot be below the tax deducted at 1%.',
                a=0.0, s='19C_TDS_GSTR7', p='Sections 39 and 51.', r='HIGH'))
        if t['no_r1']:
            F.append(dict(
                t=f'No GSTR-1 for {len(t["no_r1"])} month(s) with TDS payments',
                w=(f'{", ".join(t["no_r1"])}: the B2B invoices to the deductor GSTINs are not on record, although the deductors '
                   f'paid and deducted tax.'),
                a=0.0, s='19C_TDS_GSTR7', p='Section 37, Rule 59; Rule 59(6); section 47.', r='HIGH'))
        r1s = [(f, t['fy'][f]) for f in fys if (t['fy'][f]['short_r1'] or 0) > 1]
        if r1s:
            F.append(dict(
                t='GSTR-1 taxable value below what the deductors paid for',
                w='; '.join(f'FY {f}: Rs.{inr(a["value"])} paid for, GSTR-1 Rs.{inr(a["r1_value"])}, short Rs.{inr(a["short_r1"])}'
                            for f, a in r1s) + '.',
                a=0.0, s='19C_TDS_GSTR7 (FY position)', p='Section 37, Rule 59.', r='HIGH'))
        un = [(f, t['fy'][f]['tds_sgst_unabsorbed']) for f in fys if (t['fy'][f]['tds_sgst_unabsorbed'] or 0) > 1]
        if un:
            F.append(dict(
                t='TDS credit not used to pay tax — a cash-ledger balance building up',
                w=('; '.join(f'FY {f}: SGST deducted exceeds SGST paid in cash by Rs.{inr(v)}' for f, v in un)
                   + '. Output tax is being discharged from ITC while the TDS sits in the cash ledger, where it can be claimed '
                     'as refund.'),
                a=0.0, s='19C_TDS_GSTR7 (FY position)',
                p='Rule 87(9), section 49, section 54 and Rule 89 (refund of excess cash-ledger balance). Rule 86B.',
                r='MEDIUM — refund watch'))
        comp = []
        for f in fys:
            a = t['fy'][f]
            if a['gstr9c']: comp.append(f'FY {f}: Rs.{inr(a["value"])} from the deductors alone — GSTR-9 and self-certified GSTR-9C '
                                        f'required (s.44, Rule 80), e-invoicing from the next FY (Rule 48(4)), QRMP not available')
            elif a['gstr9']: comp.append(f'FY {f}: Rs.{inr(a["value"])} from the deductors alone ({a["first"]} to {a["last"]}) — GSTR-9 '
                                         f'required' + ('; within reach of Rs.5 crore, when GSTR-9C, e-invoicing from the next '
                                                        'FY and the end of QRMP follow' if a['value'] > 0.75 * 50_000_000 else ''))
        over = [p for f in fys for p in t['fy'][f]['over50']]
        if over: comp.append('Rule 86B threshold of Rs.50 lakh crossed on receipts subject to TDS alone in ' + ', '.join(over)
                             + ' — TDS actually used counts as cash, so test on the cash ledger before invoking it')
        if comp:
            F.append(dict(t='Compliance thresholds reached on receipts subject to TDS alone', w='; '.join(comp) + '.',
                          a=0.0, s='19C_TDS_GSTR7', p='Section 44, Rule 80, Rule 48(4), Rule 61A, Rule 86B.',
                          r='MEDIUM — compliance'))
        return F

    def _tds_detail_findings(self):
        d = self.R.get('tds_detail') or {}
        if not d.get('available'): return []
        F = []
        if d['subject_mismatch']:
            F.append(dict(t='GSTR-7 detail export titled for a different GSTIN',
                          w=f'The export names {", ".join(d["subject_mismatch"])}, not the case GSTIN. It may belong to another case.',
                          a=0.0, s='19D_TDS_PARTIES; 09_FILE_REGISTER', p='—', r='CHECK BEFORE RELYING ON IT'))
        if d['ineligible']:
            b = d['ineligible']
            F.append(dict(
                t=f'TDS deducted by {len(b)} part{"y" if len(b)==1 else "ies"} that section 51 does not allow to deduct',
                w=('In the months held (' + ', '.join(m['period'] for m in d['months']) + '), GSTR-7 records tax deducted of '
                   f'Rs.{inr(d["ineligible_tds"])} on supplies of Rs.{inr(d["ineligible_value"])} by: '
                   + '; '.join(f'{", ".join(p["names"])} ({p["gstin"]}) — {p["lines"]} line(s), Rs.{inr(p["value"])}'
                               for p in b)
                   + '. Every one holds a tax-deductor registration (D in position 14) on the PAN of an individual / '
                     'proprietor (P in position 4 of the PAN). Trade names indicate private steel and metal traders. None is a '
                     'Government department, local authority, Governmental agency, Government-controlled body or PSU, so none '
                     'is a deductor under section 51(1) or Notification 50/2018-CT, and the REG-07 registrations are themselves '
                     'irregular. The deductions nevertheless put cash-ledger credit in this taxpayer’s name. '
                   + (f'{", ".join(b[0]["names"])} alone accounts for {d["top_share"]*100:.0f}% of the value. ' if d['top_share'] else '')
                   + (f'{len(d["near_threshold"])} line(s) sit just above the Rs.2.5 lakh deduction threshold. ' if d['near_threshold'] else '')
                   + 'The reason for the arrangement is not on record and must be established.'),
                a=0.0, s='19D_TDS_PARTIES; SRC_R7_DETAIL',
                p=('Section 51(1) with Notification 50/2018-CT (who may deduct); section 24(vi), section 25 and Rule 12 '
                   '(deductor registration) — cancellation under section 29 / Rule 12(3); Rule 87(9) and section 49 (credit '
                   'to the cash ledger); section 54 and Rule 89 (refund of a cash-ledger balance); section 70 (summons).'),
                r='VERY HIGH — lead'))
        bad_m = [m for m in d['months'] if m['agrees'] is False]
        if bad_m:
            F.append(dict(t='GSTR-7 detail does not agree with the summary report',
                          w='; '.join(f'{m["period"]}: detail Rs.{inr(m["tds"])} / {m["parties"]} parties, summary '
                                      f'Rs.{inr(n0(m["s_tds"]))} / {int(n0(m["s_parties"]))}' for m in bad_m) + '.',
                          a=0.0, s='19D_TDS_PARTIES', p='—', r='CHECK THE EXPORTS'))
        return F

    def build_findings(self):
        F = self.findings(); ws = self.sheet('20_FINDINGS')
        ws['A1'] = f'CONSOLIDATED FINDINGS — {self.name}'; ws['A1'].font = BOLD
        ws['A2'] = f'GSTIN {self.gstin}. Ranked by revenue implication. Every figure carries the sheet it is computed on.'
        self.head(ws, 4, ['No.','Finding','What the returns show','Amount involved (Rs.)','Computed on sheet',
                          'Provision attracted','Risk'], [5,46,100,20,28,54,24])
        F = sorted(F, key=_frank)
        for i, f in enumerate(F, 5):
            ws.cell(row=i, column=1, value=i-4); ws.cell(row=i, column=2, value=f['t'])
            ws.cell(row=i, column=3, value=f['w']); ws.cell(row=i, column=4, value=round(f['a']))
            ws.cell(row=i, column=5, value=f['s']); ws.cell(row=i, column=6, value=f['p'])
            ws.cell(row=i, column=7, value=f['r'])
        r = 4 + len(F) + 1
        ws.cell(row=r, column=2, value='TOTAL REVENUE IMPLICATION (excluding interest under section 50 and penalty)').font = BOLD
        ws.cell(row=r, column=4, value=f'=SUM(D5:D{r-1})').font = BOLD
        for rr in range(5, r+1):
            for c in range(1, 8):
                x = ws.cell(row=rr, column=c); x.border = BD
                x.alignment = Alignment(wrap_text=True, vertical='top')
                if c == 4: x.number_format = NUM
        self.total_findings = sum(f['a'] for f in F)
        n = self.note(ws, r+2, f'Rs. {inr(self.total_findings)} before interest and penalty.', bold=True)
        n = self.note(ws, n, 'OVERLAP CAUTION: where the same credit is caught by more than one finding — for example '
                             'credit availed in a NIL period that is also in excess of GSTR-2A — the amount must not be '
                             'demanded twice. Reconcile before the notice is drafted.')
        self.note(ws, n, 'This is a preliminary investigation on returns data. It is not a finding of fact. Every figure is '
                         'to be re-verified against the common portal, the electronic ledgers, the returns themselves and '
                         'the books of account before any notice is issued.')
        return F

    # ------------------------------------- 21 REPORT / 22 ACTIONS / 23 GAPS
    def build_report(self, F):
        R = self.R; ws = self.sheet('21_REPORT')
        fy = R['fy']; fys = sorted(fy, key=lambda x: int(x.split('-')[0]))
        tot_to = sum(v['turnover'] for v in fy.values())
        tot_op = sum(v['op'] for v in fy.values())
        tot_itc = sum(v['itc'] for v in fy.values())
        cash = R['cash']['totals']['TOTAL'] if R['cash'].get('available') else None
        r1 = R['r1_vs_r3b']; fl = R['filing']
        peak = max(fys, key=lambda f: fy[f]['turnover']) if fys else None
        ap = self.ing.all_periods()
        span = (f'{len(R["master"])} tax periods, {R["master"][0]["period"]} to {R["master"][-1]["period"]}' if R['master']
                else f'{ap[0]} to {ap[-1]}' if ap else 'no tax period')
        T = [f'PRELIMINARY INVESTIGATION REPORT — ANALYSIS OF GST RETURNS DATA',
             f'{self.name.upper()}', f'GSTIN {self.gstin}', '',
             '1. SUBJECT', '',
             (f'Analysis of {len(self.ing.counted())} unique GST-Prime Analytica statements covering '
              f'{span}, to identify '
              f'indicators of evasion and to quantify the revenue at stake. '
              + (f'{len(self.ing.register)} files were received; {len(self.ing.duplicates())} were exact duplicates of '
                 f'files already held and have been excluded from every count and computation. '
                 if self.ing.duplicates() else '')
              + 'The register is at sheet 09_FILE_REGISTER.'), '',
             '2. WHAT THE RETURNS SHOW IN OUTLINE', '']
        if not fys:
            T += [('2.1  No GSTR-3B report has been supplied, so the taxpayer\'s own declarations are not yet before this '
                   'analysis. What follows rests on the third-party statements held.'), '']
        else:
          T += [(f'2.1  Across the whole period the unit declared turnover of Rs.{inr(tot_to)}, output tax of '
               f'Rs.{inr(tot_op)} and availed input tax credit of Rs.{inr(tot_itc)}'
               + (f' — an availment ratio of {tot_itc/tot_op*100:.1f} per cent' if tot_op else '')
               + (f'. Against that it has paid Rs.{inr(cash)} in cash, in {R["cash"]["count"]} challans.' if cash is not None else '.')), '']
          T += [(f'2.2  The peak year is FY {peak}, with turnover of Rs.{inr(fy[peak]["turnover"])} and output tax of '
               f'Rs.{inr(fy[peak]["op"])} over {fy[peak]["periods"]} tax periods. '
               + '; '.join(f'FY {f} Rs.{inr(fy[f]["turnover"])}' for f in fys) + '.'), '']
        if R['dormancy'].get('available') and R['dormancy']['longest']:
            a, b, c = R['dormancy']['longest']
            T += [(f'2.3  The longest stretch of NIL outward supply runs {c} consecutive tax periods, {a} to {b}, during '
                   f'which credit of Rs.{inr(R["dormancy"]["itc_in_nil_from_longest"])} was nonetheless availed.'), '']
        if R.get('unfiled'):
            T += [(f'2.4  GSTR-3B is shown as NOT FILED for {", ".join(R["unfiled"])}. That is a period with no return of '
                   f'any kind, and it has been kept out of the tax-period count and out of the NIL-turnover and dormancy '
                   f'tests, so that it is not silently read as a period of nil supply.'), '']
        td = R.get('tds') or {}
        if td.get('available'):
            T += [(f'2.5  GSTR-7 shows that the deductors deducted tax at source of Rs.{inr(td["total_tds"])} from this '
                   f'taxpayer between {td["first"]} and {td["last"]}, which at 2% proves payment for taxable supplies of at least '
                   f'Rs.{inr(td["total_value"])} ('
                   + '; '.join(f'FY {f} Rs.{inr(td["fy"][f]["value"])}' for f in sorted(td['fy'])) + '). '
                   + ('These figures have been tested against GSTR-3B and GSTR-1 at sheet 19C_TDS_GSTR7.'
                      if td['have_3b'] and td['have_r1'] else
                      'GSTR-3B and/or GSTR-1 have not yet been supplied, so the comparison at sheet 19C_TDS_GSTR7 is pending; '
                      'the tests it will run, and the provision behind each, are listed there.')), '']
        dd = R.get('tds_detail') or {}
        if dd.get('available'):
            T += [('2.6  The party-wise GSTR-7 detail is held for ' + ', '.join(m['period'] for m in dd['months'])
                   + f' ({len(dd["parties"])} parties, Rs.{inr(dd["total_value"])} of supplies, TDS Rs.{inr(dd["total_tds"])}). '
                   + dd['role'] + ' '
                   + (f'{len(dd["ineligible"])} of the {len(dd["parties"])} parties are, on the face of their GSTIN and PAN, '
                      f'private proprietors holding tax-deductor registrations — not persons who may deduct under section 51. '
                      if dd['ineligible'] else '')
                   + (f'Detail for {", ".join(dd["missing_months"])} is still to be uploaded.' if dd['missing_months'] else '')), '']
        T += ['3. PRINCIPAL FINDINGS', '']
        for i, f in enumerate(sorted(F, key=_frank), 1):
            T += [f'3.{i}  {f["t"]}.  {f["w"]}'
                  + (f'  This accounts for Rs.{inr(f["a"])} of the revenue at stake.' if f['a'] else ''), '']
        T += ['4. THE REVENUE AT STAKE', '',
              (f'4.1  On the face of the returns the revenue implication is Rs.{inr(self.total_findings)}, before interest '
               f'under section 50 and before penalty. The composition is set out in sheet 20_FINDINGS.'), '',
              '5. LIMITATION', '',
              ('5.1  Sheet 24_LIMITATION sets out the indicative outer dates by financial year, computed from the statute '
               'alone. Extension notifications are not applied there and several have been the subject of litigation, so '
               'every date must be verified against the notifications in force.'), '',
              ('5.2  Where section 73 has run out for a year, a demand for that year can be raised only under section 74, '
               'which requires fraud, wilful misstatement or suppression of facts to be alleged and made out on evidence. '
               'That decision has to be taken before the evidence is gathered, not after.'), '',
              '6. CAVEATS', '',
              ('6.1  This report is built only from the GST-Prime Analytica statements listed at sheet 09_FILE_REGISTER. '
               'It is a preliminary investigation on returns data and is not a finding of fact.'), '',
              ('6.2  Nothing has been estimated, interpolated or assumed. Where a report was not supplied, the test that '
               'needed it says so and was not run. Where a cell was blank in the source it has been kept blank, because a '
               'blank is not a nil.'), '',
              ('6.3  Wherever CGST has been taken as equal to SGST, the assumption is recorded on the sheet concerned and '
               'must be confirmed from the electronic credit ledger.'), '',
              *((('6.4  A later Prime export changed values that an earlier export had already supplied '
                  f'({sum(1 for c in self.ing.changes if c["level"]=="REVIEW")} value(s) needing review). The later value is used '
                  'throughout. Sheet 09A_RESTATEMENTS lists each difference with both values and the files they came from. '
                  'Verify on the portal before relying on either.'), '')
                if any(c['level'] == 'REVIEW' for c in self.ing.changes) else ()),
              ('6.5  Every figure is to be re-verified against the common portal, the electronic credit and cash ledgers, '
               'GSTR-1, GSTR-2A and 2B, GSTR-9 and 9C and the books of account before any notice is issued.')]
        for i, t in enumerate(T, 1):
            c = ws.cell(row=i, column=1, value=t); c.alignment = Alignment(wrap_text=True, vertical='top')
            if t and (t.isupper() or (t[0].isdigit() and '. ' in t[:5]) or t.startswith('GSTIN')): c.font = BOLD
        ws.column_dimensions['A'].width = 145

    def build_actions(self, F):
        ws = self.sheet('22_ACTION_POINTS')
        ws['A1'] = 'PROPOSED COURSE OF ACTION'; ws['A1'].font = BOLD
        A = []
        R = self.R
        if R['credit_notes'].get('available') and R['credit_notes']['events']:
            e = R['credit_notes']['events'][0]
            A.append(['IMMEDIATE', f'Call for the tax invoice, the credit note, the contract and the transport documents '
                                   f'behind the {e["period"]} entry of Rs.{inr(abs(e["turnover"]))}'
                                   + (f' under HSN {", ".join(e["hsn"])}' if e['hsn'] else '') + '.',
                      'Usually the largest single item and the freshest, so the evidence is most likely to still exist and '
                      'the recipient can still be cross-verified.', 'Section 34, section 70'])
            A.append(['IMMEDIATE', 'Examine whether ITC in the electronic credit ledger should be blocked pending verification.',
                      'A credit note that drives output tax negative can create or inflate a credit balance which may be '
                      'utilised or claimed as refund while the enquiry runs.', 'Rule 86A'])
        if R['credit_notes'].get('available') and R['credit_notes']['r1_only']:
            ps = ', '.join(x['period'] for x in R['credit_notes']['r1_only'])
            A.append(['IMMEDIATE', f'Draw the full GSTR-1 for {ps} from the portal, table by table (B2B, B2CS, CDNR, Table 12).',
                      'These returns carry negative tax that does not reconcile with their own taxable value column. They '
                      'cannot be read from a summary.', 'Section 37, section 70, section 151'])
            A.append(['IMMEDIATE', 'Establish whether any ASMT-10, DRC-01A, DRC-03, audit or intelligence proceeding was in '
                                   'progress against this taxpayer in those periods, and obtain the record.',
                      'Large credit reversals and credit notes in one window usually follow departmental contact. If a '
                      'proceeding exists it changes both the evidence available and the limitation position.',
                      'Section 6(2)(b) — no parallel proceeding on the same subject matter'])
        r1 = R['r1_vs_r3b']
        if r1.get('available') and r1['not_filed'] and r1['not_filed_turnover'] > 0:
            A.append(['HIGH', f'Obtain the invoice-level outward record for {", ".join(r1["not_filed"])} from the taxpayer, '
                              f'no GSTR-1 existing on the portal for those periods.',
                      f'Rs.{inr(r1["not_filed_turnover"])} of turnover sits in GSTR-3B with no invoice-level record. '
                      'Identify the recipients and confirm the supplies took place.', 'Section 37, section 70, section 71'])
        if r1.get('available') and r1['short_total'] > 0:
            A.append(['HIGH', f'Issue the Rule 88C intimation in FORM DRC-01B for the GSTR-1 versus GSTR-3B difference of '
                              f'Rs.{inr(r1["short_total"])}.',
                      'Self-assessed tax under the Explanation to section 75(12), recoverable under section 79 without a '
                      'show cause notice and needing no investigation to establish. The fastest realisation in the case.',
                      'Rule 88C, FORM DRC-01B, section 75(12), section 79'])
        if R['excess_itc'].get('available') and R['excess_itc']['net'] > 0:
            A.append(['HIGH', 'Issue the Rule 88D intimation in FORM DRC-01C and obtain the supplier-wise break-up for the '
                              'years carrying the excess.',
                      'The supplier trail has to be established on documents — whether the suppliers are traceable, whether '
                      'they filed, and whether their registrations have since been cancelled.',
                      'Rule 88D, FORM DRC-01C, section 16(2)(aa), Rule 36(4)'])
        if R['dormancy'].get('available') and R['dormancy']['itc_in_nil_from_longest'] > 0:
            A.append(['HIGH', f'Call for the invoices behind the ITC of Rs.{inr(R["dormancy"]["itc_in_nil_from_longest"])} '
                              f'availed in NIL-turnover periods.',
                      'Credit taken against no outward supply must be shown to relate to inputs used or intended to be used '
                      'in business.', 'Section 16(1), section 17(5), section 70'])
        A += self._tds_actions()
        A += [['HIGH', 'Draw the electronic credit and cash ledgers from registration to date, and GSTR-2B for every period.',
               'GSTR-2B and not 2A governs entitlement after 1 January 2022, and only the ledgers settle the CGST '
               'assumptions and confirm the true cash position.', 'Section 151, Rule 36(4), Rule 60'],
              ['HIGH', 'Conduct a physical verification of the declared place of business.',
               'Where a registration has been kept alive through long dormancy, the existence and functioning of the unit '
               'is a question of fact.', 'Section 67, section 71, Rule 25']]
        if R['rcm'].get('available') and R['rcm']['silent_periods'] >= 12:
            A.append(['MEDIUM', 'Call for the goods transport, legal, security and rent expenditure from the books for the '
                                'periods with no reverse-charge declaration, and quantify the liability.',
                      'A long stretch of nil RCM while credit was being availed is not consistent with a working unit.',
                      'Sections 9(3), 9(4), Notification 13/2017-CT(R), section 31(3)(f)'])
        if R['hsn'].get('available'):
            A.append(['MEDIUM', 'Verify the section 18(6) liability on any supply of capital goods, including plant and '
                                'machinery on which credit was taken.',
                      'Tax is the higher of the credit taken less the prescribed reduction, or the tax on transaction value.',
                      'Section 18(6), Rule 44(6)'])
        need = [x for x in R['annual']['rows'] if x['gstr9_required'] and not x['on_record']]
        if need:
            A.append(['MEDIUM', 'Verify the filing of GSTR-9 and GSTR-9C for '
                                + ', '.join(f'FY {x["fy"]}' for x in need) + ', and initiate late fee where not filed.',
                      'GSTR-9C is the reconciliation statement that ties the returns to the audited accounts for the years '
                      'carrying the exposure.', 'Section 44, Rule 80, section 47(2)'])
        if R['rule86b'].get('available') and R['rule86b']['flagged']:
            A.append(['MEDIUM', 'Re-run the Rule 86B test on the electronic cash ledger for '
                                + ', '.join(R['rule86b']['flagged']) + ', and rule out the exceptions in the proviso.',
                      'The present result rests on an SGST-only proxy. The exceptions must be eliminated before the rule '
                      'can be invoked.', 'Rule 86B'])
        A += [['MEDIUM', 'Obtain the audited financial statements, income tax returns, Form 26AS, bank statements and the '
                         'e-way bill data for the whole period.',
               'Turnover per the books against turnover per the returns is the cleanest independent test. E-way bills are '
               'the test of whether the goods moved.', 'Section 71, section 151, Rule 138'],
              ['AS ADVISED', 'Fix the limitation position for each financial year against the notifications in force, and '
                             'settle the section under which each year is to be taken up.',
               'Where section 73 has run out, section 74 requires suppression to be alleged and proved, so the route must '
               'be decided before the evidence is gathered.', 'Sections 73, 74, 74A']]
        PR = {'IMMEDIATE': 0, 'HIGH': 1, 'MEDIUM': 2}
        A = sorted(A, key=lambda a: PR.get(a[0], 3))          # stable: order within a priority is kept
        self.head(ws, 3, ['No.','Priority','Action','Why it comes in this order','Provision / form'], [5,12,78,78,46])
        for i, a in enumerate(A, 4):
            ws.cell(row=i, column=1, value=i-3)
            for j, v in enumerate(a, 2): ws.cell(row=i, column=j, value=v)
        for rr in range(4, 4+len(A)):
            for c in range(1, 6):
                x = ws.cell(row=rr, column=c); x.border = BD
                x.alignment = Alignment(wrap_text=True, vertical='top')

    def _tds_actions(self):
        t = self.R.get('tds') or {}
        if not t.get('available'): return []
        A = []
        if not t['have_3b'] or not t['have_r1']:
            A.append(['IMMEDIATE', 'Export from GST-Prime, for the same GSTIN, the GSTR-3B reports (TAX PAID DETAILS, R3B TAX PAYABLE '
                                   'DETAILS, TURNOVER OUTWARD TAX SUPPLIES, R3B PAYMENT SUMMARY), GSTR-1 (R1 FILINGS, HSN LIST), '
                                   'GSTR-2A (R2A FILINGS, EXCESS ITC CLAIMED) and PAYMENT DETAILS, upload them and run again.',
                      'Every test of the GSTR-7 figures against the taxpayer’s own returns needs them. Until then only the '
                      'value the deductors paid for is established.', 'Section 151, section 70'])
        d = self.R.get('tds_detail') or {}
        if d.get('available') and d['ineligible']:
            names = '; '.join(f'{", ".join(p["names"])} ({p["gstin"]})' for p in d['ineligible'])
            A.append(['IMMEDIATE', f'Confirm on the portal the registration type, constitution, status and jurisdiction of: {names}. '
                                   'Find the ordinary (Z) registration held under the same PAN.',
                      'One search per GSTIN settles that these are tax-deductor registrations held by private proprietors — the '
                      'basis of the finding.', 'Section 25, Rule 12'])
            A.append(['IMMEDIATE', 'Refer the irregular tax-deductor registrations to the officers having jurisdiction over them, '
                                   'for cancellation and for examination of every GSTR-7 they have filed and every deductee they '
                                   'have credited.',
                      'The arrangement is unlikely to be confined to this taxpayer.', 'Section 29(2), Rule 12(3), section 51'])
            A.append(['IMMEDIATE', 'Check whether this taxpayer accepted the TDS credit, how it was used, and whether any refund of '
                                   'the cash-ledger balance has been claimed or sanctioned; hold any pending refund for verification.',
                      'Credit created by deductions that section 51 does not permit is the money at risk.',
                      'Rule 87(9), section 49, section 54, Rule 89'])
            A.append(['HIGH', 'Record statements of the proprietor of this taxpayer and of each listed party: the goods or services '
                              'supplied, the invoices, the payments through bank, and why tax was deducted.',
                      'Nothing in the returns explains why private traders would deduct and deposit 2% in cash.', 'Section 70'])
        if d.get('available') and d['missing_months']:
            A.append(['HIGH', 'Export the "R7 Filing Details For The Month" report for ' + ', '.join(d['missing_months'])
                              + ' and upload it.', 'The party-wise test has so far run on '
                              + str(len(d['months'])) + ' of ' + str(len(d['months']) + len(d['missing_months'])) + ' months.',
                      'Section 151'])
        A.append(['IMMEDIATE', 'Draw the deductor-wise GSTR-7 detail and the GSTR-7A certificates: deductor GSTIN, contract or '
                               'work-order reference, amount paid and date, TDS deducted, month by month.',
                  'The statement used here gives only counts and totals. The deductor-wise detail names each deductor, so the '
                  'supplies can be matched invoice by invoice to GSTR-1.', 'Section 51, Rule 66, Rule 87(9)'])
        if not (d.get('available') and d['parties'] and len(d['ineligible']) == len(d['parties'])):
          A.append(['HIGH', 'Write to the drawing and disbursing officer of each deductor for the work orders, running and final '
                          'bills, measurement books and payment vouchers for the period.',
                  'Independent third-party evidence of the gross contract value, the bill dates (time of supply) and the GST '
                  'charged on each bill — the rate question is settled here.', 'Section 70, section 151'])
        if t['no_3b']:
            A.append(['IMMEDIATE', f'Issue the section 46 notice in FORM GSTR-3A for {", ".join(t["no_3b"])}; if the returns are '
                                   f'not filed within 15 days, assess under section 62 on the GSTR-7 value.',
                      'TDS payments prove supply in months with no return.', 'Sections 46, 62, 47, 50'])
        if any((t['fy'][f]['short_3b'] or 0) > 1 or (t['fy'][f]['max_cum_short'] or 0) > 1 for f in t['fy']) or t['nil_3b']:
            A.append(['HIGH', 'Quantify the shortfall bill by bill from the deductor records, fix the rate from the contract, and '
                              'proceed by DRC-01A intimation and show cause notice.',
                      'The GSTR-7 value is a floor; the bills give the true value and date of each supply.',
                      'Rule 142(1A), sections 73 / 74 / 74A, section 50'])
        A.append(['HIGH', 'Watch for, and verify before sanctioning, any refund claim of the electronic cash-ledger balance.',
                  'TDS credit lands in the cash ledger. If output tax is paid from ITC, the TDS accumulates and can be claimed '
                  'back as refund — before the shortfall is established.', 'Section 54, Rule 89, Rule 87(9)'])
        A.append(['MEDIUM', 'Obtain Form 26AS for the same years and compare the section 194C income-tax TDS by the same '
                            'deductors.',
                  'The same payments are reported to the Income Tax Department. An independent check on the gross amounts paid.',
                  'Section 151'])
        if any(t['fy'][f]['gstr9'] for f in t['fy']):
            A.append(['MEDIUM', 'Calendar the GSTR-9 / GSTR-9C and e-invoicing position for '
                                + ', '.join(f'FY {f}' for f in sorted(t['fy']) if t['fy'][f]['gstr9'])
                                + ', and check whether the taxpayer is on QRMP.',
                      'The thresholds are reached on receipts subject to TDS alone.', 'Section 44, Rule 80, Rule 48(4), Rule 61A'])
        return A

    def build_gaps(self):
        ws = self.sheet('23_DATA_GAPS')
        ws['A1'] = 'DATA POSITION — what is held, and what is still required'; ws['A1'].font = BOLD
        self.head(ws, 3, ['Status','Report','What it drives','Position'], [22, 48, 62, 44])
        r = 4
        for code, sp in REPORT_SPECS.items():
            have = self.ing.have(code)
            ws.cell(row=r, column=1, value='HELD' if have else 'NOT SUPPLIED')
            if not have and self.mark: ws.cell(row=r, column=1).fill = MISSING_FILL
            ws.cell(row=r, column=2, value=f'{sp["title"]}  [{code}]')
            ws.cell(row=r, column=3, value=DRIVES.get(code, ''))
            ws.cell(row=r, column=4, value=(f'{len(self.ing.data[code])} rows from '
                                            f'{len({x["_file"] for x in self.ing.data[code]})} file(s)')
                    if have else 'The tests that need it were not run.')
            r += 1
        for x in EXTERNAL:
            ws.cell(row=r, column=1, value='NOT IN GST-PRIME')
            if self.mark: ws.cell(row=r, column=1).fill = MISSING_FILL
            ws.cell(row=r, column=2, value=x[0]); ws.cell(row=r, column=3, value=x[1])
            ws.cell(row=r, column=4, value=x[2]); r += 1
        for rr in range(4, r):
            for c in range(1, 5):
                y = ws.cell(row=rr, column=c); y.border = BD
                y.alignment = Alignment(wrap_text=True, vertical='top')
        bad = [x for x in self.ing.counted() if x['note']]
        if bad:
            r += 1
            r = self.note(ws, r, 'INTEGRITY NOTES ON THE EXPORTS SUPPLIED', bold=True)
            for x in bad:
                r = self.note(ws, r, f'{x["file"]}: {x["note"]}')

DRIVES = {
 'R3B_TURNOVER':'Turnover, and the reverse-charge inward value used in the RCM test.',
 'R3B_TAX_PAYABLE':'Outward tax at Table 3.1(a), the basis of the GSTR-1 comparison.',
 'TAX_PAID':'Turnover, output tax, ITC availed and cash — the spine of every sheet.',
 'EXCESS_ITC':'The head-wise excess credit computation. THE DEMAND FIGURE.',
 'SGST_SETTLEMENT':'Settlement and gross SGST collected.',
 'HSN':'The HSN profile and the credit-note test against the original supply.',
 'GSTR9':'Whether the annual return and reconciliation statement were filed.',
 'R1_MONTHLY':'GSTR-1 against GSTR-3B, and which periods have no GSTR-1 at all.',
 'R2A_MONTHLY':'The independent credit test, with supplier and invoice counts.',
 'R3B_FILING':'Date of filing, delay in days, late fee and interest exposure.',
 'PAYMENTS':'Cash actually paid, challan by challan.',
 'R3B_FY_SUMMARY':'FY-level cross-check of the ITC figures.',
 'R2A_FY_SUMMARY':'FY-level cross-check of the credit available.',
 'R1_FY_SUMMARY':'FY-level cross-check of the GSTR-1 totals.',
 'R7_DETAIL':'Party-wise GSTR-7 detail for a month: who deducted, on what taxable value; tested against section 51.',
 'R7_TDS':'Tax deducted under section 51: the value of supplies the deductors paid for, tested against GSTR-3B and GSTR-1.',
}
EXTERNAL = [
 ('Deductor-wise GSTR-7 detail and GSTR-7A certificates',
  'Names each deductor, the contract and the payment, so the TDS can be matched to GSTR-1 invoice by invoice.',
  'Draw from the portal / back-office; the Prime statement gives counts and totals only.'),
 ('Supplier-wise (GSTIN-wise) GSTR-2A / 2B detail',
  'Who the suppliers are, whether they exist, whether they filed, whether they have been cancelled. '
  'The section 74 case usually turns on this.', 'Draw from the portal or from GST-Prime supplier reports.'),
 ('GSTR-2B for every period',
  'Entitlement after 1 January 2022 is governed by 2B, not 2A. The DRC-01C intimation must be built on 2B.',
  'Not available in the statements used here.'),
 ('Electronic credit ledger and cash ledger',
  'Confirms the CGST assumptions, the true cash position for Rule 86B, and what a credit note did to the balance.',
  'Draw from the portal.'),
 ('Full GSTR-1, table by table (B2B, B2CS, CDNR, Table 12)',
  'Needed wherever a return-level total does not reconcile with its own taxable value column.', 'Draw from the portal.'),
 ('E-way bill data',
  'Whether the goods actually moved, for high-value supplies and for the goods behind any credit note.',
  'GST-Prime carries an E-Way Bill tab; export separately.'),
 ('Audited accounts, income tax returns, Form 26AS, bank statements',
  'Independent test of turnover, and needed to rule out the Rule 86B exceptions.', 'Call for from the taxpayer.'),
 ('Departmental record — ASMT-10, DRC-01A, DRC-03, audit, intelligence',
  'Explains any unwinding of credit, and determines whether section 6(2)(b) bars a fresh proceeding.',
  'Obtain from the jurisdictional record.'),
]

# ------------------------------------------------------------- index etc.
def _register(b):
    ws = b.sheet('09_FILE_REGISTER')
    ing = b.ing
    ws['A1'] = 'REGISTER OF FILES RECEIVED'; ws['A1'].font = BOLD
    ws['A2'] = ('Duplicates are identified by MD5 checksum of the file itself, not by file name, so a file renamed on '
                're-upload is still caught. A duplicate is excluded from every count and every computation.')
    b.head(ws, 4, ['No.','File as uploaded','Batch','Checksum (MD5)','Report identified','Rows parsed',
                   'Records NEW to the case','Records REPLACED (already held)','Values CHANGED in those records',
                   'Records declared by the report','Status','Note'],
           [5,46,8,36,40,12,15,17,17,16,26,74])
    for i, r in enumerate(ing.register, 5):
        ws.cell(row=i, column=1, value=i-4); ws.cell(row=i, column=2, value=r['file'])
        ws.cell(row=i, column=3, value=r['batch']); ws.cell(row=i, column=4, value=r['md5'])
        ws.cell(row=i, column=5, value=r['title'] or '')
        ws.cell(row=i, column=6, value=r['rows'] or 0)
        ws.cell(row=i, column=7, value=r.get('rows_new', '') if r['status'] == 'COUNTED' else '')
        cc = ws.cell(row=i, column=8, value=r.get('rows_replaced', '') if r['status'] == 'COUNTED' else '')
        if r.get('rows_replaced') and b.mark: cc.fill = MISSING_FILL
        vc = ws.cell(row=i, column=9, value=r.get('values_changed', '') if r['status'] == 'COUNTED' else '')
        if r.get('values_changed') and b.mark: vc.fill = MISSING_FILL
        ws.cell(row=i, column=10, value=r['declared'] if r['declared'] is not None else '')
        c = ws.cell(row=i, column=11, value=r['status'])
        if r['status'] != 'COUNTED' and b.mark: c.fill = MISSING_FILL
        ws.cell(row=i, column=12, value=r['note'] or r['detect_note'] or '')
    last = 4 + len(ing.register)
    for rr in range(5, last+1):
        for c in range(1, 13):
            x = ws.cell(row=rr, column=c); x.border = BD
            x.alignment = Alignment(wrap_text=True, vertical='top')
    ws.cell(row=last+2, column=2, value='Files received').font = BOLD
    ws.cell(row=last+2, column=6, value=len(ing.register)).font = BOLD
    ws.cell(row=last+3, column=2, value='Exact duplicates excluded (byte-identical)').font = BOLD
    ws.cell(row=last+3, column=6, value=f'=COUNTIF(K5:K{last},"DUPLICATE*")').font = BOLD
    ws.cell(row=last+4, column=2, value='Not recognised or unreadable').font = BOLD
    ws.cell(row=last+4, column=6, value=f'=COUNTIF(K5:K{last},"NOT RECOGNISED")+COUNTIF(K5:K{last},"UNREADABLE")').font = BOLD
    ws.cell(row=last+5, column=2, value='Unique reports relied on').font = BOLD
    ws.cell(row=last+5, column=6, value=f'=F{last+2}-F{last+3}-F{last+4}').font = BOLD
    ws.cell(row=last+6, column=2, value='Records REPLACED by a later export (not added twice)').font = BOLD
    ws.cell(row=last+6, column=6, value=f'=SUM(H5:H{last})').font = BOLD
    ws.cell(row=last+7, column=2, value='Values that DIFFER from the earlier export — see 09A_RESTATEMENTS').font = BOLD
    ws.cell(row=last+7, column=6, value=f'=SUM(I5:I{last})').font = BOLD
    ws.freeze_panes = 'A5'
    n = last + 9
    b.note(ws, n, 'A file is rejected as a DUPLICATE only when it is byte-identical to one already held. '
                  'A later export of the same report that carries the same records plus new ones is NOT a '
                  'duplicate — it is accepted, and the records already held are REPLACED rather than added '
                  'a second time. Column H counts those. Without that, a re-export would roughly double '
                  'every total built on the overlapping records.', bold=True)

def _derivation(b):
    ws = b.sheet('08_DERIVATION_LOGIC')
    ws['A1'] = 'DERIVATION LOGIC — every derived column, its formula, its source and the reason for deriving it'
    ws['A1'].font = BOLD
    b.head(ws, 3, ['Sheet','Derived column','Formula logic','Source read','Reason / legal basis'], [24,28,46,28,86])
    for i, row in enumerate(DERIV, 4):
        for j, v in enumerate(row, 1):
            c = ws.cell(row=i, column=j, value=Builder._safe(v)); c.border = BD
            c.alignment = Alignment(wrap_text=True, vertical='top')

DERIV = [
 ['10_MONTHLY_MASTER','Every figure','INDEX/MATCH on Return Period into the SRC_ sheet named in column D',
  'SRC_TAX_PAID, SRC_R3B_TAX_PAYABLE, SRC_R3B_TURNOVER, SRC_EXCESS_ITC',
  'Nothing is retyped. Each cell is a live lookup, so a value can be traced to the exact source row it came from.'],
 ['10_MONTHLY_MASTER','Implied RCM / other liability','[GST OP Total] minus [Outward total tax 3.1(a)]',
  'SRC_TAX_PAID; SRC_R3B_TAX_PAYABLE',
  'GST OP Total includes the Table 3.1(d) reverse-charge liability while Table 3.1(a) does not. The difference isolates '
  'RCM so it can be tested on its own under section 9(3) and 9(4).'],
 ['10_MONTHLY_MASTER','NIL turnover / ITC availed in NIL period','IF(turnover = 0, "NIL", "") and IF(NIL, ITC, 0)',
  'Same sheet','Isolates credit availed with no corresponding business activity. Section 16(1).'],
 ['10_MONTHLY_MASTER','GSTR-2A data availability','Set where the 2A column was BLANK, not zero, in the source',
  'SRC_EXCESS_ITC','A blank is not a nil. Treating an unpopulated 2A as zero credit would overstate the excess.'],
 ['11_FY_SUMMARY','All columns','SUMIFS / COUNTIFS over 10_MONTHLY_MASTER by financial year','10_MONTHLY_MASTER',
  'Demands, annual returns and limitation all run financial-year wise.'],
 ['13_EXCESS_ITC','Excess claimed','Taken as reported by GST-Prime, head by head; CGST mirrored from SGST',
  'SRC_EXCESS_ITC','Section 16(2)(aa) and Rule 36(4) restrict credit to what the supplier reported. Heads are kept '
  'separate because an excess in one head is not set off against a shortfall in another for the purpose of a demand.'],
 ['13_EXCESS_ITC','Net versus gross','Net sums every period; gross counts only periods of over-claim','Same sheet',
  'A short claim followed by a larger claim is ordinarily only timing. Net is the defensible exposure.'],
 ['17_R1_vs_R3B','GSTR-1 filing status','Three-way test: period absent from the report / row present but blank / filed',
  'SRC_R1_MONTHLY','A period that does not appear, or appears blank, has no GSTR-1 on record. Distinguishing absent from '
  'filed-nil is what turns a blank into a section 37 contravention.'],
 ['17_R1_vs_R3B','Difference in tax','[GSTR-1 tax] minus [GSTR-3B Table 3.1(a) tax]','SRC_R1_MONTHLY; 10_MONTHLY_MASTER',
  'Compared against 3.1(a) and not against OP Total, because OP Total carries the RCM liability which has no counterpart '
  'in GSTR-1. A positive difference is self-assessed tax under the Explanation to section 75(12).'],
 ['19B_CREDIT_NOTES','Positive supply ever declared under the same HSN',
  'Every negative outward declaration is matched against every positive HSN row in the whole record',
  'SRC_HSN; 10_MONTHLY_MASTER',
  'Section 34(1) requires a credit note to be referable to an identified original tax invoice. If the taxpayer has never '
  'declared a positive supply under that HSN, there is nothing for the credit note to be referable to.'],
 ['19B_CREDIT_NOTES','Internally inconsistent',
  'Flagged where GSTR-1 tax is negative while GSTR-1 taxable value is not','SRC_R1_MONTHLY',
  'The two columns of the same return contradict each other. The return cannot be read from a summary and must be drawn '
  'table by table from the portal.'],
 ['18_RETURN_FILING','Statutory due date','20th of the month following the tax period','Constant, not from the source data',
  'The monthly filer due date. If the taxpayer was on QRMP in any period, that period must be re-set before late fee is computed.'],
 ['18_RETURN_FILING','Delay in days','MAX(0, [date of filing] minus [due date])','SRC_R3B_FILING',
  'Late fee under section 47(1) and interest under section 50 both run on days of delay.'],
 ['19_CASH_RECON','Cash as a proportion of output tax','[total cash paid] divided by [total output tax declared]',
  '19_CASH_RECON; 11_FY_SUMMARY','The single ratio that describes the case: what was declared against what was parted with.'],
 ['19A_ITC_vs_2A','GAP','[ITC availed] minus [2A credit available], suppressed where no 2A report exists',
  'SRC_R2A_MONTHLY; 10_MONTHLY_MASTER',
  'Built from reports that do not feed sheet 13, so the excess credit finding rests on two independent sources.'],
 ['19A_ITC_vs_2A','Do the two routes agree?','IF(ABS((monthly gap + untestable ITC) - annual gap) < 100, "yes", "CHECK")',
  'Same sheet','A live consistency test between the month-wise files and the annual summary. If a later upload breaks the '
  'agreement the cell says so.'],
 ['15_RULE_86B','Cash proxy','[SGST cash paid] x 2','SRC_TAX_PAID',
  'Only the SGST head of cash is reported. CGST cash is taken as equal; IGST cash is not captured. Prima facie only — '
  'cross-check against the challans in sheet 19.'],
 ['19C_TDS_GSTR7','Value paid for by the deductors','([Total GST] minus [Cess]) / 2%','SRC_R7_TDS',
  'Section 51: 1% CGST + 1% SGST, or 2% IGST, of the value of supply excluding the tax on the invoice. Total GST is used '
  'because Prime rounds each head but totals the unrounded heads. Cess is not deducted under section 51.'],
 ['19C_TDS_GSTR7','Cumulative value / cumulative GSTR-3B turnover','SUMIF of the rows so far in the same FY; SUM of the '
  'master rows from the first period of the FY to this period','This sheet; 10_MONTHLY_MASTER',
  'Payment month is not invoice month. By s.13(2) (services) and s.31(1) (goods) a supply paid for by a month must have been '
  'declared by that month, so the cumulative position is the firm test; a single month can be timing.'],
 ['19C_TDS_GSTR7','SGST TDS not absorbed','MAX(0, SGST deducted - SGST paid in cash in GSTR-3B), by FY',
  'This sheet; 10_MONTHLY_MASTER','TDS is credited to the cash ledger (Rule 87(9)). If it is not used, a balance builds up '
  'that can be claimed as refund under section 54.'],
 ['19C_TDS_GSTR7','Indicative tax','Shortfall x the rate typed in H3','This sheet',
  'The only assumed figure in the workbook, kept on this sheet and out of 20_FINDINGS.'],
 ['19D_TDS_PARTIES','Registration type; PAN holder type','Position 14 of the GSTIN (D = tax-deductor registration, Z = '
  'ordinary); position 4 of the PAN (P individual, F firm, C company, G Government, L local authority ...)','SRC_R7_DETAIL',
  'Read off the GSTIN itself. Section 51(1) and Notification 50/2018-CT limit deductors to Government, local authorities, '
  'Governmental agencies, Government-controlled bodies and PSUs. To be confirmed on the portal.'],
 ['19D_TDS_PARTIES','Lines / taxable value / TDS','COUNTIF / SUMIF of SRC_R7_DETAIL by GSTIN and by month','SRC_R7_DETAIL',
  'Live totals over the source rows.'],
 ['19D_TDS_PARTIES','Agrees?','TDS difference within Re.1 plus half a rupee per line, and the party count equal to the summary\'s '
  '"No Of Deductors"','SRC_R7_DETAIL; SRC_R7_TDS','The detail heads are rounded per line, so a few rupees of rounding is expected.'],
 ['SRC_R7_DETAIL','Return Period','Taken from the FILE NAME (..._Month_Mar-2026.xlsx)','File name',
  'The export carries no period inside it. The register says so against every such file.'],
 ['24_LIMITATION','All dates','Annual return due 31 December following the FY; section 73 order +3 years, notice 3 months '
  'earlier; section 74 order +5 years, notice 6 months earlier','Computed from the statute',
  'Indicative only. Extension notifications are not applied and several have been litigated.'],
 ['SRC_ sheets','Record history column; amber row; orange cell',
  'History text on every record replaced by a later export; amber row and orange cell only where a value differs',
  'The engine, at ingest','A trail inside the workbook, not only in the register. Hover over an orange cell for the value it replaced.'],
 ['09A_RESTATEMENTS','Difference (later minus earlier)','IF both values are numbers, later minus earlier',
  'Same sheet','Shows the size of the movement. SNo, Growth and Trend columns are ignored: SNo is only the row position in an '
  'export and the other two are derived. A difference under Re.1 is treated as rounding.'],
 ['09A_RESTATEMENTS','Level (REVIEW / NOTE / INFO)',
  'INFO for GSTR-2A, which moves whenever a supplier files or amends; NOTE for blank-to-populated; REVIEW otherwise',
  'Report type and field','GSTR-2A changing is expected and is not a restatement by the taxpayer. GSTR-3B and challans cannot be '
  'revised, so a change there is the one to look at.'],
 ['09_FILE_REGISTER','Unique reports relied on','[files received] minus duplicates minus unrecognised','Same sheet',
  'Duplicates are identified by MD5 of the file, so a renamed re-upload is still caught.'],
]

def _index(b, version, run_dt):
    ws = b.sheet('00_INDEX')
    rows = [['GST-PRIME PRELIMINARY INVESTIGATION', '', ''],
            ['Taxpayer', b.name, ''],
            ['GSTIN', b.gstin, '']]
    for k, v in (b.profile or {}).items():
        rows.append([k, v, ''])
    m = b.R['master']
    ap = b.ing.all_periods()
    cover = (f'{m[0]["period"]} to {m[-1]["period"]} ({len(m)} tax periods with GSTR-3B)' if m else
             (f'{ap[0]} to {ap[-1]} — NO GSTR-3B REPORT SUPPLIED; only ' +
              ', '.join(REPORT_SPECS[c]['title'] for c in REPORT_SPECS if b.ing.have(c)) if ap else 'no periods'))
    rows += [['Period covered', cover, ''],
             ['Periods with no GSTR-3B', ', '.join(b.R.get('unfiled') or []) or 'none'
                 + ('  (kept out of the period count and the NIL tests)' if b.R.get('unfiled') else ''), ''],
             ['Reports relied on', f'{len(b.ing.counted())} unique GST-Prime statements '
                                   f'({len(b.ing.register)} files received, {len(b.ing.duplicates())} exact duplicates '
                                   f'excluded) — see 09_FILE_REGISTER', ''],
             ['Revenue implication', f'Rs. {inr(b.total_findings)} before interest and penalty — see 20_FINDINGS', ''],
             ['', '', ''],
             ['HOW THIS WORKBOOK IS PUT TOGETHER', '', ''],
             ['SRC_ sheets', 'Every GST-Prime report, reproduced exactly as received. No value altered. A cell shaded '
                             'light red was BLANK in the source, which is not the same as nil.', ''],
             ['08_DERIVATION_LOGIC', 'For every derived column: the formula, the source it reads, and the reason for '
                                     'deriving it.', ''],
             ['09_FILE_REGISTER', 'Every file received, with checksum, what it was identified as, and whether it was counted.', ''],
             ['10 to 19D', 'The computation sheets. Every figure is a live Excel formula reading an SRC_ sheet or the '
                           'monthly master — nothing is retyped.', ''],
             ['20 to 24', 'Findings, report, action points, data position and limitation.', ''],
             ['', '', ''],
             ['SHEET INDEX', '', '']]
    order = [s for s in b.wb.sheetnames]
    for s in order:
        rows.append([s, SHEET_NOTE.get(s, 'Source data, reproduced as received.' if s.startswith('SRC_') else ''), ''])
    rows += [['', '', ''],
             ['VERSION', '', ''],
             ['Engine version', version, ''],
             ['Run at', run_dt.strftime('%d-%m-%Y %H:%M'), ''],
             ['', '', ''],
             ['SCOPE', '', ''],
             ['This is a PRELIMINARY INVESTIGATION built only from GST-Prime Analytica statements. It is an analysis of '
              'returns data and is not a finding of fact.', '', ''],
             ['Nothing has been estimated, interpolated or assumed. Where a report was not supplied, the test that needed '
              'it says so and was not run.', '', ''],
             ['Every figure is to be re-verified against the GST common portal, the electronic ledgers, the returns '
              'themselves and the books of account before any notice is issued.', '', '']]
    for i, r in enumerate(rows, 1):
        for j, v in enumerate(r, 1):
            if v:
                c = ws.cell(row=i, column=j, value=v); c.alignment = Alignment(wrap_text=True, vertical='top')
    for i, r in enumerate(rows, 1):
        if r[0] and r[0].isupper() and not r[1]: ws.cell(row=i, column=1).font = BOLD
    ws.cell(row=1, column=1).font = BOLD
    for i, w in enumerate([32, 92, 20], 1): ws.column_dimensions[get_column_letter(i)].width = w

SHEET_NOTE = {
 '00_INDEX':'This sheet.',
 '08_DERIVATION_LOGIC':'Every derived column explained.',
 '09_FILE_REGISTER':'Files received, checksums, duplicates.',
 '09A_RESTATEMENTS':'What a later export changed from an earlier one.',
 '10_MONTHLY_MASTER':'One row per tax period; every cell a live lookup.',
 '11_FY_SUMMARY':'Financial-year roll-up.',
 '12_HSN_PROFILE':'What the taxpayer says it supplies, in order.',
 '13_EXCESS_ITC':'ITC against GSTR-2A, head by head. THE DEMAND FIGURE.',
 '14_DORMANCY_ITC':'Credit availed in NIL-turnover periods.',
 '15_RULE_86B':'Cash component of the output tax discharge.',
 '16_RCM_CHECK':'Reverse charge declared and implied.',
 '17_R1_vs_R3B':'GSTR-1 against GSTR-3B, every period, with filing status.',
 '18_RETURN_FILING':'Filing dates against due dates; delay in days.',
 '19_CASH_RECON':'Every challan, and cash against declared liability.',
 '19A_ITC_vs_2A':'The credit test re-run from independent reports.',
 '19B_CREDIT_NOTES':'Credit notes tested against the original supply.',
 '19D_TDS_PARTIES':'GSTR-7 party by party: who deducted, whether section 51 allows them to, and the month reconciliation.',
 '19C_TDS_GSTR7':'What the deductors paid for (GSTR-7 TDS / 2%) against GSTR-3B and GSTR-1; the legal cross-checks.',
 '20_FINDINGS':'Consolidated findings with quantification.',
 '21_REPORT':'The report in narrative form.',
 '22_ACTION_POINTS':'What to do, in the order to do it.',
 '23_DATA_GAPS':'What is held and what is still required.',
 '24_LIMITATION':'Indicative outer dates by financial year.',
}

ORDER_HINT = ['00_INDEX'] + [f'SRC_{c}'[:31] for c in REPORT_SPECS] + \
             ['08_DERIVATION_LOGIC','09_FILE_REGISTER','09A_RESTATEMENTS','10_MONTHLY_MASTER','11_FY_SUMMARY','12_HSN_PROFILE',
              '13_EXCESS_ITC','14_DORMANCY_ITC','15_RULE_86B','16_RCM_CHECK','17_R1_vs_R3B','18_RETURN_FILING',
              '19_CASH_RECON','19A_ITC_vs_2A','19B_CREDIT_NOTES','19C_TDS_GSTR7','19D_TDS_PARTIES','20_FINDINGS','21_REPORT','22_ACTION_POINTS',
              '23_DATA_GAPS','24_LIMITATION']

def build_workbook(ing, R, gstin, name, out_path, mark_missing_light_red=True, profile=None):
    b = Builder(ing, R, gstin, name, mark_missing_light_red, profile)
    b.build_sources()
    b.build_master(); b.build_fy()
    b.build_hsn(); b.build_excess(); b.build_dormancy(); b.build_86b(); b.build_rcm()
    b.build_r1(); b.build_filing(); b.build_cash(); b.build_itc_2a(); b.build_credit_notes(); b.build_tds(); b.build_tds_parties()
    F = b.build_findings(); b.build_report(F); b.build_actions(F); b.build_gaps(); b.build_limitation()
    b.build_restatements(); _derivation(b); _register(b)
    _index(b, BUILD_VERSION, datetime.datetime.now())
    names = [s for s in ORDER_HINT if s in b.wb.sheetnames] + \
            [s for s in b.wb.sheetnames if s not in ORDER_HINT]
    b.wb._sheets = [b.wb[s] for s in names]
    for s in b.wb.worksheets:
        s.page_setup.orientation = 'landscape'; s.page_setup.fitToWidth = 1
        s.sheet_properties.pageSetUpPr.fitToPage = True
    b.wb.active = 0
    b.wb.save(out_path)
    return out_path, b.total_findings


In [ ]:
# Keep a copy of the engine in this taxpayer's folder, then load it.
require_drive()
import importlib
for _f in ("gpe_core.py", "gpe_checks.py", "gpe_build.py"):
    if os.path.exists(_f):
        shutil.copy(_f, os.path.join(DIRS["engine"], _f))
    elif os.path.exists(os.path.join(DIRS["engine"], _f)):
        shutil.copy(os.path.join(DIRS["engine"], _f), _f)   # restored from Drive
    else:
        raise FileNotFoundError(f"{_f} is missing — run the three engine cells above.")
sys.path.insert(0, os.getcwd())

try:
    import openpyxl
except ImportError:
    !pip -q install openpyxl
    import openpyxl

for _m in ("gpe_core", "gpe_checks", "gpe_build"):
    if _m in sys.modules: importlib.reload(sys.modules[_m])
import gpe_core, gpe_checks, gpe_build
from gpe_core import Ingest, REPORT_SPECS, inr
from gpe_checks import run_all
from gpe_build import build_workbook

print("Engine loaded — core", gpe_core.ENGINE_VERSION,
      "| checks", gpe_checks.CHECKS_VERSION,
      "| builder", gpe_build.BUILD_VERSION)
print("Engine copy kept at", DIRS["engine"])
print(f"\n{len(REPORT_SPECS)} GST-Prime report types recognised:")
for c, sp in REPORT_SPECS.items():
    print(f"   {c:<18} {sp['title']}")


## Step 6 · Upload the GST-Prime exports

The banner names the taxpayer these files will be filed under — **check it before you pick the files.** If it is the wrong taxpayer, go back to Step 2 and change the GSTIN.

Upload whatever you have, all at once or in batches over several sessions. Anything already in `01_RAW_UPLOADS` is reused, so you never upload the same file twice. Every file is recorded in `_MANIFEST.csv` with its checksum, the time it arrived and the GSTIN it was filed under.

**You do not need every report.** The workbook runs on whatever is present and tells you, on sheet `23_DATA_GAPS`, which tests could not be run and what to fetch.

In [ ]:
require_drive()
bar = "=" * 78
print(bar)
print("FILES WILL BE FILED UNDER".center(78))
print(f"{GSTIN}   {TAXPAYER_NAME}".center(78))
print(bar)

MANIFEST = os.path.join(DIRS["raw"], "_MANIFEST.csv")
DUPDIR   = os.path.join(DIRS["raw"], "_DUPLICATES")
MAN_COLS = ["gstin", "taxpayer", "stored_as", "original_name", "md5",
            "size_bytes", "uploaded_at", "status"]

def _md5_bytes(b):
    return hashlib.md5(b).hexdigest()

def _md5(p, buf=1 << 20):
    h = hashlib.md5()
    with open(p, "rb") as fh:
        while True:
            c = fh.read(buf)
            if not c: break
            h.update(c)
    return h.hexdigest()

def _manifest_rows():
    if not os.path.exists(MANIFEST):
        return []
    return list(csv.DictReader(open(MANIFEST, encoding="utf-8")))

def _manifest_migrate():
    """Older runs wrote the manifest without a status column. Bring it forward once."""
    if not os.path.exists(MANIFEST):
        return
    rows = _manifest_rows()
    if rows and "status" in rows[0]:
        return
    with open(MANIFEST, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=MAN_COLS)
        w.writeheader()
        for r in rows:
            r["status"] = "stored"
            w.writerow({k: r.get(k, "") for k in MAN_COLS})
    print("Manifest brought forward to the current format.")

def _manifest_append(rows):
    new = not os.path.exists(MANIFEST)
    with open(MANIFEST, "a", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=MAN_COLS)
        if new: w.writeheader()
        for r in rows: w.writerow(r)

_manifest_migrate()

# ---- housekeeping: quarantine byte-identical copies already sitting in the folder -----
def _held_index():
    """md5 -> filename, for every .xlsx currently in 01_RAW_UPLOADS."""
    idx = {}
    for f in sorted(os.listdir(DIRS["raw"])):
        if f.lower().endswith((".xlsx", ".xls")) and not f.startswith("~$"):
            idx.setdefault(_md5(os.path.join(DIRS["raw"], f)), f)
    return idx

_seen, _quar = {}, []
for f in sorted(os.listdir(DIRS["raw"])):
    if not f.lower().endswith((".xlsx", ".xls")) or f.startswith("~$"):
        continue
    p = os.path.join(DIRS["raw"], f)
    h = _md5(p)
    if h in _seen:
        os.makedirs(DUPDIR, exist_ok=True)
        shutil.move(p, os.path.join(DUPDIR, f))
        _quar.append((f, _seen[h]))
    else:
        _seen[h] = f
if _quar:
    print(f"\nHousekeeping — {len(_quar)} byte-identical copy(ies) already in the folder "
          f"moved to _DUPLICATES/ so they are not read twice:")
    for f, keep in _quar:
        print(f"    {f}   (identical to {keep})")
    print("    Review or clear them with the 'Review _DUPLICATES' cell at the end of the notebook.")

# ---- upload ------------------------------------------------------------------------
held = _held_index()
stored_n = skipped_n = 0
if IN_COLAB:
    from google.colab import files
    print("\nChoose the GST-Prime .xlsx exports (press Cancel if you have nothing new to add)\n")
    try:
        up = files.upload()
    except Exception:
        up = {}
    now = datetime.datetime.now().isoformat(timespec="seconds")
    rows = []
    for name, blob in up.items():
        h = _md5_bytes(blob)
        if os.path.exists(name):
            os.remove(name)                      # Colab drops a copy in /content
        if h in held:                            # SAME CONTENT -> do not store again
            skipped_n += 1
            same = held[h]
            print(f"  ALREADY HELD, not stored again : {name}"
                  + (f"   (identical to {same})" if same != name else ""))
            rows.append(dict(gstin=GSTIN, taxpayer=TAXPAYER_NAME, stored_as=same,
                             original_name=name, md5=h, size_bytes=len(blob),
                             uploaded_at=now, status="skipped-duplicate"))
            continue
        stored = name                            # new content
        dest = os.path.join(DIRS["raw"], stored)
        if os.path.exists(dest):                 # same name, different content -> keep both
            base, ext = os.path.splitext(name)
            stored = f"{base}__{datetime.datetime.now():%Y%m%d%H%M%S}{ext}"
            dest = os.path.join(DIRS["raw"], stored)
            print(f"  NEW CONTENT under an existing name, stored as : {stored}")
        with open(dest, "wb") as fh:
            fh.write(blob)
        held[h] = stored
        stored_n += 1
        rows.append(dict(gstin=GSTIN, taxpayer=TAXPAYER_NAME, stored_as=stored,
                         original_name=name, md5=h, size_bytes=len(blob),
                         uploaded_at=now, status="stored"))
    if rows:
        _manifest_append(rows)
    print(f"\n{stored_n} file(s) stored, {skipped_n} skipped as already held.")
else:
    print("\nNot in Colab — drop the .xlsx files into:\n   ", DIRS["raw"])
    known = {r["stored_as"] for r in _manifest_rows()}
    now = datetime.datetime.now().isoformat(timespec="seconds")
    rows = [dict(gstin=GSTIN, taxpayer=TAXPAYER_NAME, stored_as=f, original_name=f,
                 md5=_md5(os.path.join(DIRS["raw"], f)),
                 size_bytes=os.path.getsize(os.path.join(DIRS["raw"], f)),
                 uploaded_at=now, status="stored")
            for f in sorted(os.listdir(DIRS["raw"]))
            if f.lower().endswith((".xlsx", ".xls")) and f not in known]
    if rows:
        _manifest_append(rows)

files_held = sorted(f for f in os.listdir(DIRS["raw"]) if f.lower().endswith((".xlsx", ".xls")))
print(f"\n{len(files_held)} file(s) now held for {GSTIN} — {TAXPAYER_NAME}:")
for f in files_held:
    print("   ", f)
print("\nManifest:", MANIFEST)
print("A file that is byte-identical to one already held is never stored again. A LATER EXPORT")
print("carrying the same records plus new ones IS stored — the records already held are then")
print("replaced rather than counted twice. Step 7 reports how many.")


## Step 7 · Extract

Every file is identified from its header columns, parsed, and the result written to `02_EXTRACTED`. Read the register carefully — it is where a bad export shows up.

In [ ]:
require_drive()
print(f"Extracting for {GSTIN} — {TAXPAYER_NAME}\n")

# Read in the order the files arrived, so that where a later export corrects an earlier one,
# the later record is the one kept. The manifest carries the upload time; anything not in it
# falls back to the file's own timestamp.
_when = {}
for _r in _manifest_rows():
    if _r.get("status") == "stored":
        _when[_r["stored_as"]] = _r.get("uploaded_at", "")
def _order(f):
    p = os.path.join(DIRS["raw"], f)
    return (_when.get(f) or datetime.datetime.fromtimestamp(os.path.getmtime(p)).isoformat(), f)

_files = [f for f in os.listdir(DIRS["raw"])
          if f.lower().endswith((".xlsx", ".xls")) and not f.startswith("~$")]
ing = Ingest(GSTIN)      # exports that name a GSTIN in their title are checked against this case
for f in sorted(_files, key=_order):
    # 'batch' is the upload time from the manifest: two files with the same batch arrived together,
    # and nothing then shows which of them is the newer export.
    ing.add(os.path.join(DIRS["raw"], f), batch=_order(f)[0])

_wrong = [r for r in ing.register if r.get("subject_gstin") and r["subject_gstin"] != GSTIN]
if _wrong:
    print("!" * 78)
    print("WRONG TAXPAYER? These exports are titled for a different GSTIN:".center(78))
    for r in _wrong:
        print(f"   {r['file']}  ->  {r['subject_gstin']}")
    print("Move them to the right case with the re-file utility, or delete them, and run Step 7 again.")
    print("!" * 78 + "\n")
print("=" * 78)
print(f"{'FILE':<48}{'IDENTIFIED AS':<22}{'ROWS':>6}  STATUS")
print("=" * 78)
for r in ing.register:
    print(f"{r['file'][:46]:<48}{(r['code'] or '-'):<22}{r['rows']:>6}  {r['status']}")
print("=" * 78)
_repl = sum(r.get("rows_replaced", 0) for r in ing.counted())
print(f"received {len(ing.register)}  |  counted {len(ing.counted())}  |  "
      f"duplicates {len(ing.duplicates())}  |  not recognised {len(ing.problems())}")
if _repl:
    print(f"\n{_repl} record(s) came from a later export of a report already held. They REPLACED the")
    print("earlier copy instead of being added again, so no total is counted twice:")
    for r in ing.counted():
        if r.get("rows_replaced"):
            print(f"    {r['file'][:52]:<54} {r['rows_replaced']:>4} replaced, {r.get('rows_new',0):>4} new")

# ---- RESTATEMENT WARNING: a later export changed a value the earlier export already gave us -----
_rev  = [c for c in ing.changes if c["level"] == "REVIEW"]
_note = [c for c in ing.changes if c["level"] == "NOTE"]
_info = [c for c in ing.changes if c["level"] == "INFO"]
if ing.changes:
    print("\n" + "=" * 78)
    print(("RESTATEMENT WARNING" if _rev else "CHANGES BETWEEN EXPORTS").center(78))
    print("=" * 78)
    print(f"A later export differs from an earlier one in {len(ing.changes)} value(s):")
    print(f"   {len(_rev):>4}  REVIEW  a taxpayer-reported value or a challan changed")
    print(f"   {len(_note):>4}  NOTE    blank before, populated now (most likely a return filed since)")
    print(f"   {len(_info):>4}  INFO    GSTR-2A moved (expected: suppliers file late and amend)")
    _unc = [c for c in ing.changes if c.get("order_uncertain")]
    if _unc:
        print(f"\n   {len(_unc)} of these come from files that arrived in the SAME upload. Prime exports carry no date,")
        print("   so nothing shows which is newer: the value called 'later' below is only the one that sorted last")
        print("   by filename. Work out which export was taken more recently, delete the other from")
        print("   01_RAW_UPLOADS, and run Step 7 again.")
    if _rev:
        print("\nREVIEW — verify these on the portal before relying on either figure:")
        for c in _rev[:40]:
            print(f"   {c['record']:<26}{c['field']:<26}{c['old']}  ->  {c['new']}"
                  + ("   [ORDER UNCERTAIN]" if c.get("order_uncertain") else ""))
            print(f"        {c['old_file']}  ->  {c['new_file']}")
        if len(_rev) > 40:
            print(f"   ... and {len(_rev) - 40} more — all of them are in sheet 09A_RESTATEMENTS")
        print("\nGSTR-3B and payment challans cannot be revised, so a change in either is unusual. It may mean")
        print("the taxpayer amended something, Prime refreshed its data, or the earlier export was taken before")
        print("the period was final. The LATER value is the one used from here on.")
        print("If the earlier value is the correct one, remove the later file and run again.")
    print("=" * 78)

if ing.duplicates():
    print("\nDUPLICATES REJECTED (byte-identical to a file already held):")
    for r in ing.duplicates():
        print("  ", r["file"], "->", r["note"])
if ing.problems():
    print("\nNOT USED — look at these:")
    for r in ing.problems():
        print("  ", r["file"], "->", r["note"])

notes = [r for r in ing.counted() if r["note"]]
if notes:
    print("\nINTEGRITY NOTES ON THE EXPORTS (the export itself is inconsistent):")
    for r in notes:
        print("  ", r["file"], "->", r["note"])

missing = [c for c in REPORT_SPECS if not ing.have(c)]
if missing:
    print("\nNOT SUPPLIED — the tests that need these will be skipped, not guessed:")
    for c in missing:
        print("   ", c, "-", REPORT_SPECS[c]["title"])
else:
    print(f"\nAll {len(REPORT_SPECS)} report types are present.")

stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
ex_json = os.path.join(DIRS["extract"], f"extracted_{CASE_ID}_{stamp}.json")
with open(ex_json, "w", encoding="utf-8") as fh:
    json.dump({"gstin": GSTIN, "taxpayer": TAXPAYER_NAME, "extracted_at": stamp,
               "register": ing.register, "data": ing.data,
               "restatements": ing.changes}, fh, default=str, indent=1)
print("\nExtraction saved to", ex_json)


## Step 8 · Run the checks and build the report

The workbook is written to this taxpayer's `03_REPORTS` folder and also downloaded. The filename carries the GSTIN, the version and the date, so earlier runs are never overwritten.

In [ ]:
require_drive()
R = run_all(ing)

def _line(k, v): print(f"  {k:<46}{v}")
print("=" * 78); print(f"SUMMARY — {GSTIN}  {TAXPAYER_NAME}"); print("=" * 78)
m = R["master"]
if m:
    _line("Tax periods covered", f"{len(m)}  ({m[0]['period']} to {m[-1]['period']})")
else:
    _line("Tax periods covered", "NONE — no GSTR-3B report uploaded; only the third-party statements are analysed")
tot_op = sum(x["op_total"] for x in m); tot_itc = sum(x["itc"] for x in m)
if m:   # with no GSTR-3B these are not zero, they are unknown — so they are not printed
    _line("Turnover declared", "Rs. " + inr(sum(x['turnover'] for x in m)))
    _line("Output tax declared", "Rs. " + inr(tot_op))
    _line("ITC availed", "Rs. " + inr(tot_itc))
if R["cash"].get("available"):
    _line("Cash actually paid", f"Rs. {inr(R['cash']['totals']['TOTAL'])}  ({R['cash']['count']} challans)")
for key, label in [("r1_vs_r3b", "GSTR-1 vs GSTR-3B"), ("excess_itc", "Excess ITC over GSTR-2A"),
                   ("itc_vs_2a", "ITC vs 2A (independent)"), ("dormancy", "NIL-period credit"),
                   ("rule86b", "Rule 86B"), ("rcm", "Reverse charge"),
                   ("credit_notes", "Credit notes"), ("filing", "Return filing"),
                   ("hsn", "HSN profile"), ("annual", "Annual return"), ("tds", "GSTR-7 TDS")]:
    r = R[key]
    if not r.get("available"):
        print(f"  {label:<46}NOT RUN — needs {', '.join(r.get('missing', []))}")
if R["tds"].get("available"):
    _t = R["tds"]
    _line("GSTR-7: value TDS was deducted on (TDS / 2%)", f"Rs. {inr(_t['total_value'])}  ({_t['first']} to {_t['last']})")
    _d = R.get("tds_detail") or {}
    if _d.get("available"):
        _line("GSTR-7 party-wise detail", f"{len(_d['months'])} month(s), {len(_d['parties'])} part(ies)"
              + (f" — {len(_d['ineligible'])} NOT PERMITTED to deduct under s.51" if _d['ineligible'] else ""))
    if not (_t["have_3b"] and _t["have_r1"]):
        print("  " + " " * 46 + "-> cross-checks against GSTR-3B / GSTR-1 pending: upload those reports")
print("-" * 78)
if R["r1_vs_r3b"].get("available"):
    _line("GSTR-1 tax not carried into GSTR-3B", "Rs. " + inr(R["r1_vs_r3b"]["short_total"]))
    _line("Periods with NO GSTR-1 on record", f"{len(R['r1_vs_r3b']['not_filed'])}  "
          f"{', '.join(R['r1_vs_r3b']['not_filed']) or '-'}")
if R["excess_itc"].get("available"):
    _line("Excess ITC over 2A (net)", "Rs. " + inr(R["excess_itc"]["net"]))
if R["dormancy"].get("available") and R["dormancy"]["longest"]:
    a, b, c = R["dormancy"]["longest"]
    _line("Longest NIL run", f"{c} periods, {a} to {b}")
if R["credit_notes"].get("available"):
    for e in R["credit_notes"]["events"]:
        _line(f"Credit note {e['period']}", "Rs. " + inr(abs(e["tax"])) +
              ("  — NO ORIGINAL SUPPLY UNDER THIS HSN"
               if any(not v for v in e["prior_positive_same_hsn"].values()) else ""))
    for x in R["credit_notes"]["r1_only"]:
        _line(f"GSTR-1 credit note {x['period']}", "Rs. " + inr(abs(x["r1_tax"])) +
              ("  — INTERNALLY INCONSISTENT" if x["inconsistent"] else ""))

stamp = datetime.datetime.now().strftime("%Y-%m-%d")
out_name = f"Preliminary_Investigation_{CASE_ID}_v{REPORT_VERSION}_{stamp}.xlsx"
out_path = os.path.join(DIRS["reports"], out_name)

path, total = build_workbook(ing, R, GSTIN, TAXPAYER_NAME, out_path,
                             mark_missing_light_red=MARK_MISSING_LIGHT_RED,
                             profile=PROFILE)
print("=" * 78)
print("REVENUE IMPLICATION ON THE FACE OF THE RETURNS:  Rs. " + inr(total))
print("   (before interest under section 50 and before penalty)")
print("=" * 78)
print("Report written to:", path)
if ing.changes:
    print(f"NOTE: {len(ing.changes)} value(s) differ between exports — see sheet 09A_RESTATEMENTS "
          f"({len([c for c in ing.changes if c['level']=='REVIEW'])} need review).")

if IN_COLAB:
    try:
        from google.colab import files as _f
        _f.download(path)
    except Exception as e:
        print("Download skipped:", e, "— the file is in Drive.")


## Utility · Review the `_DUPLICATES` folder

Byte-identical copies found in `01_RAW_UPLOADS` are moved into `_DUPLICATES/` so they are never read twice. Nothing is deleted automatically. This cell shows what is there and, once you are satisfied, clears it.

**A file is only ever deleted if an identical copy is still held in `01_RAW_UPLOADS`.** The checksum is re-checked at the moment of deletion. A quarantined file with no identical twin is kept and flagged, because deleting it would lose data.

Tick the box and run again to delete. Left unticked it only lists.

In [ ]:
#@title  Review the _DUPLICATES folder  { run: "auto", display-mode: "form" }
DELETE_QUARANTINED = False  #@param {type:"boolean"}
require_drive()

DUPDIR = os.path.join(DIRS["raw"], "_DUPLICATES")
MANIFEST = os.path.join(DIRS["raw"], "_MANIFEST.csv")
MAN_COLS = ["gstin", "taxpayer", "stored_as", "original_name", "md5",
            "size_bytes", "uploaded_at", "status"]

def _md5f(p, buf=1 << 20):
    h = hashlib.md5()
    with open(p, "rb") as fh:
        while True:
            c = fh.read(buf)
            if not c: break
            h.update(c)
    return h.hexdigest()

_held_md5 = {}
for _f in sorted(os.listdir(DIRS["raw"])):
    if _f.lower().endswith((".xlsx", ".xls")) and not _f.startswith("~$"):
        _held_md5.setdefault(_md5f(os.path.join(DIRS["raw"], _f)), _f)

_q = []
if os.path.isdir(DUPDIR):
    for _f in sorted(os.listdir(DUPDIR)):
        _p = os.path.join(DUPDIR, _f)
        if os.path.isfile(_p):
            _h = _md5f(_p)
            _q.append(dict(file=_f, path=_p, md5=_h, size=os.path.getsize(_p),
                           when=datetime.datetime.fromtimestamp(os.path.getmtime(_p)).strftime("%d-%m-%Y %H:%M"),
                           twin=_held_md5.get(_h)))

# how many upload attempts were turned away at the door, from the manifest
_skipped = [r for r in (list(csv.DictReader(open(MANIFEST, encoding="utf-8"))) if os.path.exists(MANIFEST) else [])
            if r.get("status") == "skipped-duplicate"]

print(f"{GSTIN}  {TAXPAYER_NAME}")
print(f"Upload attempts turned away as already held (never stored): {len(_skipped)}")
print(f"Files in _DUPLICATES: {len(_q)}\n")

if not _q:
    print("Nothing quarantined.")
else:
    print(f"{'FILE':<50}{'SIZE':>9}  {'MOVED':<17}  IDENTICAL TO (still held)")
    print("-" * 118)
    for _x in _q:
        print(f"{_x['file'][:48]:<50}{_x['size']:>9,}  {_x['when']:<17}  "
              + (_x["twin"] if _x["twin"] else "*** NO TWIN HELD — would be kept ***"))
    _safe   = [x for x in _q if x["twin"]]
    _unsafe = [x for x in _q if not x["twin"]]
    print(f"\n{len(_safe)} safe to delete (an identical copy is still in 01_RAW_UPLOADS)")
    if _unsafe:
        print(f"{len(_unsafe)} have NO identical copy held and will be KEPT — restore them to 01_RAW_UPLOADS if wanted.")

    if not DELETE_QUARANTINED:
        print("\nNothing deleted. Tick DELETE_QUARANTINED above and run again to clear the safe ones.")
    else:
        _now = datetime.datetime.now().isoformat(timespec="seconds")
        _log, _done = [], 0
        for _x in _safe:
            # re-check at the moment of deletion, not just at listing
            _twin_path = os.path.join(DIRS["raw"], _x["twin"])
            if (os.path.exists(_twin_path) and _md5f(_x["path"]) == _x["md5"]
                    and _md5f(_twin_path) == _x["md5"]):
                os.remove(_x["path"]); _done += 1
                _log.append(dict(gstin=GSTIN, taxpayer=TAXPAYER_NAME, stored_as=_x["twin"],
                                 original_name=_x["file"], md5=_x["md5"], size_bytes=_x["size"],
                                 uploaded_at=_now, status="quarantine-deleted"))
        if _log:
            _new = not os.path.exists(MANIFEST)
            with open(MANIFEST, "a", newline="", encoding="utf-8") as fh:
                w = csv.DictWriter(fh, fieldnames=MAN_COLS)
                if _new: w.writeheader()
                for r in _log: w.writerow(r)
        print(f"\nDeleted {_done} file(s). Each is recorded in the manifest as quarantine-deleted.")
        if _unsafe:
            print(f"{len(_unsafe)} file(s) with no identical copy were left in place.")
        if os.path.isdir(DUPDIR) and not os.listdir(DUPDIR):
            os.rmdir(DUPDIR); print("_DUPLICATES is now empty and has been removed.")


## Utility · Re-file a case under the correct GSTIN

Only needed if files were uploaded before the GSTIN was set — for example under an example GSTIN. Set the two values and run. It moves the raw uploads across, keeps the manifest straight, and leaves the old folder's reports alone so nothing is lost.

*Leave `WRONG_GSTIN` blank to do nothing.*

In [ ]:
WRONG_GSTIN   = ""  #@param {type:"string"}
CORRECT_GSTIN = ""  #@param {type:"string"}

WRONG_GSTIN, CORRECT_GSTIN = WRONG_GSTIN.strip().upper(), CORRECT_GSTIN.strip().upper()
require_drive()
if not WRONG_GSTIN:
    print("Nothing to do — WRONG_GSTIN is blank.")
else:
    src = os.path.join(DRIVE_ROOT, WRONG_GSTIN)
    dst = os.path.join(DRIVE_ROOT, CORRECT_GSTIN)
    if not os.path.isdir(src):
        raise FileNotFoundError(f"No case folder for {WRONG_GSTIN}")
    if not re.match(r"^[0-9]{2}[A-Z]{5}[0-9]{4}[A-Z][1-9A-Z]Z[0-9A-Z]$", CORRECT_GSTIN):
        raise ValueError(f"'{CORRECT_GSTIN}' is not a valid GSTIN")
    for sub in ("00_ENGINE", "01_RAW_UPLOADS", "02_EXTRACTED", "03_REPORTS"):
        os.makedirs(os.path.join(dst, sub), exist_ok=True)
    moved = []
    s_raw, d_raw = os.path.join(src, "01_RAW_UPLOADS"), os.path.join(dst, "01_RAW_UPLOADS")
    for f in sorted(os.listdir(s_raw)) if os.path.isdir(s_raw) else []:
        if not f.lower().endswith((".xlsx", ".xls")):
            continue
        target = os.path.join(d_raw, f)
        if os.path.exists(target):
            print("   already present, left in place:", f); continue
        shutil.move(os.path.join(s_raw, f), target)
        moved.append(f)
    if moved:
        man = os.path.join(d_raw, "_MANIFEST.csv")
        new = not os.path.exists(man)
        now = datetime.datetime.now().isoformat(timespec="seconds")
        with open(man, "a", newline="", encoding="utf-8") as fh:
            w = csv.writer(fh)
            if new:
                w.writerow(["gstin", "taxpayer", "stored_as", "original_name", "md5",
                            "size_bytes", "uploaded_at"])
            for f in moved:
                p = os.path.join(d_raw, f)
                w.writerow([CORRECT_GSTIN, "", f, f, _md5(p), os.path.getsize(p),
                            now + "  (re-filed from " + WRONG_GSTIN + ")"])
    print(f"Moved {len(moved)} file(s) from {WRONG_GSTIN} to {CORRECT_GSTIN}.")
    print("The old folder and its reports are untouched — delete it by hand once you are satisfied.")
    print("Now set the GSTIN in Step 2 to", CORRECT_GSTIN, "and run Steps 2, 6, 7 and 8 again.")


---

### Reading the output

Open the workbook at **`00_INDEX`** first — it lists every sheet and what it holds.

- **`20_FINDINGS`** is the one to read if you have two minutes. Findings are ranked by revenue implication and each carries the sheet it was computed on.
- **`21_REPORT`** is the same thing in narrative form.
- **`22_ACTION_POINTS`** is what to do, in the order to do it.
- **`23_DATA_GAPS`** is what to fetch next.
- **`24_LIMITATION`** is indicative only — extension notifications are not applied and several have been litigated. Check every date against the notifications in force.

### Before anything goes into a notice

This is a preliminary investigation on returns data. Verify against the common portal, the electronic credit and cash ledgers, the returns themselves, GSTR-2B, and the books of account. The workbook is built so that every figure can be traced back to the source cell it came from — use that trail when a senior asks.

<div style="text-align:right;font-size:12px;color:#444;">Created by <b>Ram Lakhan Meena, INSPECTOR CBIC</b><br>2026-10-09</div>
